In [ ]:
DECLARE @AdminID INT = 303
DECLARE @Month   INT = 5
DECLARE @Year    INT = 2026

;WITH
LastAllocated AS (
    SELECT h.OrderID, h.AssignedTo_UserID,
        ROW_NUMBER() OVER (PARTITION BY h.OrderID ORDER BY h.Date_Ctreated DESC) AS rn
    FROM Table_OrderStatushistory h
    WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
),
LastAlloc AS (SELECT OrderID, AssignedTo_UserID FROM LastAllocated WHERE rn=1)

SELECT
    od.Order_Number                                          AS [Order #],
    od.CompanyName                                           AS [Company],
    -- All possible date fields
    CAST(od.CompletionOn AS DATE)                            AS [od.CompletionOn],
    CAST(m.Retail_CompletionDate AS DATE)                    AS [m.Retail_CompletionDate],
    CAST(m.Jaz_CompletionDate AS DATE)                       AS [m.Jaz_CompletionDate],
    CAST(m.ConversionEndDate_1 AS DATE)                      AS [m.ConversionEndDate_1],
    CAST(m.ConversionEndDate_2 AS DATE)                      AS [m.ConversionEndDate_2],
    CAST(m.Date_OrderReviewStatus AS DATE)                   AS [m.Date_OrderReviewStatus],
    -- What post_conversion.py currently uses
    CASE
        WHEN m.Date_OrderReviewStatus IS NOT NULL THEN CAST(m.Date_OrderReviewStatus AS DATE)
        WHEN m.ConversionEndDate_1    IS NOT NULL THEN CAST(m.ConversionEndDate_1    AS DATE)
        WHEN m.Retail_CompletionDate  IS NOT NULL THEN CAST(m.Retail_CompletionDate  AS DATE)
        WHEN m.Jaz_CompletionDate     IS NOT NULL THEN CAST(m.Jaz_CompletionDate     AS DATE)
        ELSE                                           CAST(m.ConversionEndDate_2    AS DATE)
    END                                                      AS [post_conversion date],
    -- What missing_status uses
    CAST(COALESCE(m.Retail_CompletionDate, od.CompletionOn) AS DATE) AS [missing_status date]

FROM OrderDetails od
INNER JOIN LastAlloc la ON la.OrderID = od.ID
LEFT  JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
WHERE la.AssignedTo_UserID = @AdminID
  AND od.Flag_Deleted = 0
  AND (
    MONTH(CASE
        WHEN m.Date_OrderReviewStatus IS NOT NULL THEN m.Date_OrderReviewStatus
        WHEN m.ConversionEndDate_1    IS NOT NULL THEN m.ConversionEndDate_1
        WHEN m.Retail_CompletionDate  IS NOT NULL THEN m.Retail_CompletionDate
        WHEN m.Jaz_CompletionDate     IS NOT NULL THEN m.Jaz_CompletionDate
        ELSE m.ConversionEndDate_2
    END) = @Month
    AND YEAR(CASE
        WHEN m.Date_OrderReviewStatus IS NOT NULL THEN m.Date_OrderReviewStatus
        WHEN m.ConversionEndDate_1    IS NOT NULL THEN m.ConversionEndDate_1
        WHEN m.Retail_CompletionDate  IS NOT NULL THEN m.Retail_CompletionDate
        WHEN m.Jaz_CompletionDate     IS NOT NULL THEN m.Jaz_CompletionDate
        ELSE m.ConversionEndDate_2
    END) = @Year
  )
ORDER BY [post_conversion date]

In [1]:
"""
Fetches all Keka DBdash tables and prints row counts + sample data.
Run locally: python inspect_keka_tables.py
Paste the full output back so the structure can be reviewed.
"""
import requests
import json

AUTH_KEY = 'keywWPoCKnwcJAA'
DB_ID    = '6a17ec96a95e7ac45342c0e4'
BASE     = f'https://table-api.viasocket.com/{DB_ID}'
HEADERS  = {'auth-key': AUTH_KEY}

TABLES = {
    'keka_employees':              'tblrj1w62',
    'keka_departments':            'tblq2rvma',
    'keka_job_titles':             'tblznqw5q',
    'keka_policies':               'tbl0gobt2',
    'keka_managers':                'tbl7oud8d',
    'keka_employee_details':       'tblp2kh5y',
    'keka_employee_status':        'tbll6kzp6',
    'keka_employee_groups':        'tblto5irg',
    'keka_custom_fields':          'tblrl3zaj',
    'keka_employee_custom_values': 'tblak189v',
    'keka_attendance':             'tblr5wgh0',
    'keka_leaves':                 'tbluhz03p',
    'keka_sync_log':               'tblctjkjp',
}

def fetch_table(name, table_id):
    url = f'{BASE}/{table_id}'
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
        data = r.json()
        rows = data.get('data', {}).get('rows', [])
        return rows, None
    except Exception as e:
        return [], str(e)

print('='*70)
print('  KEKA DBDASH TABLES — INSPECTION REPORT')
print('='*70)

for name, table_id in TABLES.items():
    rows, err = fetch_table(name, table_id)
    print(f'\n{"-"*70}')
    print(f'TABLE: {name}  (id: {table_id})')
    print(f'{"-"*70}')

    if err:
        print(f'  ERROR: {err}')
        continue

    print(f'  Row count: {len(rows)}')

    if rows:
        print(f'  Columns found: {list(rows[0].keys())}')
        print(f'\n  Sample row (first record):')
        print(json.dumps(rows[0], indent=4, default=str))
    else:
        print('  (table is empty — no rows yet)')

print('\n' + '='*70)
print('  END OF REPORT')
print('='*70)

  KEKA DBDASH TABLES — INSPECTION REPORT

----------------------------------------------------------------------
TABLE: keka_employees  (id: tblrj1w62)
----------------------------------------------------------------------
  Row count: 200
  Columns found: ['rowid', 'autonumber', 'createdat', 'createdby', 'updatedat', 'updatedby', 'name', 'employee_number', 'first_name', 'middle_name', 'last_name', 'display_name', 'email', 'personal_email', 'gender', 'date_of_birth', 'blood_group', 'nationality', 'marriage_date', 'attendance_number', 'professional_summary', 'created_at', 'updated_at']

  Sample row (first record):
{
    "rowid": "row2b6wqmuwo",
    "autonumber": 6,
    "createdat": "2026-06-18T05:13:03.371Z",
    "createdby": {
        "id": "SocketKey_21295_id4cvu"
    },
    "updatedat": null,
    "updatedby": null,
    "name": "f131410e-74dc-4648-b594-83443ae10021",
    "employee_number": "03",
    "first_name": "Arjun  ",
    "middle_name": "null",
    "last_name": "Bhadoriya",
   

In [2]:
"""
KEKA DBDASH — FULL DATA DUMP (READ-ONLY)
========================================
Fetches ALL rows from ALL 13 tables (with pagination) and:
  1. Prints row count + columns + 2 sample rows per table to the console
  2. Saves each table to a CSV file in the current folder for your reference

Run in a Jupyter cell OR plain terminal:  python keka_dump_all.py
Nothing is written back to the database — read only.
"""

import requests
import json
import csv
import os

# ── Config ────────────────────────────────────────────────────────────────────
AUTH_KEY = 'keywWPoCKnwcJAA'
DB_ID    = '6a17ec96a95e7ac45342c0e4'
BASE     = f'https://table-api.viasocket.com/{DB_ID}'
HEADERS  = {'auth-key': AUTH_KEY}

TABLES = {
    'keka_employees':              'tblrj1w62',
    'keka_departments':            'tblq2rvma',
    'keka_job_titles':             'tblznqw5q',
    'keka_policies':               'tbl0gobt2',
    'keka_managers':                'tbl7oud8d',
    'keka_employee_details':       'tblp2kh5y',
    'keka_employee_status':        'tbll6kzp6',
    'keka_employee_groups':        'tblto5irg',
    'keka_custom_fields':          'tblrl3zaj',
    'keka_employee_custom_values': 'tblak189v',
    'keka_attendance':             'tblr5wgh0',
    'keka_leaves':                 'tbluhz03p',
    'keka_sync_log':               'tblctjkjp',
}

OUT_DIR = 'keka_dump'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Fetch ALL rows with offset pagination ─────────────────────────────────────
def fetch_all(table_id):
    rows, offset, pages = [], None, 0
    while True:
        params = {'offset': offset} if offset else {}
        r = requests.get(f'{BASE}/{table_id}', headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        data = r.json().get('data', {})
        batch = data.get('rows', [])
        rows.extend(batch)
        pages += 1
        offset = data.get('offset')
        if not offset or not batch or pages > 50:   # safety cap
            break
    return rows

# ── Flatten nested values (like createdby dict) for CSV ───────────────────────
def flatten_value(v):
    if isinstance(v, (dict, list)):
        return json.dumps(v, default=str)
    return v

# ── Main ──────────────────────────────────────────────────────────────────────
print('=' * 80)
print('  KEKA DBDASH — FULL DATA DUMP')
print('=' * 80)

grand_total = 0
report = []

for name, table_id in TABLES.items():
    print(f'\n{"-" * 80}')
    print(f'TABLE: {name}   (id: {table_id})')
    print(f'{"-" * 80}')

    try:
        rows = fetch_all(table_id)
    except Exception as e:
        print(f'  ERROR fetching: {e}')
        report.append((name, 'ERROR', str(e)))
        continue

    grand_total += len(rows)
    print(f'  Total rows: {len(rows)}')

    if not rows:
        print('  (empty table)')
        report.append((name, 0, 'empty'))
        continue

    cols = list(rows[0].keys())
    print(f'  Columns ({len(cols)}): {cols}')

    # Print 2 sample rows
    print('\n  Sample rows:')
    for i, row in enumerate(rows[:2]):
        print(f'  --- row {i+1} ---')
        for k, v in row.items():
            print(f'      {k}: {v}')

    # Save to CSV
    csv_path = os.path.join(OUT_DIR, f'{name}.csv')
    try:
        # Collect union of all keys across rows (some rows may differ)
        all_keys = []
        for row in rows:
            for k in row.keys():
                if k not in all_keys:
                    all_keys.append(k)
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=all_keys)
            writer.writeheader()
            for row in rows:
                writer.writerow({k: flatten_value(row.get(k, '')) for k in all_keys})
        print(f'\n  ✅ Saved {len(rows)} rows → {csv_path}')
        report.append((name, len(rows), csv_path))
    except Exception as e:
        print(f'  ⚠️  CSV save failed: {e}')
        report.append((name, len(rows), f'csv-fail: {e}'))

# ── Summary ───────────────────────────────────────────────────────────────────
print('\n' + '=' * 80)
print('  SUMMARY')
print('=' * 80)
print(f'{"Table":<32}{"Rows":<8}{"Output"}')
print('-' * 80)
for name, count, out in report:
    print(f'{name:<32}{str(count):<8}{out}')
print('-' * 80)
print(f'{"TOTAL ROWS":<32}{grand_total}')
print(f'\nCSV files saved in folder: ./{OUT_DIR}/')
print('All read-only. Nothing written to the database.')

  KEKA DBDASH — FULL DATA DUMP

--------------------------------------------------------------------------------
TABLE: keka_employees   (id: tblrj1w62)
--------------------------------------------------------------------------------
  Total rows: 286
  Columns (23): ['rowid', 'autonumber', 'createdat', 'createdby', 'updatedat', 'updatedby', 'name', 'employee_number', 'first_name', 'middle_name', 'last_name', 'display_name', 'email', 'personal_email', 'gender', 'date_of_birth', 'blood_group', 'nationality', 'marriage_date', 'attendance_number', 'professional_summary', 'created_at', 'updated_at']

  Sample rows:
  --- row 1 ---
      rowid: row2b6wqmuwo
      autonumber: 6
      createdat: 2026-06-18T05:13:03.371Z
      createdby: {'id': 'SocketKey_21295_id4cvu'}
      updatedat: None
      updatedby: None
      name: f131410e-74dc-4648-b594-83443ae10021
      employee_number: 03
      first_name: Arjun  
      middle_name: null
      last_name: Bhadoriya
      display_name: Arjun Bhado

In [3]:
"""
KEKA ATTENDANCE — DERIVATION VERIFICATION (READ-ONLY)
=====================================================
Run this in a Jupyter cell (or plain python). It does NOT write anything yet.
It fetches attendance + employee-status, derives late/early/hours/absent per
punch row, handles mid-month joiners, and prints a per-employee monthly summary
so you can verify the numbers before we enable DB writeback.

Policy (per your spec):
  - Late grace  : 20 min after shift_start_time   (UI may *show* 15, that's display-only)
  - Early exit  : left more than 10 min before shift_end_time
  - Joiners     : working-day denominator starts at joining_date, not the 1st
"""

import requests
import json
from datetime import datetime, timedelta
from collections import defaultdict

# ── Config ────────────────────────────────────────────────────────────────────
AUTH_KEY = 'keywWPoCKnwcJAA'
DB_ID    = '6a17ec96a95e7ac45342c0e4'
BASE     = f'https://table-api.viasocket.com/{DB_ID}'
HEADERS  = {'auth-key': AUTH_KEY}

TBL_ATTENDANCE = 'tblr5wgh0'
TBL_STATUS     = 'tbll6kzp6'
TBL_EMPLOYEES  = 'tblrj1w62'

LATE_GRACE_MIN  = 20   # minutes after shift start before it counts as "late"
EARLY_EXIT_MIN  = 10   # minutes before shift end before it counts as "early exit"

# Which month to verify
TARGET_MONTH = 4    # data sample showed April punches
TARGET_YEAR  = 2026

# ── Fetch helpers ─────────────────────────────────────────────────────────────
def fetch_all(table_id):
    """Fetch every row, following offset pagination if present."""
    rows, offset = [], None
    while True:
        url = f'{BASE}/{table_id}'
        params = {'offset': offset} if offset else {}
        r = requests.get(url, headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        data = r.json().get('data', {})
        batch = data.get('rows', [])
        rows.extend(batch)
        offset = data.get('offset')
        if not offset or not batch:
            break
    return rows

# ── Parsing helpers ───────────────────────────────────────────────────────────
def parse_dt(s):
    """Parse 'YYYY-MM-DD HH:MM:SS' or ISO 'YYYY-MM-DDTHH:MM:SSZ'. Return None on junk."""
    if not s or s in ('null', 'undefined', 'None'):
        return None
    s = s.strip().replace('T', ' ').replace('Z', '')
    for fmt in ('%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M', '%Y-%m-%d'):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            continue
    return None

def minutes_between(a, b):
    """Whole minutes from a to b (b - a). None if either missing."""
    if not a or not b:
        return None
    return int((b - a).total_seconds() // 60)

# ── Step 1: pull data ─────────────────────────────────────────────────────────
print('Fetching attendance...')
att_rows = fetch_all(TBL_ATTENDANCE)
print(f'  attendance rows: {len(att_rows)}')

print('Fetching employee status (for joining dates)...')
status_rows = fetch_all(TBL_STATUS)
print(f'  status rows: {len(status_rows)}')

print('Fetching employees (for names)...')
emp_rows = fetch_all(TBL_EMPLOYEES)
print(f'  employee rows: {len(emp_rows)}')

# Map keka_id -> joining_date, and employee_no/name -> display_name
joining_by_id = {}
for s in status_rows:
    jd = parse_dt(s.get('joining_date'))
    if s.get('keka_id'):
        joining_by_id[s['keka_id']] = jd

name_by_empno = {}
for e in emp_rows:
    if e.get('attendance_number'):
        name_by_empno[str(e['attendance_number'])] = e.get('display_name', '?')

# ── Step 2: derive per punch row ──────────────────────────────────────────────
# NOTE: in the synced data the 'date' column wrongly holds the employee NAME,
# and 'name' holds a keka_id. We take the real date from punch_in instead.
derived = []
for row in att_rows:
    punch_in   = parse_dt(row.get('punch_in'))
    punch_out  = parse_dt(row.get('punch_out'))
    shift_in   = parse_dt(row.get('shift_start_time'))
    shift_out  = parse_dt(row.get('shift_end_time'))

    if not punch_in:
        continue  # no usable record

    work_date = punch_in.date()
    if work_date.month != TARGET_MONTH or work_date.year != TARGET_YEAR:
        continue

    late_min  = minutes_between(shift_in, punch_in)   if shift_in  else None
    early_min = minutes_between(punch_out, shift_out) if (shift_out and punch_out) else None
    hours     = round((punch_out - punch_in).total_seconds() / 3600, 2) if punch_out else None

    is_late       = (late_min is not None and late_min > LATE_GRACE_MIN)
    is_early_exit = (early_min is not None and early_min > EARLY_EXIT_MIN)

    derived.append({
        'employee_no': str(row.get('employee_no', '')),
        'keka_id':     row.get('name', ''),     # 'name' col actually holds keka_id
        'date':        work_date,
        'punch_in':    punch_in.strftime('%H:%M'),
        'punch_out':   punch_out.strftime('%H:%M') if punch_out else '--',
        'shift_in':    shift_in.strftime('%H:%M') if shift_in else '--',
        'shift_out':   shift_out.strftime('%H:%M') if shift_out else '--',
        'late_min':    late_min if late_min and late_min > 0 else 0,
        'early_min':   early_min if early_min and early_min > 0 else 0,
        'hours':       hours,
        'is_late':     is_late,
        'is_early_exit': is_early_exit,
    })

print(f'\nDerived {len(derived)} usable punch rows for {TARGET_MONTH}/{TARGET_YEAR}')

# ── Step 3: working-day helper (Mon-Sat, excl Sunday) ─────────────────────────
def working_days_in_range(start_date, end_date):
    """Count Mon-Sat days between start and end inclusive."""
    if start_date > end_date:
        return 0
    n, d = 0, start_date
    while d <= end_date:
        if d.weekday() != 6:   # 6 = Sunday
            n += 1
        d += timedelta(days=1)
    return n

# Month boundaries
month_start = datetime(TARGET_YEAR, TARGET_MONTH, 1).date()
if TARGET_MONTH == 12:
    month_end = datetime(TARGET_YEAR, 12, 31).date()
else:
    month_end = (datetime(TARGET_YEAR, TARGET_MONTH + 1, 1) - timedelta(days=1)).date()

# ── Step 4: per-employee monthly summary ──────────────────────────────────────
by_emp = defaultdict(list)
for d in derived:
    by_emp[d['employee_no']].append(d)

print('\n' + '=' * 100)
print(f'  MONTHLY ATTENDANCE SUMMARY — {TARGET_MONTH}/{TARGET_YEAR}')
print(f'  Late grace: {LATE_GRACE_MIN}min   |   Early-exit threshold: {EARLY_EXIT_MIN}min   |   Sundays excluded')
print('=' * 100)

summary = []
for empno, days in sorted(by_emp.items()):
    keka_id = days[0]['keka_id']
    name    = name_by_empno.get(empno, '?')

    # Mid-month joiner handling: denominator starts at max(month_start, joining_date)
    join_dt = joining_by_id.get(keka_id)
    eff_start = month_start
    if join_dt and join_dt.date() > month_start:
        eff_start = join_dt.date()

    expected_wd = working_days_in_range(eff_start, month_end)
    present_days = len(set(d['date'] for d in days))
    late_days    = sum(1 for d in days if d['is_late'])
    early_days   = sum(1 for d in days if d['is_early_exit'])
    unrecorded   = max(0, expected_wd - present_days)   # absent OR leave (cannot distinguish)
    avg_hours    = round(sum(d['hours'] for d in days if d['hours']) / len([d for d in days if d['hours']]), 2) if any(d['hours'] for d in days) else 0

    joiner_flag = ''
    if join_dt and join_dt.date() > month_start:
        joiner_flag = f'  [joined {join_dt.date()} — prorated to {expected_wd}wd]'

    summary.append({
        'empno': empno, 'name': name,
        'expected_wd': expected_wd, 'present': present_days,
        'late': late_days, 'early': early_days,
        'unrecorded': unrecorded, 'avg_hours': avg_hours,
    })

    print(f'\n{name}  (#{empno}){joiner_flag}')
    print(f'   Expected working days : {expected_wd}')
    print(f'   Present (punched)     : {present_days}')
    print(f'   Late (> {LATE_GRACE_MIN}min)        : {late_days}')
    print(f'   Early exit (> {EARLY_EXIT_MIN}min)  : {early_days}')
    print(f'   Unrecorded/absent     : {unrecorded}')
    print(f'   Avg hours/day         : {avg_hours}')

# ── Step 5: a few sample daily rows so you can eyeball the late/early logic ────
print('\n' + '=' * 100)
print('  SAMPLE DAILY ROWS (first 15) — verify late/early math by hand')
print('=' * 100)
print(f'{"Emp":<6}{"Date":<12}{"In":<7}{"ShiftIn":<9}{"Late":<6}{"Out":<7}{"ShiftOut":<10}{"Early":<7}{"Hrs":<6}{"Flags"}')
for d in derived[:15]:
    flags = []
    if d['is_late']:       flags.append('LATE')
    if d['is_early_exit']: flags.append('EARLY')
    print(f'{d["employee_no"]:<6}{str(d["date"]):<12}{d["punch_in"]:<7}{d["shift_in"]:<9}'
          f'{d["late_min"]:<6}{d["punch_out"]:<7}{d["shift_out"]:<10}{d["early_min"]:<7}'
          f'{str(d["hours"]):<6}{",".join(flags)}')

print('\nDONE — read-only. No data written. Verify the numbers above, then we enable writeback.')

Fetching attendance...
  attendance rows: 1002
Fetching employee status (for joining dates)...
  status rows: 286
Fetching employees (for names)...
  employee rows: 286

Derived 551 usable punch rows for 4/2026

  MONTHLY ATTENDANCE SUMMARY — 4/2026
  Late grace: 20min   |   Early-exit threshold: 10min   |   Sundays excluded

?  (#06)
   Expected working days : 26
   Present (punched)     : 23
   Late (> 20min)        : 7
   Early exit (> 10min)  : 2
   Unrecorded/absent     : 3
   Avg hours/day         : 7.74

Abhishek Chowdhary  (#10)
   Expected working days : 26
   Present (punched)     : 12
   Late (> 20min)        : 2
   Early exit (> 10min)  : 4
   Unrecorded/absent     : 14
   Avg hours/day         : 7.52

Ayushi Modi  (#106)
   Expected working days : 26
   Present (punched)     : 25
   Late (> 20min)        : 0
   Early exit (> 10min)  : 3
   Unrecorded/absent     : 1
   Avg hours/day         : 7.94

Sagar Jaiswal  (#11)
   Expected working days : 26
   Present (punched)     

In [4]:
"""
KEKA ATTENDANCE — DERIVATION VERIFICATION v2 (READ-ONLY, HARDENED)
==================================================================
Fixes over v1:
  1. JOIN on keka_id (attendance 'name' col == employees 'name' col), not employee_no
     → fixes all the '?' names
  2. Detect INCOMPLETE records (missing punch_out, or a lone after-hours punch)
     → excluded from late/early/hours so we don't get "513 min late"
  3. Count UNIQUE working dates only (fixes present > expected like 28 > 26)
  4. Exclude Sundays AND each person's weekly-off day from "expected" where known
  5. Separate "SUSPICIOUS / INCOMPLETE" list so you can eyeball bad rows
  6. Mid-month joiner proration via joining_date

Policy:
  Late grace   : 20 min after shift_start  (UI shows 15 — display only)
  Early exit   : left > 10 min before shift_end
  Incomplete   : missing punch_in OR punch_out, OR single punch after 14:00
  Min valid hrs: a real working day pair should be >= 1 hour apart

READ ONLY. Nothing is written back.
"""

import requests
from datetime import datetime, timedelta
from collections import defaultdict

# ── Config ────────────────────────────────────────────────────────────────────
AUTH_KEY = 'keywWPoCKnwcJAA'
DB_ID    = '6a17ec96a95e7ac45342c0e4'
BASE     = f'https://table-api.viasocket.com/{DB_ID}'
HEADERS  = {'auth-key': AUTH_KEY}

TBL_ATTENDANCE = 'tblr5wgh0'
TBL_STATUS     = 'tbll6kzp6'
TBL_EMPLOYEES  = 'tblrj1w62'

LATE_GRACE_MIN   = 20
EARLY_EXIT_MIN   = 10
LONE_PUNCH_HOUR  = 14      # a single punch after 2 PM with no pair = suspicious/incomplete
MIN_PAIR_MINUTES = 60      # punch_in/out less than this apart = suspicious

TARGET_MONTH = 4
TARGET_YEAR  = 2026

# ── Fetch with pagination ─────────────────────────────────────────────────────
def fetch_all(table_id):
    rows, offset, pages = [], None, 0
    while True:
        params = {'offset': offset} if offset else {}
        r = requests.get(f'{BASE}/{table_id}', headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        d = r.json().get('data', {})
        batch = d.get('rows', [])
        rows.extend(batch)
        pages += 1
        offset = d.get('offset')
        if not offset or not batch or pages > 50:
            break
    return rows

def parse_dt(s):
    if not s or s in ('null', 'undefined', 'None', None):
        return None
    s = str(s).strip().replace('T', ' ').replace('Z', '')
    for fmt in ('%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M', '%Y-%m-%d'):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            continue
    return None

def mins(a, b):
    if not a or not b:
        return None
    return int((b - a).total_seconds() // 60)

# ── Load ──────────────────────────────────────────────────────────────────────
print('Fetching attendance / status / employees ...')
att      = fetch_all(TBL_ATTENDANCE)
status   = fetch_all(TBL_STATUS)
emps     = fetch_all(TBL_EMPLOYEES)
print(f'  attendance={len(att)}  status={len(status)}  employees={len(emps)}')

# keka_id -> name, joining date
name_by_kid = {}
join_by_kid = {}
for e in emps:
    kid = e.get('name')           # 'name' col holds keka_id
    if kid:
        name_by_kid[kid] = e.get('display_name', '?')
for s in status:
    kid = s.get('keka_id')
    if kid:
        join_by_kid[kid] = parse_dt(s.get('joining_date'))

# ── Classify each attendance row ──────────────────────────────────────────────
clean   = []     # usable rows
suspect = []     # incomplete / suspicious rows

for row in att:
    kid       = row.get('name')                 # keka_id
    punch_in  = parse_dt(row.get('punch_in'))
    punch_out = parse_dt(row.get('punch_out'))
    shift_in  = parse_dt(row.get('shift_start_time'))
    shift_out = parse_dt(row.get('shift_end_time'))
    weekly_off= row.get('weekly_off_policy')

    anchor = punch_in or punch_out
    if not anchor:
        continue
    if anchor.month != TARGET_MONTH or anchor.year != TARGET_YEAR:
        continue

    work_date = anchor.date()
    reason = None

    # Incomplete / suspicious detection
    if not punch_in or not punch_out:
        # single punch — if it's late in the day, likely a forgotten morning punch
        lone = punch_in or punch_out
        if lone and lone.hour >= LONE_PUNCH_HOUR:
            reason = 'lone_after_hours_punch'
        else:
            reason = 'missing_punch'
    else:
        pair_min = mins(punch_in, punch_out)
        if pair_min is not None and pair_min < MIN_PAIR_MINUTES:
            reason = 'pair_too_short'

    if reason:
        suspect.append({
            'kid': kid, 'name': name_by_kid.get(kid, '?'),
            'date': work_date,
            'punch_in':  punch_in.strftime('%H:%M') if punch_in else '--',
            'punch_out': punch_out.strftime('%H:%M') if punch_out else '--',
            'reason': reason,
        })
        continue

    # Clean row — compute metrics
    late_min  = mins(shift_in, punch_in) if shift_in else None
    early_min = mins(punch_out, shift_out) if shift_out else None
    hours     = round((punch_out - punch_in).total_seconds() / 3600, 2)

    clean.append({
        'kid': kid, 'name': name_by_kid.get(kid, '?'),
        'date': work_date,
        'punch_in': punch_in.strftime('%H:%M'),
        'punch_out': punch_out.strftime('%H:%M'),
        'shift_in': shift_in.strftime('%H:%M') if shift_in else '--',
        'shift_out': shift_out.strftime('%H:%M') if shift_out else '--',
        'late_min':  late_min if (late_min and late_min > 0) else 0,
        'early_min': early_min if (early_min and early_min > 0) else 0,
        'hours': hours,
        'is_late':       bool(late_min is not None and late_min > LATE_GRACE_MIN),
        'is_early_exit': bool(early_min is not None and early_min > EARLY_EXIT_MIN),
        'weekly_off': weekly_off,
    })

print(f'\nClean rows: {len(clean)}   Suspicious/incomplete: {len(suspect)}')

# ── Working-day helper ────────────────────────────────────────────────────────
def working_days(start_d, end_d):
    if start_d > end_d:
        return 0
    n, d = 0, start_d
    while d <= end_d:
        if d.weekday() != 6:    # exclude Sunday
            n += 1
        d += timedelta(days=1)
    return n

month_start = datetime(TARGET_YEAR, TARGET_MONTH, 1).date()
month_end = (datetime(TARGET_YEAR, TARGET_MONTH + 1, 1) - timedelta(days=1)).date() \
            if TARGET_MONTH < 12 else datetime(TARGET_YEAR, 12, 31).date()

# ── Per-employee summary ──────────────────────────────────────────────────────
by_emp = defaultdict(list)
for c in clean:
    by_emp[c['kid']].append(c)

# also bucket suspects per employee for visibility
suspect_by_emp = defaultdict(list)
for s in suspect:
    suspect_by_emp[s['kid']].append(s)

print('\n' + '=' * 100)
print(f'  MONTHLY SUMMARY — {TARGET_MONTH}/{TARGET_YEAR}   (Sundays excluded, incompletes separated)')
print(f'  Late > {LATE_GRACE_MIN}min  |  Early-exit > {EARLY_EXIT_MIN}min')
print('=' * 100)

for kid, days in sorted(by_emp.items(), key=lambda x: name_by_kid.get(x[0], '?')):
    name = name_by_kid.get(kid, '?')

    join_dt = join_by_kid.get(kid)
    eff_start = month_start
    joiner = ''
    if join_dt and join_dt.date() > month_start:
        eff_start = join_dt.date()
        joiner = f'  [joined {join_dt.date()}]'

    expected = working_days(eff_start, month_end)
    present  = len(set(d['date'] for d in days))         # UNIQUE dates only
    present  = min(present, expected)                     # cap — can't exceed expected
    late     = sum(1 for d in days if d['is_late'])
    early    = sum(1 for d in days if d['is_early_exit'])
    n_suspect= len(suspect_by_emp.get(kid, []))
    unrecorded = max(0, expected - present)
    valid_hours = [d['hours'] for d in days if d['hours']]
    avg_hours = round(sum(valid_hours) / len(valid_hours), 2) if valid_hours else 0

    print(f'\n{name}{joiner}')
    print(f'   Expected working days : {expected}')
    print(f'   Present (unique days) : {present}')
    print(f'   Late (> {LATE_GRACE_MIN}min)        : {late}')
    print(f'   Early exit (> {EARLY_EXIT_MIN}min)  : {early}')
    print(f'   Unrecorded/absent     : {unrecorded}')
    print(f'   Incomplete punches    : {n_suspect}')
    print(f'   Avg hours/day         : {avg_hours}')

# ── Suspicious rows listing ───────────────────────────────────────────────────
print('\n' + '=' * 100)
print(f'  SUSPICIOUS / INCOMPLETE ROWS  ({len(suspect)} total) — first 30')
print('=' * 100)
print(f'{"Name":<24}{"Date":<12}{"In":<7}{"Out":<7}{"Reason"}')
for s in suspect[:30]:
    print(f'{s["name"][:23]:<24}{str(s["date"]):<12}{s["punch_in"]:<7}{s["punch_out"]:<7}{s["reason"]}')

# ── Reason breakdown ──────────────────────────────────────────────────────────
reason_counts = defaultdict(int)
for s in suspect:
    reason_counts[s['reason']] += 1
print('\nIncomplete reason breakdown:')
for r, c in sorted(reason_counts.items(), key=lambda x: -x[1]):
    print(f'   {r:<26}: {c}')

# ── Clean sample for hand-verification ────────────────────────────────────────
print('\n' + '=' * 100)
print('  CLEAN DAILY SAMPLE (first 15) — verify late/early by hand')
print('=' * 100)
print(f'{"Name":<22}{"Date":<12}{"In":<7}{"SIn":<7}{"Late":<6}{"Out":<7}{"SOut":<7}{"Early":<7}{"Hrs":<6}{"Flags"}')
for c in clean[:15]:
    flags = ','.join([x for x, on in [('LATE', c['is_late']), ('EARLY', c['is_early_exit'])] if on])
    print(f'{c["name"][:21]:<22}{str(c["date"]):<12}{c["punch_in"]:<7}{c["shift_in"]:<7}'
          f'{c["late_min"]:<6}{c["punch_out"]:<7}{c["shift_out"]:<7}{c["early_min"]:<7}'
          f'{str(c["hours"]):<6}{flags}')

# ── Active users count (for dashboard) ────────────────────────────────────────
active = sum(1 for s in status if str(s.get('employment_status')) == '0'
                              and str(s.get('account_status')) == '1')
print('\n' + '=' * 100)
print(f'  ACTIVE USERS (employment_status=0 AND account_status=1): {active}')
print(f'  Total employees in system: {len(emps)}')
print('=' * 100)

print('\nDONE — read only. Review names, incomplete handling, present<=expected, then we enable writeback.')

Fetching attendance / status / employees ...
  attendance=1002  status=286  employees=286

Clean rows: 517   Suspicious/incomplete: 34

  MONTHLY SUMMARY — 4/2026   (Sundays excluded, incompletes separated)
  Late > 20min  |  Early-exit > 10min

Aarti Panwar
   Expected working days : 26
   Present (unique days) : 21
   Late (> 20min)        : 0
   Early exit (> 10min)  : 2
   Unrecorded/absent     : 5
   Incomplete punches    : 0
   Avg hours/day         : 8.14

Aastha Sadhwani
   Expected working days : 26
   Present (unique days) : 21
   Late (> 20min)        : 1
   Early exit (> 10min)  : 0
   Unrecorded/absent     : 5
   Incomplete punches    : 1
   Avg hours/day         : 8.95

Abhishek Gour
   Expected working days : 26
   Present (unique days) : 1
   Late (> 20min)        : 0
   Early exit (> 10min)  : 0
   Unrecorded/absent     : 25
   Incomplete punches    : 0
   Avg hours/day         : 8.5

Antim Rathore
   Expected working days : 26
   Present (unique days) : 26
   Late (> 

In [5]:
"""
KEKA ATTENDANCE — DERIVATION VERIFICATION v2 (READ-ONLY, HARDENED)
==================================================================
Fixes over v1:
  1. JOIN on keka_id (attendance 'name' col == employees 'name' col), not employee_no
     → fixes all the '?' names
  2. Detect INCOMPLETE records (missing punch_out, or a lone after-hours punch)
     → excluded from late/early/hours so we don't get "513 min late"
  3. Count UNIQUE working dates only (fixes present > expected like 28 > 26)
  4. Exclude Sundays AND each person's weekly-off day from "expected" where known
  5. Separate "SUSPICIOUS / INCOMPLETE" list so you can eyeball bad rows
  6. Mid-month joiner proration via joining_date

Policy:
  Late grace   : 20 min after shift_start  (UI shows 15 — display only)
  Early exit   : left > 10 min before shift_end
  Incomplete   : missing punch_in OR punch_out, OR single punch after 14:00
  Min valid hrs: a real working day pair should be >= 1 hour apart

READ ONLY. Nothing is written back.
"""

import requests
from datetime import datetime, timedelta
from collections import defaultdict

# ── Config ────────────────────────────────────────────────────────────────────
AUTH_KEY = 'keywWPoCKnwcJAA'
DB_ID    = '6a17ec96a95e7ac45342c0e4'
BASE     = f'https://table-api.viasocket.com/{DB_ID}'
HEADERS  = {'auth-key': AUTH_KEY}

TBL_ATTENDANCE = 'tblr5wgh0'
TBL_STATUS     = 'tbll6kzp6'
TBL_EMPLOYEES  = 'tblrj1w62'

LATE_GRACE_MIN   = 20
EARLY_EXIT_MIN   = 10
LONE_PUNCH_HOUR  = 14      # a single punch after 2 PM with no pair = suspicious/incomplete
MIN_PAIR_MINUTES = 60      # punch_in/out less than this apart = suspicious

TARGET_MONTH = 5
TARGET_YEAR  = 2026

# ── Fetch with pagination ─────────────────────────────────────────────────────
def fetch_all(table_id):
    rows, offset, pages = [], None, 0
    while True:
        params = {'offset': offset} if offset else {}
        r = requests.get(f'{BASE}/{table_id}', headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        d = r.json().get('data', {})
        batch = d.get('rows', [])
        rows.extend(batch)
        pages += 1
        offset = d.get('offset')
        if not offset or not batch or pages > 50:
            break
    return rows

def parse_dt(s):
    if not s or s in ('null', 'undefined', 'None', None):
        return None
    s = str(s).strip().replace('T', ' ').replace('Z', '')
    for fmt in ('%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M', '%Y-%m-%d'):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            continue
    return None

def mins(a, b):
    if not a or not b:
        return None
    return int((b - a).total_seconds() // 60)

# ── Load ──────────────────────────────────────────────────────────────────────
print('Fetching attendance / status / employees ...')
att      = fetch_all(TBL_ATTENDANCE)
status   = fetch_all(TBL_STATUS)
emps     = fetch_all(TBL_EMPLOYEES)
print(f'  attendance={len(att)}  status={len(status)}  employees={len(emps)}')

# keka_id -> name, joining date
name_by_kid = {}
join_by_kid = {}
for e in emps:
    kid = e.get('name')           # 'name' col holds keka_id
    if kid:
        name_by_kid[kid] = e.get('display_name', '?')
for s in status:
    kid = s.get('keka_id')
    if kid:
        join_by_kid[kid] = parse_dt(s.get('joining_date'))

# ── Classify each attendance row ──────────────────────────────────────────────
clean   = []     # usable rows
suspect = []     # incomplete / suspicious rows

for row in att:
    kid       = row.get('name')                 # keka_id
    punch_in  = parse_dt(row.get('punch_in'))
    punch_out = parse_dt(row.get('punch_out'))
    shift_in  = parse_dt(row.get('shift_start_time'))
    shift_out = parse_dt(row.get('shift_end_time'))
    weekly_off= row.get('weekly_off_policy')

    anchor = punch_in or punch_out
    if not anchor:
        continue
    if anchor.month != TARGET_MONTH or anchor.year != TARGET_YEAR:
        continue

    work_date = anchor.date()
    reason = None

    # Incomplete / suspicious detection
    if not punch_in or not punch_out:
        # single punch — if it's late in the day, likely a forgotten morning punch
        lone = punch_in or punch_out
        if lone and lone.hour >= LONE_PUNCH_HOUR:
            reason = 'lone_after_hours_punch'
        else:
            reason = 'missing_punch'
    else:
        pair_min = mins(punch_in, punch_out)
        if pair_min is not None and pair_min < MIN_PAIR_MINUTES:
            reason = 'pair_too_short'

    if reason:
        suspect.append({
            'kid': kid, 'name': name_by_kid.get(kid, '?'),
            'date': work_date,
            'punch_in':  punch_in.strftime('%H:%M') if punch_in else '--',
            'punch_out': punch_out.strftime('%H:%M') if punch_out else '--',
            'reason': reason,
        })
        continue

    # Clean row — compute metrics
    late_min  = mins(shift_in, punch_in) if shift_in else None
    early_min = mins(punch_out, shift_out) if shift_out else None
    hours     = round((punch_out - punch_in).total_seconds() / 3600, 2)

    clean.append({
        'kid': kid, 'name': name_by_kid.get(kid, '?'),
        'date': work_date,
        'punch_in': punch_in.strftime('%H:%M'),
        'punch_out': punch_out.strftime('%H:%M'),
        'shift_in': shift_in.strftime('%H:%M') if shift_in else '--',
        'shift_out': shift_out.strftime('%H:%M') if shift_out else '--',
        'late_min':  late_min if (late_min and late_min > 0) else 0,
        'early_min': early_min if (early_min and early_min > 0) else 0,
        'hours': hours,
        'is_late':       bool(late_min is not None and late_min > LATE_GRACE_MIN),
        'is_early_exit': bool(early_min is not None and early_min > EARLY_EXIT_MIN),
        'weekly_off': weekly_off,
    })

print(f'\nClean rows: {len(clean)}   Suspicious/incomplete: {len(suspect)}')

# ── Working-day helper ────────────────────────────────────────────────────────
def working_days(start_d, end_d):
    if start_d > end_d:
        return 0
    n, d = 0, start_d
    while d <= end_d:
        if d.weekday() != 6:    # exclude Sunday
            n += 1
        d += timedelta(days=1)
    return n

month_start = datetime(TARGET_YEAR, TARGET_MONTH, 1).date()
month_end = (datetime(TARGET_YEAR, TARGET_MONTH + 1, 1) - timedelta(days=1)).date() \
            if TARGET_MONTH < 12 else datetime(TARGET_YEAR, 12, 31).date()

# ── Per-employee summary ──────────────────────────────────────────────────────
by_emp = defaultdict(list)
for c in clean:
    by_emp[c['kid']].append(c)

# also bucket suspects per employee for visibility
suspect_by_emp = defaultdict(list)
for s in suspect:
    suspect_by_emp[s['kid']].append(s)

print('\n' + '=' * 100)
print(f'  MONTHLY SUMMARY — {TARGET_MONTH}/{TARGET_YEAR}   (Sundays excluded, incompletes separated)')
print(f'  Late > {LATE_GRACE_MIN}min  |  Early-exit > {EARLY_EXIT_MIN}min')
print('=' * 100)

for kid, days in sorted(by_emp.items(), key=lambda x: name_by_kid.get(x[0], '?')):
    name = name_by_kid.get(kid, '?')

    join_dt = join_by_kid.get(kid)
    eff_start = month_start
    joiner = ''
    if join_dt and join_dt.date() > month_start:
        eff_start = join_dt.date()
        joiner = f'  [joined {join_dt.date()}]'

    expected = working_days(eff_start, month_end)
    present  = len(set(d['date'] for d in days))         # UNIQUE dates only
    present  = min(present, expected)                     # cap — can't exceed expected
    late     = sum(1 for d in days if d['is_late'])
    early    = sum(1 for d in days if d['is_early_exit'])
    n_suspect= len(suspect_by_emp.get(kid, []))
    unrecorded = max(0, expected - present)
    valid_hours = [d['hours'] for d in days if d['hours']]
    avg_hours = round(sum(valid_hours) / len(valid_hours), 2) if valid_hours else 0

    print(f'\n{name}{joiner}')
    print(f'   Expected working days : {expected}')
    print(f'   Present (unique days) : {present}')
    print(f'   Late (> {LATE_GRACE_MIN}min)        : {late}')
    print(f'   Early exit (> {EARLY_EXIT_MIN}min)  : {early}')
    print(f'   Unrecorded/absent     : {unrecorded}')
    print(f'   Incomplete punches    : {n_suspect}')
    print(f'   Avg hours/day         : {avg_hours}')

# ── Suspicious rows listing ───────────────────────────────────────────────────
print('\n' + '=' * 100)
print(f'  SUSPICIOUS / INCOMPLETE ROWS  ({len(suspect)} total) — first 30')
print('=' * 100)
print(f'{"Name":<24}{"Date":<12}{"In":<7}{"Out":<7}{"Reason"}')
for s in suspect[:30]:
    print(f'{s["name"][:23]:<24}{str(s["date"]):<12}{s["punch_in"]:<7}{s["punch_out"]:<7}{s["reason"]}')

# ── Reason breakdown ──────────────────────────────────────────────────────────
reason_counts = defaultdict(int)
for s in suspect:
    reason_counts[s['reason']] += 1
print('\nIncomplete reason breakdown:')
for r, c in sorted(reason_counts.items(), key=lambda x: -x[1]):
    print(f'   {r:<26}: {c}')

# ── Clean sample for hand-verification ────────────────────────────────────────
print('\n' + '=' * 100)
print('  CLEAN DAILY SAMPLE (first 15) — verify late/early by hand')
print('=' * 100)
print(f'{"Name":<22}{"Date":<12}{"In":<7}{"SIn":<7}{"Late":<6}{"Out":<7}{"SOut":<7}{"Early":<7}{"Hrs":<6}{"Flags"}')
for c in clean[:15]:
    flags = ','.join([x for x, on in [('LATE', c['is_late']), ('EARLY', c['is_early_exit'])] if on])
    print(f'{c["name"][:21]:<22}{str(c["date"]):<12}{c["punch_in"]:<7}{c["shift_in"]:<7}'
          f'{c["late_min"]:<6}{c["punch_out"]:<7}{c["shift_out"]:<7}{c["early_min"]:<7}'
          f'{str(c["hours"]):<6}{flags}')

# ── Active users count (for dashboard) ────────────────────────────────────────
active = sum(1 for s in status if str(s.get('employment_status')) == '0'
                              and str(s.get('account_status')) == '1')
print('\n' + '=' * 100)
print(f'  ACTIVE USERS (employment_status=0 AND account_status=1): {active}')
print(f'  Total employees in system: {len(emps)}')
print('=' * 100)

print('\nDONE — read only. Review names, incomplete handling, present<=expected, then we enable writeback.')

Fetching attendance / status / employees ...
  attendance=2639  status=572  employees=286

Clean rows: 1846   Suspicious/incomplete: 240

  MONTHLY SUMMARY — 5/2026   (Sundays excluded, incompletes separated)
  Late > 20min  |  Early-exit > 10min

Aarti Jagtap
   Expected working days : 26
   Present (unique days) : 23
   Late (> 20min)        : 1
   Early exit (> 10min)  : 1
   Unrecorded/absent     : 3
   Incomplete punches    : 2
   Avg hours/day         : 6.71

Aastha Sadhwani
   Expected working days : 26
   Present (unique days) : 22
   Late (> 20min)        : 4
   Early exit (> 10min)  : 0
   Unrecorded/absent     : 4
   Incomplete punches    : 4
   Avg hours/day         : 8.73

Aastha trivedi
   Expected working days : 26
   Present (unique days) : 21
   Late (> 20min)        : 1
   Early exit (> 10min)  : 3
   Unrecorded/absent     : 5
   Incomplete punches    : 3
   Avg hours/day         : 8.29

Abhishek Vishwakarma 
   Expected working days : 26
   Present (unique days) : 7


In [6]:
"""
KEKA ATTENDANCE — UI DATA SCRIPT v4
====================================
Produces the JSON shape the Scorecard UI needs, per employee, per month:
  - present / late / early / absent / avg hours
  - correct denominator for mid-month JOINERS and LEAVERS
  - systemic punch-machine days excluded company-wide
  - modal (most common) shift time per person, not per-row
  - department grouping for filtering

READ ONLY in this version — prints JSON to console. Wire into Flask once
the numbers are confirmed correct against what you see in Keka's own UI
for 2-3 known employees.

=====================================================================
LOGIC REFERENCE — every rule used below, explained
=====================================================================

1. WORKING DAYS DENOMINATOR (mid-month joiners AND leavers)
   -----------------------------------------------------------------
   Expected days is NOT always the full month. It's clipped on both ends:

     effective_start = max(month_start, joining_date)
     effective_end   = min(month_end,   exit_date or month_end)

   Why: an employee who joined on the 20th cannot be scored against 26
   working days — they were only employable for ~6 of them. Same logic
   applies symmetrically to someone who exits mid-month: days after
   their exit_date aren't theirs to be "absent" on.

   exit_date comes from keka_employee_status. If exit_date is null,
   effective_end = month_end (still employed).

2. SYSTEMIC PUNCH-MACHINE DAYS (company-wide, not per-employee)
   -----------------------------------------------------------------
   Some dates (e.g. month-start/month-end) show a spike of incomplete
   punches across MANY employees simultaneously — a hardware/system
   issue, not individual absence. Detected by:

     for each date, incomplete_ratio = incomplete_rows / total_rows
     if incomplete_ratio >= 30% AND total_rows >= 10 employees
       -> mark that date as SYSTEMIC

   Systemic dates are removed from the working-day denominator for
   EVERY employee that month — nobody is penalized for a system outage
   that wasn't their fault. This must run as a first pass across ALL
   employees before any individual scoring happens.

3. INCOMPLETE PUNCH ROW (per row, before any time math)
   -----------------------------------------------------------------
   A row is incomplete if ANY of:
     a) punch_in is missing OR punch_out is missing
     b) only one punch exists and it's after 14:00 -> almost certainly
        a forgotten morning punch, not a real "arrived at 7pm" event
     c) both punches exist but are < 60 minutes apart -> likely two
        unrelated punches (e.g. door badge + system) paired by mistake,
        not a real shift

   Incomplete rows are excluded from late/early/hours math entirely.
   They are NOT counted as present, but they ARE distinguished from a
   true absence in the data (shown separately) so HR can tell "system
   didn't capture it" from "didn't show up at all".

4. MODAL SHIFT TIME (per person, not per row)
   -----------------------------------------------------------------
   Per-row shift_start_time/shift_end_time can occasionally be wrong
   (sync glitches). Instead of trusting each row individually, we look
   at ALL of a person's clean rows for the month and take the MOST
   COMMON shift_start and shift_end values. That single stable pair is
   then used for every late/early calculation for that person, so one
   bad row's shift time doesn't distort their whole month.

5. LATE / EARLY THRESHOLDS
   -----------------------------------------------------------------
     late       = punch_in is more than 20 minutes after modal shift start
     early_exit = punch_out is more than 10 minutes before modal shift end
   (UI may DISPLAY "15 min grace" as a simplified figure to employees —
   that's a display choice only; the actual computed threshold is 20.)

6. IMPLAUSIBLE-HOURS ROW (after incomplete filter, before late/early)
   -----------------------------------------------------------------
   A row can pass the "incomplete" check (both punches exist, >60min
   apart) but still produce abnormally low total hours (e.g. 2-3h on a
   normal day) — this usually means the punch_in and punch_out belong
   to two different sessions that got mismatched in the source sync.

     if total_hours < 3.0 -> exclude from hours/late/early stats
     BUT still count the date as PRESENT (they did show up)

7. DEPARTMENT GROUPING
   -----------------------------------------------------------------
   keka_employee_groups holds a row per (employee, group) membership.
   We only use rows where group_type = 2 (Department) — the same table
   also holds Location, PayGroup, and BusinessUnit memberships under
   different group_type values, which are NOT department and must be
   filtered out before grouping for the UI's department filter.

   *** Pending: re-fetch this table if you've corrected its data ***

8. ACTIVE USERS (for dashboard headline number)
   -----------------------------------------------------------------
     active = employment_status == 0 AND account_status == 1
   employment_status: 0 = Active, 1 = Inactive
   account_status:    0 = Inactive, 1 = Active (login account state)
   Both must be true — an employee can be employment-active but have a
   deactivated login, or vice versa during offboarding transition.
"""

import requests
from datetime import datetime, timedelta
from collections import defaultdict, Counter
import json

# ── Config ────────────────────────────────────────────────────────────────────
AUTH_KEY = 'keywWPoCKnwcJAA'
DB_ID    = '6a17ec96a95e7ac45342c0e4'
BASE     = f'https://table-api.viasocket.com/{DB_ID}'
HEADERS  = {'auth-key': AUTH_KEY}

TBL_ATTENDANCE = 'tblr5wgh0'
TBL_STATUS     = 'tbll6kzp6'
TBL_EMPLOYEES  = 'tblrj1w62'
TBL_GROUPS     = 'tblto5irg'   # department membership

LATE_GRACE_MIN     = 20
EARLY_EXIT_MIN     = 10
LONE_PUNCH_HOUR    = 14
MIN_PAIR_MINUTES   = 60
MIN_PLAUSIBLE_HRS  = 3.0
SYSTEMIC_DAY_RATIO = 0.30
SYSTEMIC_MIN_ROWS  = 10
DEPARTMENT_GROUP_TYPE = '2'   # filters keka_employee_groups to departments only

TARGET_MONTH = 5
TARGET_YEAR  = 2026

# ── Fetch with pagination ─────────────────────────────────────────────────────
def fetch_all(table_id):
    rows, offset, pages = [], None, 0
    while True:
        params = {'offset': offset} if offset else {}
        r = requests.get(f'{BASE}/{table_id}', headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        d = r.json().get('data', {})
        batch = d.get('rows', [])
        rows.extend(batch)
        pages += 1
        offset = d.get('offset')
        if not offset or not batch or pages > 50:
            break
    return rows

def parse_dt(s):
    if not s or s in ('null', 'undefined', 'None', None):
        return None
    s = str(s).strip().replace('T', ' ').replace('Z', '')
    for fmt in ('%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M', '%Y-%m-%d'):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            continue
    return None

def mins(a, b):
    if not a or not b:
        return None
    return int((b - a).total_seconds() // 60)

def working_days(start_d, end_d, exclude=set()):
    """Count Mon-Sat days in [start_d, end_d], minus any systemic-excluded dates."""
    if start_d > end_d:
        return 0
    n, d = 0, start_d
    while d <= end_d:
        if d.weekday() != 6 and d not in exclude:   # 6 = Sunday
            n += 1
        d += timedelta(days=1)
    return n

# ── Load all source tables ─────────────────────────────────────────────────────
print('Fetching attendance / status / employees / groups ...')
att     = fetch_all(TBL_ATTENDANCE)
status  = fetch_all(TBL_STATUS)
emps    = fetch_all(TBL_EMPLOYEES)
groups  = fetch_all(TBL_GROUPS)
print(f'  attendance={len(att)}  status={len(status)}  employees={len(emps)}  groups={len(groups)}')

name_by_kid = {e.get('name'): e.get('display_name', '?') for e in emps if e.get('name')}

join_by_kid = {}
exit_by_kid = {}
for s in status:
    kid = s.get('keka_id')
    if not kid:
        continue
    join_by_kid[kid] = parse_dt(s.get('joining_date'))
    ed = s.get('exit_date')
    exit_by_kid[kid] = parse_dt(ed) if ed not in (None, 'null') else None

# Department lookup — RULE 7: only group_type == Department, exclude Location/PayGroup/etc
dept_by_kid = {}
for g in groups:
    if str(g.get('group_type')) == DEPARTMENT_GROUP_TYPE:
        dept_by_kid[g.get('keka_id')] = g.get('dept_title', 'Unknown')

# ── Pass 1: classify every row, find systemic days (RULE 2) ────────────────────
raw_rows = []
for row in att:
    kid       = row.get('name')
    punch_in  = parse_dt(row.get('punch_in'))
    punch_out = parse_dt(row.get('punch_out'))
    shift_in  = parse_dt(row.get('shift_start_time'))
    shift_out = parse_dt(row.get('shift_end_time'))

    anchor = punch_in or punch_out
    if not anchor or anchor.month != TARGET_MONTH or anchor.year != TARGET_YEAR:
        continue
    work_date = anchor.date()

    # RULE 3: incomplete row detection
    incomplete = (not punch_in or not punch_out)
    if not incomplete:
        if (punch_in or punch_out).hour >= LONE_PUNCH_HOUR and (not punch_in or not punch_out):
            incomplete = True
        pair_min = mins(punch_in, punch_out)
        if pair_min is not None and pair_min < MIN_PAIR_MINUTES:
            incomplete = True

    raw_rows.append({
        'kid': kid, 'date': work_date,
        'punch_in': punch_in, 'punch_out': punch_out,
        'shift_in': shift_in, 'shift_out': shift_out,
        'incomplete': incomplete,
    })

# RULE 2: detect systemic days across ALL employees first
by_date = defaultdict(lambda: {'total': 0, 'incomplete': 0})
for r in raw_rows:
    by_date[r['date']]['total'] += 1
    if r['incomplete']:
        by_date[r['date']]['incomplete'] += 1

systemic_days = {
    d for d, c in by_date.items()
    if c['total'] >= SYSTEMIC_MIN_ROWS and (c['incomplete'] / c['total']) >= SYSTEMIC_DAY_RATIO
}
print(f'\nSystemic days excluded for everyone: {sorted(systemic_days)}')

# ── Pass 2: modal shift time per person (RULE 4), from clean non-systemic rows ─
clean_for_modal = defaultdict(list)
for r in raw_rows:
    if r['incomplete'] or r['date'] in systemic_days:
        continue
    if r['shift_in'] and r['shift_out']:
        clean_for_modal[r['kid']].append(r)

modal_shift = {}
for kid, rows in clean_for_modal.items():
    in_times  = Counter(r['shift_in'].strftime('%H:%M') for r in rows)
    out_times = Counter(r['shift_out'].strftime('%H:%M') for r in rows)
    modal_shift[kid] = (in_times.most_common(1)[0][0], out_times.most_common(1)[0][0])

# ── Pass 3: final per-row classification (RULES 5, 6) ──────────────────────────
final_rows = []
for r in raw_rows:
    kid = r['kid']
    if r['date'] in systemic_days:
        continue   # excluded entirely from this person's month

    if r['incomplete']:
        final_rows.append({**r, 'status': 'incomplete'})
        continue

    hours = round((r['punch_out'] - r['punch_in']).total_seconds() / 3600, 2)

    m_in_str, m_out_str = modal_shift.get(kid, (None, None))
    m_in  = datetime.combine(r['date'], datetime.strptime(m_in_str, '%H:%M').time()) if m_in_str else r['shift_in']
    m_out = datetime.combine(r['date'], datetime.strptime(m_out_str, '%H:%M').time()) if m_out_str else r['shift_out']

    if hours < MIN_PLAUSIBLE_HRS:
        # RULE 6: present, but excluded from hours/late/early
        final_rows.append({**r, 'status': 'present_implausible', 'hours': hours})
        continue

    late_min  = mins(m_in, r['punch_in'])
    early_min = mins(r['punch_out'], m_out)
    is_late       = bool(late_min is not None and late_min > LATE_GRACE_MIN)
    is_early_exit = bool(early_min is not None and early_min > EARLY_EXIT_MIN)

    final_rows.append({
        **r, 'status': 'present', 'hours': hours,
        'late_min': max(0, late_min or 0), 'early_min': max(0, early_min or 0),
        'is_late': is_late, 'is_early_exit': is_early_exit,
    })

# ── Build per-employee monthly UI record ───────────────────────────────────────
month_start = datetime(TARGET_YEAR, TARGET_MONTH, 1).date()
month_end = (datetime(TARGET_YEAR, TARGET_MONTH + 1, 1) - timedelta(days=1)).date() \
            if TARGET_MONTH < 12 else datetime(TARGET_YEAR, 12, 31).date()

by_emp = defaultdict(list)
for r in final_rows:
    by_emp[r['kid']].append(r)

ui_records = []
for kid, rows in by_emp.items():
    name = name_by_kid.get(kid, '?')

    # RULE 1: clip BOTH ends for joiners and leavers
    join_dt = join_by_kid.get(kid)
    exit_dt = exit_by_kid.get(kid)

    eff_start = month_start
    if join_dt and join_dt.date() > month_start:
        eff_start = join_dt.date()

    eff_end = month_end
    if exit_dt and exit_dt.date() < month_end:
        eff_end = exit_dt.date()

    expected = working_days(eff_start, eff_end, exclude=systemic_days)

    present_rows = [r for r in rows if r['status'] in ('present', 'present_implausible')]
    present = min(len(set(r['date'] for r in present_rows)), expected)

    late  = sum(1 for r in rows if r.get('is_late'))
    early = sum(1 for r in rows if r.get('is_early_exit'))
    incomplete_count = sum(1 for r in rows if r['status'] == 'incomplete')
    unrecorded = max(0, expected - present)

    valid_hours = [r['hours'] for r in rows if r['status'] == 'present' and r.get('hours')]
    avg_hours = round(sum(valid_hours) / len(valid_hours), 2) if valid_hours else 0

    m_in, m_out = modal_shift.get(kid, ('--', '--'))

    ui_records.append({
        'keka_id': kid,
        'name': name,
        'department': dept_by_kid.get(kid, 'Unknown'),
        'modal_shift_start': m_in,
        'modal_shift_end': m_out,
        'joined_mid_month': bool(join_dt and join_dt.date() > month_start),
        'joining_date': str(join_dt.date()) if join_dt else None,
        'exited_mid_month': bool(exit_dt and exit_dt.date() < month_end),
        'exit_date': str(exit_dt.date()) if exit_dt else None,
        'expected_working_days': expected,
        'present_days': present,
        'late_days': late,
        'early_exit_days': early,
        'unrecorded_absent_days': unrecorded,
        'incomplete_punch_days': incomplete_count,
        'avg_hours_per_day': avg_hours,
        'late_pct': round(100 * late / expected, 1) if expected else 0,
        'early_pct': round(100 * early / expected, 1) if expected else 0,
        'absent_pct': round(100 * unrecorded / expected, 1) if expected else 0,
    })

ui_records.sort(key=lambda r: r['name'])

# ── Active users (RULE 8) ───────────────────────────────────────────────────────
active = sum(1 for s in status if str(s.get('employment_status')) == '0'
                              and str(s.get('account_status')) == '1')

# ── Department summary (RULE 7) ────────────────────────────────────────────────
by_department = defaultdict(list)
for rec in ui_records:
    by_department[rec['department']].append(rec)

dept_summary = []
for dept, recs in by_department.items():
    avg_absent_pct = round(sum(r['absent_pct'] for r in recs) / len(recs), 1) if recs else 0
    avg_late_pct   = round(sum(r['late_pct'] for r in recs) / len(recs), 1) if recs else 0
    dept_summary.append({
        'department': dept,
        'employee_count': len(recs),
        'avg_absent_pct': avg_absent_pct,
        'avg_late_pct': avg_late_pct,
    })

# ── Output for UI ──────────────────────────────────────────────────────────────
output = {
    'month': TARGET_MONTH,
    'year': TARGET_YEAR,
    'systemic_days_excluded': [str(d) for d in sorted(systemic_days)],
    'active_users': active,
    'total_employees': len(emps),
    'department_summary': dept_summary,
    'employees': ui_records,
}

print('\n' + '=' * 80)
print(f'  UI DATA — {len(ui_records)} employee records  |  {len(dept_summary)} departments')
print('=' * 80)
print(json.dumps(output, indent=2)[:3000])
print('... (truncated for console — full structure is in the `output` variable)')

print(f'\nActive users: {active} / {len(emps)} total employees')
print(f'Departments found: {[d["department"] for d in dept_summary]}')

# Save full JSON to file for inspection
with open('keka_attendance_ui_data.json', 'w') as f:
    json.dump(output, f, indent=2)
print('\nFull output saved to keka_attendance_ui_data.json')
print('\nDONE — read only. Verify department names look correct, then this becomes the API endpoint.')

Fetching attendance / status / employees / groups ...
  attendance=3260  status=1144  employees=572  groups=572

Systemic days excluded for everyone: [datetime.date(2026, 5, 1), datetime.date(2026, 5, 30)]

  UI DATA — 81 employee records  |  9 departments
{
  "month": 5,
  "year": 2026,
  "systemic_days_excluded": [
    "2026-05-01",
    "2026-05-30"
  ],
  "active_users": 339,
  "total_employees": 572,
  "department_summary": [
    {
      "department": "Conversion",
      "employee_count": 53,
      "avg_absent_pct": 29.5,
      "avg_late_pct": 7.4
    },
    {
      "department": "Client communication",
      "employee_count": 9,
      "avg_absent_pct": 16.2,
      "avg_late_pct": 6.9
    },
    {
      "department": "Human Resources",
      "employee_count": 2,
      "avg_absent_pct": 37.5,
      "avg_late_pct": 25.0
    },
    {
      "department": "Marketing",
      "employee_count": 2,
      "avg_absent_pct": 22.9,
      "avg_late_pct": 29.1
    },
    {
      "department": "IT

In [7]:
"""
verify_before_switch.py
==========================
Run this BEFORE copying any of the 3 new files into place, and BEFORE
running switch_attendance_to_keka.py. It checks every assumption the
new code makes against your ACTUAL project files and database, without
modifying anything.

It will tell you exactly which assumptions are correct and which are
wrong, with the real values, so the new files can be corrected before
they're ever placed into kpis/ or used to touch the registry.

Run from your backend/ folder:
    python verify_before_switch.py
"""

import sqlite3
import os
import sys
import importlib.util

print('=' * 78)
print('  PRE-FLIGHT VERIFICATION — checking assumptions before any file is replaced')
print('=' * 78)

ok_count = 0
fail_count = 0

def check(label, passed, detail=''):
    global ok_count, fail_count
    mark = '✅' if passed else '❌'
    print(f'{mark}  {label}')
    if detail:
        print(f'      {detail}')
    if passed:
        ok_count += 1
    else:
        fail_count += 1


# ── 1. Does scorecard.db exist where expected? ─────────────────────────────────
print('\n--- Database file ---')
db_exists = os.path.exists('scorecard.db')
check('scorecard.db exists in current folder', db_exists,
      f'cwd={os.getcwd()}' if not db_exists else '')

if not db_exists:
    print('\nSTOP — run this script from the backend/ folder where scorecard.db lives.')
    sys.exit(1)

conn = sqlite3.connect('scorecard.db')


# ── 2. Does the kpis table exist, and what columns does it actually have? ──────
print('\n--- kpis table structure ---')
try:
    cols = [r[1] for r in conn.execute("PRAGMA table_info(kpis)").fetchall()]
    check('kpis table exists', len(cols) > 0, f'columns found: {cols}')

    for expected_col in ['key', 'module_path', 'class_name', 'is_active', 'raw_weight']:
        check(f"  column '{expected_col}' exists in kpis table", expected_col in cols)
except Exception as e:
    check('kpis table exists', False, f'error: {e}')


# ── 3. What are the ACTUAL current values for leaves/late_comings/early_leavings? ──
print('\n--- Current registry values (what we would be overwriting) ---')
actual_registry = {}
for key in ['leaves', 'late_comings', 'early_leavings']:
    try:
        row = conn.execute(
            "SELECT module_path, class_name, is_active FROM kpis WHERE key=?", (key,)
        ).fetchone()
        if row:
            actual_registry[key] = {'module_path': row[0], 'class_name': row[1], 'is_active': row[2]}
            check(f"'{key}' found in registry", True,
                  f'module_path={row[0]}  class_name={row[1]}  is_active={row[2]}')
        else:
            check(f"'{key}' found in registry", False, 'KEY NOT FOUND — check exact key spelling')
    except Exception as e:
        check(f"'{key}' found in registry", False, f'error: {e}')


# ── 4. Does kpis/base.py exist, and does it define BaseKPI with fetch/aggregate? ──
print('\n--- kpis/base.py (BaseKPI class assumption) ---')
base_path = os.path.join('kpis', 'base.py')
base_exists = os.path.exists(base_path)
check('kpis/base.py exists', base_exists)

if base_exists:
    try:
        spec = importlib.util.spec_from_file_location('kpis.base', base_path)
        base_mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(base_mod)
        has_basekpi = hasattr(base_mod, 'BaseKPI')
        check('BaseKPI class exists in kpis/base.py', has_basekpi)
        if has_basekpi:
            cls = base_mod.BaseKPI
            has_fetch = hasattr(cls, 'fetch')
            has_aggregate = hasattr(cls, 'aggregate')
            check('BaseKPI defines/expects fetch()', has_fetch)
            check('BaseKPI defines/expects aggregate()', has_aggregate)
            # Check if these are abstract (must override) or have default impl
            import inspect
            try:
                fetch_src = inspect.getsource(cls.fetch)
                print(f'      fetch() source preview: {fetch_src.strip()[:150]}')
            except Exception:
                pass
    except Exception as e:
        check('kpis/base.py loads without error', False, f'error: {e}')
else:
    print('      Cannot verify BaseKPI — file not found at expected path.')
    print('      ACTION: tell me the real path/import if different.')


# ── 5. Do the OLD kpis/leaves.py, late_comings.py, early_leavings.py exist
#       and what does their fetch() row shape actually look like? ───────────────
print('\n--- Existing KPI files (to confirm row shape compatibility) ---')
for fname, expected_class in [
    ('leaves.py', 'LeavesKPI'),
    ('late_comings.py', 'LateComingsKPI'),
    ('early_leavings.py', 'EarlyLeavingsKPI'),
]:
    fpath = os.path.join('kpis', fname)
    exists = os.path.exists(fpath)
    check(f'kpis/{fname} exists', exists)
    if exists:
        with open(fpath) as f:
            content = f.read()
        has_class = expected_class in content
        check(f"  defines class '{expected_class}' (matches registry guess)", has_class)
        has_fetch_def = 'def fetch(' in content
        has_agg_def = 'def aggregate(' in content
        check(f'  has fetch() method', has_fetch_def)
        check(f'  has aggregate() method', has_agg_def)
        # Check what keys the fetch() rows actually use
        for field in ['AdminID', 'EmployeeName', 'Department']:
            check(f"  uses field '{field}'", field in content)


# ── 6. Can we import db.py and does execute_query exist (for SQL Admin lookup)? ──
print('\n--- db.py (needed for keka_name_mapper SQL Admin lookup) ---')
db_py_exists = os.path.exists('db.py')
check('db.py exists', db_py_exists)
if db_py_exists:
    with open('db.py') as f:
        db_content = f.read()
    check("db.py defines 'execute_query'", 'def execute_query' in db_content)
    check("db.py defines 'get_connection'", 'def get_connection' in db_content)


# ── 7. Check the Admin table query keka_name_mapper assumes ─────────────────────
print('\n--- SQL Server Admin table assumption (for name matching) ---')
try:
    sys.path.insert(0, '.')
    from db import execute_query
    test_sql = """
        SELECT TOP 3 ID_Admin AS AdminID,
               Admin_FirstName + ' ' + Admin_LastName AS AdminName,
               Flag_Active, Flag_Delete
        FROM Admin
    """
    rows = execute_query(test_sql)
    check('Admin table query succeeds (ID_Admin, Admin_FirstName, Admin_LastName columns exist)',
          len(rows) > 0, f'sample: {rows[0] if rows else "no rows"}')
except Exception as e:
    check('Admin table query succeeds', False, f'error: {e}')
    print('      This means keka_name_mapper.py needs its SQL adjusted before use.')


# ── 8. Reachability check for ViaSocket/Keka table API ─────────────────────────
print('\n--- Keka DBdash API reachability ---')
try:
    import requests
    r = requests.get(
        'https://table-api.viasocket.com/6a17ec96a95e7ac45342c0e4/tblrj1w62',
        headers={'auth-key': 'keywWPoCKnwcJAA'}, timeout=15
    )
    check('Keka employees table reachable', r.status_code == 200,
          f'status={r.status_code}')
    if r.status_code == 200:
        sample = r.json().get('data', {}).get('rows', [])
        if sample:
            check('Sample row has expected fields (name, display_name)',
                  'name' in sample[0] and 'display_name' in sample[0],
                  f'keys: {list(sample[0].keys())[:8]}')
except Exception as e:
    check('Keka employees table reachable', False, f'error: {e}')


# ── Summary ───────────────────────────────────────────────────────────────────
conn.close()
print('\n' + '=' * 78)
print(f'  RESULT: {ok_count} passed, {fail_count} failed')
print('=' * 78)

if fail_count == 0:
    print('\nAll assumptions verified. Safe to proceed with copying the new files.')
else:
    print('\nDO NOT copy the new files yet. Paste this full output back so the')
    print('failing assumptions can be corrected in the actual code first.')

  PRE-FLIGHT VERIFICATION — checking assumptions before any file is replaced

--- Database file ---
✅  scorecard.db exists in current folder

--- kpis table structure ---
✅  kpis table exists
      columns found: ['id', 'key', 'name', 'description', 'module_path', 'class_name', 'raw_weight', 'is_active', 'sort_order', 'updated_by', 'updated_at']
✅    column 'key' exists in kpis table
✅    column 'module_path' exists in kpis table
✅    column 'class_name' exists in kpis table
✅    column 'is_active' exists in kpis table
✅    column 'raw_weight' exists in kpis table

--- Current registry values (what we would be overwriting) ---
✅  'leaves' found in registry
      module_path=kpis.leaves  class_name=LeavesKPI  is_active=1
✅  'late_comings' found in registry
      module_path=kpis.late_comings  class_name=LateComingsKPI  is_active=1
✅  'early_leavings' found in registry
      module_path=kpis.early_leavings  class_name=EarlyLeavingsKPI  is_active=1

--- kpis/base.py (BaseKPI class assumpt

In [8]:
"""
diagnose_name_match.py
========================
Ekta Hirke is a CONFIRMED real AdminID (344) from earlier SQL Server work
this session, yet keka_name_mapper reported her as unresolved. That's
proof the matcher has a bug, not that the data is genuinely missing.

This script pulls the raw Admin list exactly as keka_name_mapper does,
and tests the match logic step-by-step against a few known names so we
can see exactly where it's failing.
"""

import sys
sys.path.insert(0, '.')
from db import execute_query
import difflib

def normalize(name):
    if not name:
        return ''
    return ' '.join(name.strip().lower().split())

print('=' * 70)
print('STEP 1 — Pull raw Admin rows the same way keka_name_mapper does')
print('=' * 70)
sql = """
    SELECT ID_Admin AS AdminID,
           Admin_FirstName + ' ' + Admin_LastName AS AdminName
    FROM Admin
    WHERE Flag_Active = 1
      AND (Flag_Delete = 0 OR Flag_Delete IS NULL)
"""
admin_rows = execute_query(sql)
print(f'Total active admin rows returned: {len(admin_rows)}')

print('\nFirst 10 raw rows AS RETURNED (look closely at AdminName formatting):')
for r in admin_rows[:10]:
    print(f'  AdminID={r["AdminID"]:<6} AdminName={repr(r["AdminName"])}')

print('\n' + '=' * 70)
print('STEP 2 — Search for "Ekta" anywhere in the admin list')
print('=' * 70)
ekta_matches = [r for r in admin_rows if 'ekta' in (r['AdminName'] or '').lower()]
if ekta_matches:
    for r in ekta_matches:
        print(f'  FOUND: AdminID={r["AdminID"]}  AdminName={repr(r["AdminName"])}')
        print(f'         normalized={repr(normalize(r["AdminName"]))}')
else:
    print('  NOT FOUND — "Ekta" does not appear anywhere in the Admin query result.')
    print('  This means the SQL query itself is the problem (wrong filter, wrong table).')

print('\n' + '=' * 70)
print('STEP 3 — Test normalize() against the Keka display_name for Ekta')
print('=' * 70)
keka_name = 'Ekta Hirke'   # as it appeared in the unresolved list
print(f'Keka name (raw):        {repr(keka_name)}')
print(f'Keka name (normalized): {repr(normalize(keka_name))}')

if ekta_matches:
    admin_name = ekta_matches[0]['AdminName']
    print(f'Admin name (raw):        {repr(admin_name)}')
    print(f'Admin name (normalized): {repr(normalize(admin_name))}')
    print(f'Exact match after normalize? {normalize(keka_name) == normalize(admin_name)}')
    ratio = difflib.SequenceMatcher(None, normalize(keka_name), normalize(admin_name)).ratio()
    print(f'Fuzzy ratio: {ratio:.4f}  (threshold is 0.92)')

print('\n' + '=' * 70)
print('STEP 4 — Check raw Keka employee row shape (does display_name exist cleanly?)')
print('=' * 70)
import requests
r = requests.get(
    'https://table-api.viasocket.com/6a17ec96a95e7ac45342c0e4/tblrj1w62',
    headers={'auth-key': 'keywWPoCKnwcJAA'}, timeout=20
)
emps = r.json().get('data', {}).get('rows', [])
ekta_keka = [e for e in emps if 'ekta' in (e.get('display_name') or '').lower()]
if ekta_keka:
    for e in ekta_keka:
        print(f'  Keka row: name(keka_id)={e.get("name")}')
        print(f'            display_name={repr(e.get("display_name"))}')
        print(f'            first_name={repr(e.get("first_name"))}  last_name={repr(e.get("last_name"))}')
else:
    print('  Ekta not found in raw Keka employees table either — check the table ID.')

print('\n' + '=' * 70)
print('STEP 5 — Run the ACTUAL match_one logic from keka_name_mapper, with prints')
print('=' * 70)
if ekta_keka:
    test_name = ekta_keka[0].get('display_name', '')
    norm_test = normalize(test_name)
    print(f'Testing match for: {repr(test_name)} -> normalized: {repr(norm_test)}')

    exact = [a for a in admin_rows if normalize(a['AdminName']) == norm_test]
    print(f'Exact matches found: {len(exact)}')
    for a in exact:
        print(f'   {a}')

    if len(exact) == 0:
        print('\n  No exact match. Checking fuzzy candidates above 0.80 (lower bar, for diagnosis):')
        scored = []
        for a in admin_rows:
            ratio = difflib.SequenceMatcher(None, norm_test, normalize(a['AdminName'])).ratio()
            if ratio >= 0.80:
                scored.append((ratio, a))
        scored.sort(key=lambda x: -x[0])
        for ratio, a in scored[:5]:
            print(f'   ratio={ratio:.4f}  AdminID={a["AdminID"]}  AdminName={repr(a["AdminName"])}')
        if not scored:
            print('   No candidates even above 0.80 — names are structurally very different.')

print('\n' + '=' * 70)
print('STEP 6 — Total admin_rows count sanity check')
print('=' * 70)
print(f'admin_rows fetched: {len(admin_rows)}')
print('If this number looks too LOW (e.g. under 100) compared to your real')
print('employee count, the Flag_Active/Flag_Delete filter may be excluding')
print('people who are still valid for matching purposes.')

STEP 1 — Pull raw Admin rows the same way keka_name_mapper does
Total active admin rows returned: 547

First 10 raw rows AS RETURNED (look closely at AdminName formatting):
  AdminID=1      AdminName='Super Admin'
  AdminID=17     AdminName='Ankit Mehta'
  AdminID=19     AdminName='Preeti  Shinde'
  AdminID=25     AdminName='Sandeep Vishwakarma'
  AdminID=28     AdminName='Michael Penning'
  AdminID=29     AdminName='Dale Dixon'
  AdminID=49     AdminName='Daniel Lesmond'
  AdminID=51     AdminName='Geoffrey Pyman'
  AdminID=54     AdminName='Paul Goodison'
  AdminID=55     AdminName='Luke  Sorbara'

STEP 2 — Search for "Ekta" anywhere in the admin list
  NOT FOUND — "Ekta" does not appear anywhere in the Admin query result.
  This means the SQL query itself is the problem (wrong filter, wrong table).

STEP 3 — Test normalize() against the Keka display_name for Ekta
Keka name (raw):        'Ekta Hirke'
Keka name (normalized): 'ekta hirke'

STEP 4 — Check raw Keka employee row shape (do

In [9]:
"""
find_ekta_by_id.py
====================
We KNOW Ekta Hirke = AdminID 344 from earlier SQL Server work this session
(her file MMC26043034490400 was directly queried and confirmed multiple
times). She is real, active, and currently working. Yet the
Flag_Active=1 AND (Flag_Delete=0 OR Flag_Delete IS NULL) filter excluded
her entirely from the 547 rows returned.

This script looks her up directly by ID, with NO filter at all, to see
her actual flag values and prove exactly which condition is wrong.
"""

import sys
sys.path.insert(0, '.')
from db import execute_query

print('=' * 70)
print('Looking up AdminID 344 with NO filters at all')
print('=' * 70)
sql_no_filter = """
    SELECT ID_Admin, Admin_FirstName, Admin_LastName,
           Flag_Active, Flag_Delete
    FROM Admin
    WHERE ID_Admin = 344
"""
rows = execute_query(sql_no_filter)
if rows:
    for r in rows:
        print(f'  {r}')
else:
    print('  AdminID 344 not found AT ALL in the Admin table — different problem entirely.')

print('\n' + '=' * 70)
print('Checking the actual DISTINCT flag value combinations across the whole table')
print('=' * 70)
sql_distinct = """
    SELECT Flag_Active, Flag_Delete, COUNT(*) AS cnt
    FROM Admin
    GROUP BY Flag_Active, Flag_Delete
    ORDER BY cnt DESC
"""
combos = execute_query(sql_distinct)
for c in combos:
    print(f'  Flag_Active={c["Flag_Active"]}  Flag_Delete={c["Flag_Delete"]}  count={c["cnt"]}')

print('\n' + '=' * 70)
print('Total Admin rows with ZERO filters (true total)')
print('=' * 70)
total = execute_query("SELECT COUNT(*) AS cnt FROM Admin")
print(f'  Total rows in Admin table: {total[0]["cnt"]}')

Looking up AdminID 344 with NO filters at all
  {'ID_Admin': 344, 'Admin_FirstName': 'Ekta', 'Admin_LastName': 'Hirke', 'Flag_Active': False, 'Flag_Delete': False}

Checking the actual DISTINCT flag value combinations across the whole table
  Flag_Active=True  Flag_Delete=False  count=547
  Flag_Active=False  Flag_Delete=False  count=272
  Flag_Active=False  Flag_Delete=True  count=115

Total Admin rows with ZERO filters (true total)
  Total rows in Admin table: 934


In [10]:
"""
diagnose_coverage_gap.py
===========================
572 total Keka employees, but only 81 have ANY May attendance row, and
of those only 50 got an AdminID match. This script breaks down exactly
where people are falling out, so we know whether 50 is a real number or
a bug.
"""

import sys
sys.path.insert(0, '.')
import requests
from datetime import datetime

AUTH_KEY = 'keywWPoCKnwcJAA'
DB_ID    = '6a17ec96a95e7ac45342c0e4'
BASE     = f'https://table-api.viasocket.com/{DB_ID}'
HEADERS  = {'auth-key': AUTH_KEY}

def fetch_all(table_id):
    rows, offset, pages = [], None, 0
    while True:
        params = {'offset': offset} if offset else {}
        r = requests.get(f'{BASE}/{table_id}', headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        d = r.json().get('data', {})
        batch = d.get('rows', [])
        rows.extend(batch)
        pages += 1
        offset = d.get('offset')
        if not offset or not batch or pages > 50:
            break
    return rows

print('Fetching employees, attendance, status...')
emps = fetch_all('tblrj1w62')
att  = fetch_all('tblr5wgh0')
status = fetch_all('tbll6kzp6')

print(f'\nTotal Keka employees: {len(emps)}')
print(f'Total attendance rows (all months): {len(att)}')

# How many DISTINCT employees have ANY attendance row in May 2026 at all?
kids_with_may_attendance = set()
for row in att:
    pin = row.get('punch_in')
    pout = row.get('punch_out')
    raw = pin or pout
    if not raw or raw in ('null', None):
        continue
    s = str(raw).replace('T', ' ').replace('Z', '')
    try:
        dt = datetime.strptime(s[:19], '%Y-%m-%d %H:%M:%S')
    except ValueError:
        continue
    if dt.month == 5 and dt.year == 2026:
        kids_with_may_attendance.add(row.get('name'))

print(f'Distinct employees with >=1 May attendance row: {len(kids_with_may_attendance)}')

# Cross-check against employment_status
active_kids = set()
for s in status:
    if str(s.get('employment_status')) == '0' and str(s.get('account_status')) == '1':
        active_kids.add(s.get('keka_id'))

print(f'Distinct employees marked active in status table: {len(active_kids)}')

overlap = kids_with_may_attendance & active_kids
print(f'Active employees WITH May attendance: {len(overlap)}')
print(f'Active employees WITHOUT any May attendance: {len(active_kids - kids_with_may_attendance)}')
print(f'Has May attendance but NOT marked active: {len(kids_with_may_attendance - active_kids)}')

# Show a sample of active employees who have ZERO may attendance — are these real gaps?
missing = list(active_kids - kids_with_may_attendance)[:15]
name_by_kid = {e.get('name'): e.get('display_name','?') for e in emps if e.get('name')}
print(f'\nSample of ACTIVE employees with ZERO May attendance rows (first 15):')
for kid in missing:
    print(f'  {name_by_kid.get(kid, "?"):<30} keka_id={kid}')

print('\nIf these are real employees who should have punched in during May,')
print('this points to a gap in the Keka punch-machine sync itself, not in')
print('our derivation logic. If they are contractors/remote/non-punch roles,')
print('81 active-with-attendance is the correct, real population for this KPI.')

Fetching employees, attendance, status...

Total Keka employees: 572
Total attendance rows (all months): 4138
Distinct employees with >=1 May attendance row: 84
Distinct employees marked active in status table: 85
Active employees WITH May attendance: 70
Active employees WITHOUT any May attendance: 15
Has May attendance but NOT marked active: 14

Sample of ACTIVE employees with ZERO May attendance rows (first 15):
  Rashmi Yadav                   keka_id=2c0bf625-be6a-4b89-936f-1dd917c5b0f8
  Mohit singh tomar              keka_id=6a875a23-9a52-4c76-b4a7-eaac75a22f46
  Shubham Agrawal                keka_id=8a68183d-84d9-4731-9cfb-9354be1e448f
  Lakhan kori                    keka_id=0e8fa4c1-d04d-4484-a0e3-5e9a9a384757
  Ajay Malviya                   keka_id=cc874578-97a2-4df0-b9ef-b983eeb2fe05
  Shanu Mehta                    keka_id=ba648063-8894-4c14-83a5-b1e98153b514
  Shivam lodhi                   keka_id=06380524-742f-4169-b6ed-8ad5c87ed11b
  Aman Sheikh                    kek

In [11]:
"""
diagnose_active_count.py
===========================
Earlier full dump reported active_users=339. This run's diagnostic
reported only 85 marked active. Same formula (employment_status=='0'
AND account_status=='1'), same status table — so why two different
numbers? This script checks the raw value DISTRIBUTION for both fields
to find out, and also adds the requested second check: excluding anyone
whose exit_date has already passed, since employment_status alone might
lag behind a real departure (Ekta Hirke's case suggests status fields
can be stale/inconsistently set).
"""

import requests
from datetime import datetime
from collections import Counter

AUTH_KEY = 'keywWPoCKnwcJAA'
DB_ID    = '6a17ec96a95e7ac45342c0e4'
BASE     = f'https://table-api.viasocket.com/{DB_ID}'
HEADERS  = {'auth-key': AUTH_KEY}

def fetch_all(table_id):
    rows, offset, pages = [], None, 0
    while True:
        params = {'offset': offset} if offset else {}
        r = requests.get(f'{BASE}/{table_id}', headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        d = r.json().get('data', {})
        batch = d.get('rows', [])
        rows.extend(batch)
        pages += 1
        offset = d.get('offset')
        if not offset or not batch or pages > 50:
            break
    return rows

def parse_dt(s):
    if not s or s in ('null', 'undefined', 'None', None):
        return None
    s = str(s).strip().replace('T', ' ').replace('Z', '')
    try:
        return datetime.strptime(s[:19], '%Y-%m-%d %H:%M:%S')
    except ValueError:
        try:
            return datetime.strptime(s[:10], '%Y-%m-%d')
        except ValueError:
            return None

print('Fetching status table...')
status = fetch_all('tbll6kzp6')
print(f'Total status rows fetched: {len(status)}')

# ── Check for DUPLICATE keka_id rows in status table ──────────────────────────
kid_counts = Counter(s.get('keka_id') for s in status if s.get('keka_id'))
dupes = {k: c for k, c in kid_counts.items() if c > 1}
print(f'\nDistinct keka_ids in status table: {len(kid_counts)}')
print(f'keka_ids appearing MORE THAN ONCE: {len(dupes)}')
if dupes:
    print('Sample duplicates (first 5):')
    for kid, c in list(dupes.items())[:5]:
        print(f'  {kid}  appears {c} times')
        matching_rows = [s for s in status if s.get('keka_id') == kid]
        for r in matching_rows:
            print(f'      employment_status={r.get("employment_status")}  '
                  f'account_status={r.get("account_status")}  '
                  f'updatedat={r.get("updatedat")}  createdat={r.get("createdat")}')

# ── Raw value distribution for both fields ─────────────────────────────────────
print(f'\nemployment_status raw value distribution:')
emp_dist = Counter(str(s.get('employment_status')) for s in status)
for val, cnt in emp_dist.most_common():
    print(f'  {repr(val):<10} count={cnt}')

print(f'\naccount_status raw value distribution:')
acc_dist = Counter(str(s.get('account_status')) for s in status)
for val, cnt in acc_dist.most_common():
    print(f'  {repr(val):<10} count={cnt}')

# ── Recompute active count plainly, two ways ────────────────────────────────────
print(f'\n--- Recompute 1: naive, counting ALL rows (including duplicates) ---')
active_naive = sum(1 for s in status if str(s.get('employment_status')) == '0'
                                     and str(s.get('account_status')) == '1')
print(f'  active_naive = {active_naive}')

print(f'\n--- Recompute 2: DISTINCT keka_id, keep only LATEST row per person ---')
# If a keka_id has multiple rows (re-synced multiple times), keep the most
# recently updated/created one as the source of truth
latest_by_kid = {}
for s in status:
    kid = s.get('keka_id')
    if not kid:
        continue
    ts = s.get('updatedat') or s.get('createdat') or ''
    if kid not in latest_by_kid or ts > latest_by_kid[kid].get('_ts', ''):
        latest_by_kid[kid] = {**s, '_ts': ts}

active_distinct = sum(1 for s in latest_by_kid.values()
                       if str(s.get('employment_status')) == '0'
                       and str(s.get('account_status')) == '1')
print(f'  Distinct people (latest row each): {len(latest_by_kid)}')
print(f'  active_distinct = {active_distinct}')

# ── Requested: ALSO exclude anyone whose exit_date has already passed ──────────
print(f'\n--- Recompute 3: active AND (no exit_date OR exit_date is in the future) ---')
today = datetime.now()
active_with_exit_check = 0
excluded_by_exit = []
for s in latest_by_kid.values():
    is_active_flags = (str(s.get('employment_status')) == '0'
                        and str(s.get('account_status')) == '1')
    if not is_active_flags:
        continue
    exit_dt = parse_dt(s.get('exit_date'))
    if exit_dt and exit_dt <= today:
        excluded_by_exit.append((s.get('keka_id'), exit_dt))
        continue
    active_with_exit_check += 1

print(f'  active_with_exit_check = {active_with_exit_check}')
print(f'  Excluded because exit_date already passed: {len(excluded_by_exit)}')
for kid, ed in excluded_by_exit[:10]:
    print(f'    {kid}  exit_date={ed.date()}')

print(f'\n=== SUMMARY ===')
print(f'Naive (all rows, no dedup):              {active_naive}')
print(f'Distinct people (dedup by latest row):   {active_distinct}')
print(f'Distinct + exit_date safety check:       {active_with_exit_check}')

Fetching status table...
Total status rows fetched: 1144

Distinct keka_ids in status table: 286
keka_ids appearing MORE THAN ONCE: 286
Sample duplicates (first 5):
  f131410e-74dc-4648-b594-83443ae10021  appears 4 times
      employment_status=0  account_status=1  updatedat=None  createdat=2026-06-18T06:26:49.922Z
      employment_status=0  account_status=1  updatedat=None  createdat=2026-06-19T06:26:04.833Z
      employment_status=0  account_status=1  updatedat=None  createdat=2026-06-20T06:26:10.916Z
      employment_status=0  account_status=1  updatedat=None  createdat=2026-06-21T06:26:18.668Z
  f26e1582-6c0a-40e3-a085-0108cb1c70aa  appears 4 times
      employment_status=0  account_status=1  updatedat=None  createdat=2026-06-18T06:26:50.404Z
      employment_status=0  account_status=1  updatedat=None  createdat=2026-06-19T06:26:07.963Z
      employment_status=0  account_status=1  updatedat=None  createdat=2026-06-20T06:26:13.438Z
      employment_status=0  account_status=1  update

In [12]:
"""
check_employees_dupes.py
===========================
keka_employee_status had 4x duplicate rows per person. Before trusting
total_employees=len(emps) anywhere, confirm whether keka_employees has
the same issue or is actually clean (upserted correctly on sync).
"""

import requests
from collections import Counter

AUTH_KEY = 'keywWPoCKnwcJAA'
DB_ID    = '6a17ec96a95e7ac45342c0e4'
BASE     = f'https://table-api.viasocket.com/{DB_ID}'
HEADERS  = {'auth-key': AUTH_KEY}

def fetch_all(table_id):
    rows, offset, pages = [], None, 0
    while True:
        params = {'offset': offset} if offset else {}
        r = requests.get(f'{BASE}/{table_id}', headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        d = r.json().get('data', {})
        batch = d.get('rows', [])
        rows.extend(batch)
        pages += 1
        offset = d.get('offset')
        if not offset or not batch or pages > 50:
            break
    return rows

for table_name, table_id in [
    ('keka_employees', 'tblrj1w62'),
    ('keka_employee_groups', 'tblto5irg'),
    ('keka_attendance', 'tblr5wgh0'),
]:
    rows = fetch_all(table_id)
    key_field = 'name'   # keka_id is stored in the 'name' column in all 3 tables
    counts = Counter(r.get(key_field) for r in rows if r.get(key_field))
    dupes = {k: c for k, c in counts.items() if c > 1}
    print(f'{table_name}:')
    print(f'  total rows: {len(rows)}')
    print(f'  distinct {key_field}: {len(counts)}')
    print(f'  duplicated {key_field}s: {len(dupes)}')
    if dupes:
        sample_kid = list(dupes.keys())[0]
        print(f'  sample duplicate count: {dupes[sample_kid]}x for {sample_kid}')
    print()

keka_employees:
  total rows: 572
  distinct name: 286
  duplicated names: 286
  sample duplicate count: 2x for f131410e-74dc-4648-b594-83443ae10021

keka_employee_groups:
  total rows: 572
  distinct name: 0
  duplicated names: 0

keka_attendance:
  total rows: 4138
  distinct name: 97
  duplicated names: 97
  sample duplicate count: 69x for f26e1582-6c0a-40e3-a085-0108cb1c70aa



In [13]:
"""
investigate_attendance_dupes.py
==================================
keka_attendance showed 69x duplicates for one person across 4138 total
rows / 97 distinct people. Need to know: are these EXACT duplicate rows
(same date, same punch times, safe to dedup by date+punch_in), or does
each "duplicate" actually represent a DIFFERENT real punch event that
legitimately needs separate handling? Also re-checks keka_employee_groups
with the correct key field (keka_id, not name).
"""

import requests
from collections import Counter, defaultdict

AUTH_KEY = 'keywWPoCKnwcJAA'
DB_ID    = '6a17ec96a95e7ac45342c0e4'
BASE     = f'https://table-api.viasocket.com/{DB_ID}'
HEADERS  = {'auth-key': AUTH_KEY}

def fetch_all(table_id):
    rows, offset, pages = [], None, 0
    while True:
        params = {'offset': offset} if offset else {}
        r = requests.get(f'{BASE}/{table_id}', headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        d = r.json().get('data', {})
        batch = d.get('rows', [])
        rows.extend(batch)
        pages += 1
        offset = d.get('offset')
        if not offset or not batch or pages > 50:
            break
    return rows

print('=' * 70)
print('PART 1 — keka_attendance: are the 69 "duplicates" real dupes?')
print('=' * 70)
att = fetch_all('tblr5wgh0')

target_kid = 'f26e1582-6c0a-40e3-a085-0108cb1c70aa'   # Rashmi Rajput, the 69x case
rows_for_target = [r for r in att if r.get('name') == target_kid]
print(f'Total rows for this person: {len(rows_for_target)}')

# Group by (punch_in, punch_out) to see if they're truly identical
exact_dupe_check = Counter((r.get('punch_in'), r.get('punch_out')) for r in rows_for_target)
print(f'\nDistinct (punch_in, punch_out) combos: {len(exact_dupe_check)}')
print('Top 5 most repeated exact combos:')
for combo, cnt in exact_dupe_check.most_common(5):
    print(f'  punch_in={combo[0]}  punch_out={combo[1]}  -> appears {cnt} times')

# Also group by just the rowid's createdat to see the sync pattern
print(f'\nSample of 10 raw rows for this person (rowid, createdat, punch_in):')
for r in rows_for_target[:10]:
    print(f'  rowid={r.get("rowid")}  createdat={r.get("createdat")}  punch_in={r.get("punch_in")}')

# Distinct actual calendar dates this person has ANY row for
distinct_dates = set()
for r in rows_for_target:
    pin = r.get('punch_in') or r.get('punch_out') or ''
    if pin and pin not in ('null', 'undefined'):
        distinct_dates.add(str(pin)[:10])
print(f'\nDistinct calendar dates represented (regardless of duplicates): {len(distinct_dates)}')
print(f'Total raw rows: {len(rows_for_target)}')
print(f'Average rows per actual date: {len(rows_for_target)/len(distinct_dates):.1f}' if distinct_dates else 'N/A')

print('\n' + '=' * 70)
print('PART 2 — keka_employee_groups: re-check with correct key field')
print('=' * 70)
groups = fetch_all('tblto5irg')
print(f'Sample raw row to find correct key field:')
if groups:
    print(f'  {groups[0]}')

for key_field in ['keka_id', 'name']:
    counts = Counter(g.get(key_field) for g in groups if g.get(key_field))
    dupes = {k: c for k, c in counts.items() if c > 1}
    print(f'\nUsing key_field="{key_field}":')
    print(f'  distinct values: {len(counts)}')
    print(f'  duplicated: {len(dupes)}')

PART 1 — keka_attendance: are the 69 "duplicates" real dupes?
Total rows for this person: 69

Distinct (punch_in, punch_out) combos: 46
Top 5 most repeated exact combos:
  punch_in=2026-05-21 19:03:58  punch_out=null  -> appears 3 times
  punch_in=2026-05-22 10:46:47  punch_out=2026-05-22 19:02:21  -> appears 3 times
  punch_in=2026-05-23 10:51:42  punch_out=2026-05-23 19:02:18  -> appears 3 times
  punch_in=2026-05-25 19:07:42  punch_out=null  -> appears 3 times
  punch_in=2026-05-26 10:50:46  punch_out=2026-05-26 17:34:00  -> appears 3 times

Sample of 10 raw rows for this person (rowid, createdat, punch_in):
  rowid=rowbfux65wuk  createdat=2026-06-18T10:11:53.090Z  punch_in=2026-04-02 10:39:17
  rowid=rowdo2wly4sa  createdat=2026-06-18T10:11:53.098Z  punch_in=2026-04-01 10:49:23
  rowid=rowog3x5h6hr  createdat=2026-06-18T10:11:53.850Z  punch_in=2026-04-03 10:35:18
  rowid=rows54u6346v  createdat=2026-06-18T10:11:54.698Z  punch_in=2026-04-04 10:57:33
  rowid=row13j1pzqz5  createdat=2

In [14]:
"""
full_diagnostic.py
=====================
One script, one run, full visibility. For EVERY Keka employee, shows:
  - their normalized name
  - whether the SQL Admin table has an exact-normalized match
  - whether a fuzzy match exists and at what ratio
  - the FINAL verdict and why

No caching, no incremental state, no weekly-refresh logic — just a
direct, complete pass so we can see the real failure mode and fix it
in ONE pass instead of guessing repeatedly.
"""

import sys
sys.path.insert(0, '.')
import requests
import difflib
from collections import Counter
from db import execute_query

AUTH_KEY = 'keywWPoCKnwcJAA'
DB_ID    = '6a17ec96a95e7ac45342c0e4'
BASE     = f'https://table-api.viasocket.com/{DB_ID}'
HEADERS  = {'auth-key': AUTH_KEY}

def fetch_all(table_id):
    rows, offset, pages = [], None, 0
    while True:
        params = {'offset': offset} if offset else {}
        r = requests.get(f'{BASE}/{table_id}', headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        d = r.json().get('data', {})
        batch = d.get('rows', [])
        rows.extend(batch)
        pages += 1
        offset = d.get('offset')
        if not offset or not batch or pages > 50:
            break
    return rows

def normalize(name):
    if not name:
        return ''
    return ' '.join(name.strip().lower().split())

print('Fetching Keka employees (deduplicated by keka_id) and SQL Admin list...')
raw_emps = fetch_all('tblrj1w62')

# Dedup Keka employees by keka_id ('name' column), keep latest by createdat
latest_emp = {}
for e in raw_emps:
    kid = e.get('name')
    if not kid:
        continue
    ts = e.get('updatedat') or e.get('createdat') or ''
    if kid not in latest_emp or ts > latest_emp[kid].get('_ts', ''):
        latest_emp[kid] = {**e, '_ts': ts}

print(f'Raw employee rows: {len(raw_emps)}  ->  Deduplicated distinct people: {len(latest_emp)}')

admin_rows = execute_query("SELECT ID_Admin AS AdminID, Admin_FirstName + ' ' + Admin_LastName AS AdminName FROM Admin")
print(f'SQL Admin rows (no filter): {len(admin_rows)}')

# Build normalized admin lookup, tracking collisions
admin_by_norm = {}
admin_norm_collisions = Counter()
for a in admin_rows:
    norm = normalize(a['AdminName'])
    admin_norm_collisions[norm] += 1
    admin_by_norm.setdefault(norm, []).append(a)

print(f'Distinct normalized admin names: {len(admin_by_norm)}')
collided = {k: v for k, v in admin_norm_collisions.items() if v > 1}
print(f'Normalized names with >1 admin (duplicate real names in SQL): {len(collided)}')

# ── Full per-person verdict ────────────────────────────────────────────────────
results = []
for kid, emp in latest_emp.items():
    raw_name = emp.get('display_name', '')
    norm = normalize(raw_name)

    exact_candidates = admin_by_norm.get(norm, [])

    if len(exact_candidates) == 1:
        verdict = 'EXACT'
        match = exact_candidates[0]
    elif len(exact_candidates) > 1:
        verdict = 'AMBIGUOUS_EXACT'
        match = None
    else:
        # fuzzy
        scored = []
        for a in admin_rows:
            ratio = difflib.SequenceMatcher(None, norm, normalize(a['AdminName'])).ratio()
            if ratio >= 0.85:   # lowered bar for visibility — see what's close
                scored.append((ratio, a))
        scored.sort(key=lambda x: -x[0])
        if scored and scored[0][0] >= 0.92:
            if len(scored) > 1 and scored[0][0] - scored[1][0] < 0.05:
                verdict = 'AMBIGUOUS_FUZZY'
                match = None
            else:
                verdict = f'FUZZY_{scored[0][0]:.2f}'
                match = scored[0][1]
        elif scored:
            verdict = f'CLOSE_BUT_BELOW_THRESHOLD_{scored[0][0]:.2f}'
            match = None
        else:
            verdict = 'NO_CANDIDATE_AT_ALL'
            match = None

    results.append({
        'keka_id': kid, 'keka_name': raw_name, 'norm': norm,
        'verdict': verdict,
        'matched_admin_id': match['AdminID'] if match else None,
        'matched_admin_name': match['AdminName'] if match else None,
    })

# ── Summary ───────────────────────────────────────────────────────────────────
verdict_counts = Counter(r['verdict'].split('_')[0] if 'FUZZY' in r['verdict'] or 'CANDIDATE' in r['verdict']
                          else r['verdict'] for r in results)
print('\n' + '=' * 70)
print('VERDICT BREAKDOWN')
print('=' * 70)
for v, c in verdict_counts.most_common():
    print(f'  {v:<30} {c}')

resolved = sum(1 for r in results if r['matched_admin_id'] is not None)
print(f'\nTotal resolved: {resolved} / {len(results)}')

# ── Show every NO_CANDIDATE_AT_ALL case — these are the real mystery ───────────
no_candidate = [r for r in results if r['verdict'] == 'NO_CANDIDATE_AT_ALL']
print(f'\n{"=" * 70}')
print(f'NO_CANDIDATE_AT_ALL — {len(no_candidate)} people, full list')
print('=' * 70)
for r in no_candidate:
    print(f'  {r["keka_name"]:<30} norm={repr(r["norm"])}')

# ── Show CLOSE_BUT_BELOW_THRESHOLD cases — these might just need a lower bar ───
close = [r for r in results if 'CLOSE_BUT_BELOW' in r['verdict']]
print(f'\n{"=" * 70}')
print(f'CLOSE_BUT_BELOW_THRESHOLD — {len(close)} people (candidate exists but ratio < 0.92)')
print('=' * 70)
for r in close[:20]:
    print(f'  {r["keka_name"]:<30} {r["verdict"]}')

# ── Save full results for inspection ────────────────────────────────────────────
import json
with open('full_diagnostic_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nFull per-person results saved to full_diagnostic_results.json')

Fetching Keka employees (deduplicated by keka_id) and SQL Admin list...
Raw employee rows: 572  ->  Deduplicated distinct people: 286
SQL Admin rows (no filter): 934
Distinct normalized admin names: 812
Normalized names with >1 admin (duplicate real names in SQL): 41

VERDICT BREAKDOWN
  EXACT                          149
  NO                             103
  AMBIGUOUS_EXACT                23
  FUZZY                          6
  CLOSE_BUT_BELOW_THRESHOLD_0.87 1
  CLOSE_BUT_BELOW_THRESHOLD_0.88 1
  CLOSE_BUT_BELOW_THRESHOLD_0.90 1
  AMBIGUOUS                      1
  CLOSE_BUT_BELOW_THRESHOLD_0.86 1

Total resolved: 155 / 286

NO_CANDIDATE_AT_ALL — 103 people, full list
  Arjun Bhadoriya                norm='arjun bhadoriya'
  Rashmi Rajput                  norm='rashmi rajput'
  Tanushree Panwar               norm='tanushree panwar'
  Kalyani Kharate                norm='kalyani kharate'
  Vidhi Jain                     norm='vidhi jain'
  Nandan Nayak                   norm='nandan n

In [17]:
"""
verify_frontend_shape.py
===========================
Before touching any frontend file, this confirms exactly what shape of
data the new Keka-backed KPIs will hand to score_engine.py -> the API ->
the frontend, for a REAL person, side-by-side with what the OLD
ViaSocket-backed KPIs currently hand over for the same person.

This is the actual contract the frontend renders against. Run this and
paste the output before any EmployeeDetail.jsx change is made — that
way the frontend fix is based on the real before/after diff, not a
guess at what changed.
"""

import sys
sys.path.insert(0, '.')

TEST_ADMIN_ID = 346   # Ayushi Modi — confirmed real AdminID throughout this session
TEST_MONTH, TEST_YEAR = 5, 2026

print('=' * 78)
print('OLD (current live) KPI output shape — what the frontend renders TODAY')
print('=' * 78)
try:
    from kpis.leaves import LeavesKPI
    from kpis.late_comings import LateComingsKPI
    from kpis.early_leavings import EarlyLeavingsKPI

    for name, cls in [('leaves', LeavesKPI), ('late_comings', LateComingsKPI),
                       ('early_leavings', EarlyLeavingsKPI)]:
        inst = cls()
        rows = inst.fetch(TEST_MONTH, TEST_YEAR)
        emp_rows = [r for r in rows if r.get('AdminID') == TEST_ADMIN_ID]
        agg = inst.aggregate(emp_rows)
        print(f'\n--- OLD {name} ---')
        print(f'  numerator={agg["numerator"]}  denominator={agg["denominator"]}  '
              f'success_ratio={agg["success_ratio"]}')
        print(f'  orders[0] keys: {list(agg["orders"][0].keys()) if agg.get("orders") else "NO ORDERS"}')
        if agg.get('orders'):
            print(f'  orders[0] full: {agg["orders"][0]}')
except Exception as e:
    print(f'  ERROR loading old KPIs: {e}')

print('\n' + '=' * 78)
print('NEW (Keka-backed) KPI output shape — what frontend WOULD render if switched')
print('=' * 78)
try:
    from kpis.keka_attendance import KekaLeavesKPI, KekaLateComingsKPI, KekaEarlyLeavingsKPI

    for name, cls in [('keka_leaves', KekaLeavesKPI), ('keka_late_comings', KekaLateComingsKPI),
                       ('keka_early_leavings', KekaEarlyLeavingsKPI)]:
        inst = cls()
        rows = inst.fetch(TEST_MONTH, TEST_YEAR)
        emp_rows = [r for r in rows if r.get('AdminID') == TEST_ADMIN_ID]
        if not emp_rows:
            print(f'\n--- NEW {name} --- NO ROW for AdminID {TEST_ADMIN_ID} this month')
            continue
        agg = inst.aggregate(emp_rows)
        print(f'\n--- NEW {name} ---')
        print(f'  numerator={agg["numerator"]}  denominator={agg["denominator"]}  '
              f'success_ratio={agg["success_ratio"]}')
        print(f'  orders[0] keys: {list(agg["orders"][0].keys()) if agg.get("orders") else "NO ORDERS"}')
        if agg.get('orders'):
            print(f'  orders[0] full: {agg["orders"][0]}')
except Exception as e:
    print(f'  ERROR loading new KPIs: {e}')

print('\n' + '=' * 78)
print('FIELD-LEVEL DIFF — what keys exist in OLD orders[0] but NOT in NEW, and vice versa')
print('=' * 78)
print('(Paste full output above back — the actual diff will be read from it,')
print(' not computed here, since old/new field availability depends on live data)')

OLD (current live) KPI output shape — what the frontend renders TODAY
[ViaSocket] Leaves 5/2026 → 75 rows
[LeavesKPI] 5/2026 → 75 employees

--- OLD leaves ---
  numerator=25  denominator=26  success_ratio=96.15
  orders[0] keys: ['working_days', 'leave_days', 'present_days', 'leave_dates']
  orders[0] full: {'working_days': 26, 'leave_days': 1, 'present_days': 25, 'leave_dates': ['2026-05-15']}
[LateComings] 5/2026 → 33 employees

--- OLD late_comings ---
  numerator=0  denominator=0  success_ratio=None
  orders[0] keys: NO ORDERS
[EarlyLeavings] 5/2026 → 32 employees

--- OLD early_leavings ---
  numerator=0  denominator=0  success_ratio=None
  orders[0] keys: NO ORDERS

NEW (Keka-backed) KPI output shape — what frontend WOULD render if switched
[KekaAttendance] 5/2026 — fetching raw Keka tables...
[KekaAttendance] attendance=4138 status=1144 employees=572 groups=572
[KekaAdminMap] Refreshed: 0 resolved, 131 unresolved
[KekaAttendance] raw rows 2680 -> 1637 after dedup (1043 exact-du

In [18]:
"""
find_ekta_by_id.py
====================
We KNOW Ekta Hirke = AdminID 344 from earlier SQL Server work this session
(her file MMC26043034490400 was directly queried and confirmed multiple
times). She is real, active, and currently working. Yet the
Flag_Active=1 AND (Flag_Delete=0 OR Flag_Delete IS NULL) filter excluded
her entirely from the 547 rows returned.

This script looks her up directly by ID, with NO filter at all, to see
her actual flag values and prove exactly which condition is wrong.
"""

import sys
sys.path.insert(0, '.')
from db import execute_query

print('=' * 70)
print('Looking up AdminID 344 with NO filters at all')
print('=' * 70)
sql_no_filter = """
    SELECT ID_Admin, Admin_FirstName, Admin_LastName,
           Flag_Active, Flag_Delete
    FROM Admin
    WHERE ID_Admin = 344
"""
rows = execute_query(sql_no_filter)
if rows:
    for r in rows:
        print(f'  {r}')
else:
    print('  AdminID 344 not found AT ALL in the Admin table — different problem entirely.')

print('\n' + '=' * 70)
print('Checking the actual DISTINCT flag value combinations across the whole table')
print('=' * 70)
sql_distinct = """
    SELECT Flag_Active, Flag_Delete, COUNT(*) AS cnt
    FROM Admin
    GROUP BY Flag_Active, Flag_Delete
    ORDER BY cnt DESC
"""
combos = execute_query(sql_distinct)
for c in combos:
    print(f'  Flag_Active={c["Flag_Active"]}  Flag_Delete={c["Flag_Delete"]}  count={c["cnt"]}')

print('\n' + '=' * 70)
print('Total Admin rows with ZERO filters (true total)')
print('=' * 70)
total = execute_query("SELECT COUNT(*) AS cnt FROM Admin")
print(f'  Total rows in Admin table: {total[0]["cnt"]}')

Looking up AdminID 344 with NO filters at all
  {'ID_Admin': 344, 'Admin_FirstName': 'Ekta', 'Admin_LastName': 'Hirke', 'Flag_Active': False, 'Flag_Delete': False}

Checking the actual DISTINCT flag value combinations across the whole table
  Flag_Active=True  Flag_Delete=False  count=547
  Flag_Active=False  Flag_Delete=False  count=272
  Flag_Active=False  Flag_Delete=True  count=115

Total Admin rows with ZERO filters (true total)
  Total rows in Admin table: 934


In [20]:
from db import get_connection
import time
 
t0 = time.time()
conn = get_connection()
cur = conn.cursor()
cur.execute('SELECT 1')
cur.fetchone()
print(f'SQL Server responded in {time.time()-t0:.1f}s')
conn.close()

SQL Server responded in 2.8s


In [22]:
"""
debug_aggregate_query.py
============================
Runs the EXACT aggregate SQL from missing_status.py directly via pyodbc,
with full traceback visible, to see the real underlying SQL Server error
that execute_query() is currently masking.
"""
import sys
sys.path.insert(0, '.')
import traceback
from db import get_connection

month, year = 5, 2026

sql = f"""
DECLARE @Month INT={month}
DECLARE @Year  INT={year}
;WITH
LastAllocated AS (
    SELECT h.OrderID, h.AssignedTo_UserID,
        ROW_NUMBER() OVER (PARTITION BY h.OrderID ORDER BY h.Date_Ctreated DESC) AS rn
    FROM Table_OrderStatushistory h
    WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
      AND h.Date_Ctreated<=EOMONTH(DATEFROMPARTS(@Year,@Month,1))
),
LastAlloc AS (SELECT OrderID, AssignedTo_UserID FROM LastAllocated WHERE rn=1),
ResolvedAttribution AS (
    SELECT
        od.ID AS OrderID,
        COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID) AS AdminID
    FROM OrderDetails od
    LEFT JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
    LEFT JOIN LastAlloc la             ON la.OrderID = od.ID
    WHERE od.Flag_Deleted = 0
      AND MONTH(od.CompletionOn) = @Month
      AND YEAR(od.CompletionOn)  = @Year
),
WorkingDays AS (
    SELECT CAST(DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)) AS DATE) AS WorkDay
    FROM (SELECT TOP 31 ROW_NUMBER() OVER (ORDER BY (SELECT NULL))-1 AS n
          FROM master..spt_values) nums
    WHERE MONTH(DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)))=@Month
      AND DATEPART(WEEKDAY,DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)))<>1
),
TotalWD AS (SELECT COUNT(*) AS cnt FROM WorkingDays),
ActiveEmps AS (
    SELECT DISTINCT ra.AdminID
    FROM ResolvedAttribution ra
    INNER JOIN Admin af ON af.ID_Admin=ra.AdminID
                       AND af.Flag_Active=1 AND (af.Flag_Delete=0 OR af.Flag_Delete IS NULL)
    WHERE ra.AdminID IS NOT NULL
),
UpdateDays AS (
    SELECT ra.AdminID,
        COUNT(DISTINCT CAST(h.Date_Ctreated AS DATE)) AS DaysWithUpdate
    FROM Table_OrderStatushistory h
    INNER JOIN ResolvedAttribution ra ON ra.OrderID = h.OrderID
    WHERE h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
      AND MONTH(h.Date_Ctreated)=@Month AND YEAR(h.Date_Ctreated)=@Year
      AND ra.AdminID IS NOT NULL
    GROUP BY ra.AdminID
)
SELECT ae.AdminID, a.Admin_FirstName+' '+a.Admin_LastName AS EmployeeName,
    tw.cnt AS WorkingDaysInMonth,
    ISNULL(ud.DaysWithUpdate,0) AS DaysWithUpdate,
    tw.cnt - ISNULL(ud.DaysWithUpdate,0) AS MissedDays
FROM ActiveEmps ae
CROSS JOIN TotalWD tw
LEFT  JOIN UpdateDays ud ON ud.AdminID=ae.AdminID
INNER JOIN Admin a ON a.ID_Admin=ae.AdminID
    AND a.Flag_Active=1 AND (a.Flag_Delete=0 OR a.Flag_Delete IS NULL)
ORDER BY EmployeeName
"""

print("Connecting...")
conn = get_connection()
cur = conn.cursor()
print("Executing query directly (no wrapper)...")
try:
    cur.execute(sql)
    rows = cur.fetchall()
    print(f"SUCCESS: {len(rows)} rows")
    for r in rows[:5]:
        print(r)
except Exception as e:
    print("REAL ERROR:")
    print(type(e))
    print(str(e))
    traceback.print_exc()
finally:
    conn.close()

Connecting...
Executing query directly (no wrapper)...
REAL ERROR:
<class 'pyodbc.OperationalError'>
('08S01', '[08S01] [Microsoft][ODBC Driver 18 for SQL Server]TCP Provider: An established connection was aborted by the software in your host machine.\r\n (10053) (SQLExecDirectW); [08S01] [Microsoft][ODBC Driver 18 for SQL Server]Communication link failure (10053)')


Traceback (most recent call last):
  File "C:\Users\ACER\AppData\Local\Temp\ipykernel_56884\3579854798.py", line 80, in <module>
    cur.execute(sql)
pyodbc.OperationalError: ('08S01', '[08S01] [Microsoft][ODBC Driver 18 for SQL Server]TCP Provider: An established connection was aborted by the software in your host machine.\r\n (10053) (SQLExecDirectW); [08S01] [Microsoft][ODBC Driver 18 for SQL Server]Communication link failure (10053)')


In [23]:
"""
debug_encode_error.py
========================
Reproduces 'NoneType' object has no attribute 'encode' with a FULL
traceback, by calling exactly what score_engine.py calls: the
KPI's fetch() then aggregate(), same as inside the live app.
"""
import sys
sys.path.insert(0, '.')
import traceback

MONTH, YEAR = 6, 2026

from kpis.keka_attendance import KekaLeavesKPI, KekaLateComingsKPI, KekaEarlyLeavingsKPI

for name, cls in [('leaves', KekaLeavesKPI), ('late_comings', KekaLateComingsKPI),
                   ('early_leavings', KekaEarlyLeavingsKPI)]:
    print(f'\n{"="*70}')
    print(f'Testing {name}...')
    print('='*70)
    try:
        inst = cls()
        rows = inst.fetch(MONTH, YEAR)
        print(f'fetch() succeeded: {len(rows)} rows')
        if rows:
            agg = inst.aggregate([rows[0]])
            print(f'aggregate() succeeded: {agg["numerator"]}/{agg["denominator"]}')
    except Exception as e:
        print(f'FAILED: {type(e)} {e}')
        traceback.print_exc()


Testing leaves...
FAILED: <class 'AttributeError'> 'NoneType' object has no attribute 'encode'

Testing late_comings...
FAILED: <class 'AttributeError'> 'NoneType' object has no attribute 'encode'

Testing early_leavings...
FAILED: <class 'AttributeError'> 'NoneType' object has no attribute 'encode'


Traceback (most recent call last):
  File "C:\Users\ACER\AppData\Local\Temp\ipykernel_56884\1385725905.py", line 23, in <module>
    rows = inst.fetch(MONTH, YEAR)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Desktop\MMC_Scorecard_COMPLETE\backend\kpis\keka_attendance.py", line 512, in fetch
    # template `${e.expected_hours}h` renders literally as "h",
                                               ^^^^^^^^^^^^^^^^^
  File "c:\Desktop\MMC_Scorecard_COMPLETE\backend\kpis\keka_attendance.py", line 500, in _synthetic_admin_id
    continue
            ^
AttributeError: 'NoneType' object has no attribute 'encode'
Traceback (most recent call last):
  File "C:\Users\ACER\AppData\Local\Temp\ipykernel_56884\1385725905.py", line 23, in <module>
    rows = inst.fetch(MONTH, YEAR)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Desktop\MMC_Scorecard_COMPLETE\backend\kpis\keka_attendance.py", line 546, in fetch
    d = eff_start
                  
  File "c:\Desktop\MMC_Scorecard_COMPLETE\backend\

In [25]:
"""
find_none_keka_id.py
=======================
_synthetic_admin_id(None) is the crash. Find exactly which employee
record in keka_attendance.py's computed output has keka_id=None, so we
can fix the ROOT cause (why does this record exist with no keka_id at
all) rather than just paper over the crash site.
"""
import sys
sys.path.insert(0, '.')
from kpis.keka_attendance import get_month_data

MONTH, YEAR = 6, 2026

data = get_month_data(MONTH, YEAR, force_refresh=True)

print(f'Total employee records: {len(data["employees"])}')

none_kid = [e for e in data['employees'] if not e.get('keka_id')]
print(f'\nRecords with missing/None keka_id: {len(none_kid)}')
for e in none_kid:
    print(f'  {e}')

none_name = [e for e in data['employees'] if not e.get('name') or e.get('name') == '?']
print(f'\nRecords with missing/unknown name: {len(none_name)}')
for e in none_name[:10]:
    print(f'  keka_id={e.get("keka_id")}  name={e.get("name")}  admin_id={e.get("admin_id")}')

[KekaAttendance] 6/2026 — fetching raw Keka tables...
[KekaAttendance] attendance=6374 status=2005 employees=572 groups=572
[KekaAdminMap] Refreshed: 0 resolved, 131 unresolved
[KekaAttendance] raw rows 2513 -> 1227 after dedup (1286 exact-duplicate syncs removed)
[KekaAttendance] systemic days excluded: [datetime.date(2026, 6, 9), datetime.date(2026, 6, 10), datetime.date(2026, 6, 13), datetime.date(2026, 6, 15), datetime.date(2026, 6, 22), datetime.date(2026, 6, 23), datetime.date(2026, 6, 24)]
Total employee records: 81

Records with missing/None keka_id: 1
  {'keka_id': None, 'admin_id': None, 'name': '?', 'department': 'Unknown', 'modal_shift_start': '10:30', 'modal_shift_end': '19:00', 'joined_mid_month': False, 'joining_date': None, 'exited_mid_month': False, 'exit_date': None, 'expected_working_days': 19, 'present_days': 8, 'late_days': 1, 'early_exit_days': 13, 'unrecorded_absent_days': 11, 'incomplete_punch_days': 2, 'avg_hours_per_day': 8.14, 'late_dates': ['2026-06-11'], 'e

In [26]:
"""
find_raw_null_keka_id_rows.py
=================================
The computed employee record has keka_id=None, but it has REAL
attendance data (punch times, late/early entries). That means somewhere
in keka_attendance.py's raw attendance fetch, a row's 'name' column
(which holds the keka_id per this project's convention) is itself NULL
in the source data. This finds those exact raw rows.
"""
import sys
sys.path.insert(0, '.')
from kpis.keka_attendance import _fetch_all, TBL_ATTENDANCE

att = _fetch_all(TBL_ATTENDANCE)
print(f'Total raw attendance rows: {len(att)}')

null_name_rows = [r for r in att if not r.get('name')]
print(f'\nRows with null/missing name (keka_id) field: {len(null_name_rows)}')
for r in null_name_rows[:10]:
    print(f'  rowid={r.get("rowid")}  name={r.get("name")!r}  '
          f'employee_no={r.get("employee_no")}  date={r.get("date")}  '
          f'punch_in={r.get("punch_in")}  punch_out={r.get("punch_out")}')

# Also check: do any of these null-name rows still have an employee_no
# we could use to recover their real identity?
print('\nemployee_no values present on these null-name rows:')
empnos = set(r.get('employee_no') for r in null_name_rows)
print(f'  {empnos}')

Total raw attendance rows: 6374

Rows with null/missing name (keka_id) field: 119
  rowid=row9mh7qll8i  name=None  employee_no=None  date=None  punch_in=None  punch_out=None
  rowid=rowq3l6id5q6  name=None  employee_no=None  date=None  punch_in=None  punch_out=None
  rowid=rownfbel18ih  name=None  employee_no=08  date=Shivam lodhi  punch_in=2026-06-12 10:20:49  punch_out=2026-06-12 18:58:36
  rowid=rowr7vlseb8x  name=None  employee_no=08  date=Shivam lodhi  punch_in=2026-06-13 10:14:20  punch_out=null
  rowid=rowr3k4y3waj  name=None  employee_no=08  date=Shivam lodhi  punch_in=2026-06-15 10:31:29  punch_out=2026-06-15 18:59:31
  rowid=rowd1b9wvzyh  name=None  employee_no=08  date=Shivam lodhi  punch_in=2026-06-16 10:21:01  punch_out=2026-06-16 19:01:24
  rowid=rowwfocx3g4w  name=None  employee_no=08  date=Shivam lodhi  punch_in=2026-06-17 10:26:00  punch_out=2026-06-17 18:59:46
  rowid=row9iisp8xj3  name=None  employee_no=08  date=Shivam lodhi  punch_in=2026-06-18 10:19:10  punch_out=2

In [27]:
"""
check_recoverable_employees.py
==================================
For each employee_no found on null-name attendance rows, check whether
that person ALREADY has a proper keka_id elsewhere (meaning this null-
name data is redundant/safely ignorable), or whether it's their ONLY
June attendance data (meaning we're currently losing real data for them).
"""
import sys
sys.path.insert(0, '.')
from kpis.keka_attendance import _fetch_all, TBL_ATTENDANCE, TBL_EMPLOYEES

att = _fetch_all(TBL_ATTENDANCE)
emps = _fetch_all(TBL_EMPLOYEES)

# Map employee_no -> their REAL keka_id, from rows that DO have a name
empno_to_kid = {}
for r in att:
    if r.get('name') and r.get('employee_no'):
        empno_to_kid.setdefault(r['employee_no'], set()).add(r['name'])

# Map keka_id -> display_name for readability
name_by_kid = {e.get('name'): e.get('display_name', '?') for e in emps if e.get('name')}

null_name_rows = [r for r in att if not r.get('name') and r.get('employee_no')]
empnos = sorted(set(r['employee_no'] for r in null_name_rows))

print(f'Employee numbers appearing on null-name rows: {empnos}\n')

for empno in empnos:
    rows_for_this_empno_null = [r for r in null_name_rows if r['employee_no'] == empno]
    real_kids = empno_to_kid.get(empno, set())

    print(f'employee_no={empno}:')
    print(f'  Null-name rows for this employee_no: {len(rows_for_this_empno_null)}')
    print(f'  Real keka_id(s) found elsewhere for this employee_no: {real_kids}')
    for kid in real_kids:
        print(f'    -> {kid} = {name_by_kid.get(kid, "NOT IN keka_employees")}')

    if real_kids:
        # Check: does the person's REAL keka_id already have June rows
        # covering the SAME dates as the null-name rows? If yes, this is
        # likely a harmless duplicate. If no, we're losing real coverage.
        null_dates = set()
        for r in rows_for_this_empno_null:
            pin = r.get('punch_in')
            if pin:
                null_dates.add(str(pin)[:10])
        real_dates = set()
        for kid in real_kids:
            for r in att:
                if r.get('name') == kid and r.get('punch_in'):
                    real_dates.add(str(r['punch_in'])[:10])
        missing_coverage = null_dates - real_dates
        print(f'  Dates only present in NULL-NAME rows (not covered by real keka_id elsewhere): {sorted(missing_coverage)}')
    print()

Employee numbers appearing on null-name rows: ['08', '09', '106', '11', '114', '117', '120', '126', '130', '135']

employee_no=08:
  Null-name rows for this employee_no: 11
  Real keka_id(s) found elsewhere for this employee_no: {'06380524-742f-4169-b6ed-8ad5c87ed11b'}
    -> 06380524-742f-4169-b6ed-8ad5c87ed11b = Shivam lodhi
  Dates only present in NULL-NAME rows (not covered by real keka_id elsewhere): []

employee_no=09:
  Null-name rows for this employee_no: 11
  Real keka_id(s) found elsewhere for this employee_no: {'0e8fa4c1-d04d-4484-a0e3-5e9a9a384757'}
    -> 0e8fa4c1-d04d-4484-a0e3-5e9a9a384757 = Lakhan kori
  Dates only present in NULL-NAME rows (not covered by real keka_id elsewhere): []

employee_no=106:
  Null-name rows for this employee_no: 17
  Real keka_id(s) found elsewhere for this employee_no: {'3c1a94d7-21a7-47bf-b9ed-4251099667e0'}
    -> 3c1a94d7-21a7-47bf-b9ed-4251099667e0 = Ayushi Modi
  Dates only present in NULL-NAME rows (not covered by real keka_id elsewher

In [29]:
"""
verify_shift_and_grace.py
=============================
Confirms exactly what format modal_shift_start is stored in, for real
employees, and computes the correct shift_start + 15min grace label by
hand — so we know definitively what the UI SHOULD show before trusting
the JS computation added to Dashboard.jsx.
"""
import sys
sys.path.insert(0, '.')
from kpis.keka_attendance import get_month_data

MONTH, YEAR = 5, 2026

data = get_month_data(MONTH, YEAR)

print(f'Total employees: {len(data["employees"])}\n')
print('=' * 80)
print('Sample of modal_shift_start / modal_shift_end values (raw, as stored)')
print('=' * 80)

seen_formats = set()
for e in data['employees'][:20]:
    ss = e.get('modal_shift_start')
    se = e.get('modal_shift_end')
    seen_formats.add(type(ss).__name__)
    print(f'  {e["name"]:<25} modal_shift_start={ss!r:<10} modal_shift_end={se!r:<10}')

print(f'\nTypes seen for modal_shift_start: {seen_formats}')

print('\n' + '=' * 80)
print('Computing correct grace label by hand (shift_start + 15 min, 12hr display)')
print('=' * 80)

def compute_grace(shift_start_str):
    """Same logic the UI SHOULD apply."""
    if not shift_start_str or shift_start_str == '--':
        return '— (no shift data)'
    try:
        hours, mins = map(int, shift_start_str.split(':'))
    except Exception as ex:
        return f'PARSE ERROR: {ex}'
    total_mins = (hours * 60 + mins + 15) % (24 * 60)
    grace_h = total_mins // 60
    grace_m = total_mins % 60
    period = 'pm' if grace_h >= 12 else 'am'
    display_h = 12 if grace_h % 12 == 0 else grace_h % 12
    return f'{display_h:02d}:{grace_m:02d} {period}  (raw shift_start was {shift_start_str})'

for e in data['employees'][:20]:
    ss = e.get('modal_shift_start')
    grace = compute_grace(ss)
    print(f'  {e["name"]:<25} -> Grace: {grace}')

print('\n' + '=' * 80)
print('How many employees have NO late entries this month (the ones who')
print('previously showed BLANK grace under the old entries[0]-based logic)')
print('=' * 80)
no_late = [e for e in data['employees'] if e.get('late_days', 0) == 0]
print(f'Count: {len(no_late)} / {len(data["employees"])}')
print('Sample of THEIR modal_shift_start (this is what the FIXED version reads instead):')
for e in no_late[:10]:
    ss = e.get('modal_shift_start')
    grace = compute_grace(ss)
    print(f'  {e["name"]:<25} late_days={e.get("late_days")}  modal_shift_start={ss!r:<10} -> Grace: {grace}')

Total employees: 81

Sample of modal_shift_start / modal_shift_end values (raw, as stored)
  Rashmi Rajput             modal_shift_start='10:30'    modal_shift_end='19:00'   
  Ayushi Modi               modal_shift_start='10:30'    modal_shift_end='19:00'   
  Ekta Hirke                modal_shift_start='10:00'    modal_shift_end='18:30'   
  Rajat Mehra               modal_shift_start='09:30'    modal_shift_end='18:00'   
  Mahima Pardeshi           modal_shift_start='10:30'    modal_shift_end='19:00'   
  Roshani Panchore          modal_shift_start='10:30'    modal_shift_end='19:00'   
  Sakshi Jatav              modal_shift_start='10:30'    modal_shift_end='19:00'   
  Yash Purohit              modal_shift_start='10:30'    modal_shift_end='19:00'   
  Deepak Kumar Namdeo       modal_shift_start='10:30'    modal_shift_end='19:00'   
  Sakshi Mirg               modal_shift_start='10:30'    modal_shift_end='18:30'   
  Deepak Mishra             modal_shift_start='10:30'    modal_shift_

In [30]:
"""
check_specific_people_shift.py
==================================
Anchal Patidar and Ankit Mehta show blank Grace in the UI. Checking
whether their KekaLateComingsKPI output actually carries a usable
shift_start at the orders[0] level — the exact field the fixed
Dashboard.jsx code reads.
"""
import sys
sys.path.insert(0, '.')
from kpis.keka_attendance import KekaLateComingsKPI

MONTH, YEAR = 5, 2026

inst = KekaLateComingsKPI()
rows = inst.fetch(MONTH, YEAR)

targets = ['Anchal', 'Ankit Mehta']

for target in targets:
    matches = [r for r in rows if target.lower() in (r.get('EmployeeName') or '').lower()]
    print(f'\n--- Searching for "{target}" ---')
    if not matches:
        print('  NOT FOUND in KekaLateComingsKPI.fetch() output at all.')
        continue
    for r in matches:
        print(f'  EmployeeName: {r.get("EmployeeName")}')
        print(f'  AdminID: {r.get("AdminID")}  IsRealAdminID: {r.get("IsRealAdminID")}')
        print(f'  ShiftStart (raw field on the row): {r.get("ShiftStart")!r}')

        agg = inst.aggregate([r])
        o = agg['orders'][0] if agg.get('orders') else None
        print(f'  aggregate() orders[0]: {o}')
        if o:
            print(f'  orders[0].shift_start: {o.get("shift_start")!r}')


--- Searching for "Anchal" ---
  EmployeeName: Chanchal Keshariya
  AdminID: 871  IsRealAdminID: True
  ShiftStart (raw field on the row): '10:30'
  aggregate() orders[0]: {'working_days': 26, 'late_days': 8, 'on_time_days': 18, 'late_entries': ['2026-05-09', '2026-05-11', '2026-05-16', '2026-05-18', '2026-05-22', '2026-05-25', '2026-05-27', '2026-05-28'], 'shift_start': '10:30'}
  orders[0].shift_start: '10:30'
  EmployeeName: Anchal patidar
  AdminID: 907  IsRealAdminID: True
  ShiftStart (raw field on the row): '10:30'
  aggregate() orders[0]: {'working_days': 26, 'late_days': 0, 'on_time_days': 26, 'late_entries': [], 'shift_start': '10:30'}
  orders[0].shift_start: '10:30'

--- Searching for "Ankit Mehta" ---
  NOT FOUND in KekaLateComingsKPI.fetch() output at all.


In [31]:
"""
check_specific_people_shift.py
==================================
Anchal Patidar and Ankit Mehta show blank Grace in the UI. Checking
whether their KekaLateComingsKPI output actually carries a usable
shift_start at the orders[0] level — the exact field the fixed
Dashboard.jsx code reads.
"""
import sys
sys.path.insert(0, '.')
from kpis.keka_attendance import KekaLateComingsKPI

MONTH, YEAR = 5, 2026

inst = KekaLateComingsKPI()
rows = inst.fetch(MONTH, YEAR)

targets = ['Anchal', 'Ankit Mehta']

for target in targets:
    matches = [r for r in rows if target.lower() in (r.get('EmployeeName') or '').lower()]
    print(f'\n--- Searching for "{target}" ---')
    if not matches:
        print('  NOT FOUND in KekaLateComingsKPI.fetch() output at all.')
        continue
    for r in matches:
        print(f'  EmployeeName: {r.get("EmployeeName")}')
        print(f'  AdminID: {r.get("AdminID")}  IsRealAdminID: {r.get("IsRealAdminID")}')
        print(f'  ShiftStart (raw field on the row): {r.get("ShiftStart")!r}')

        agg = inst.aggregate([r])
        o = agg['orders'][0] if agg.get('orders') else None
        print(f'  aggregate() orders[0]: {o}')
        if o:
            print(f'  orders[0].shift_start: {o.get("shift_start")!r}')


--- Searching for "Anchal" ---
  EmployeeName: Chanchal Keshariya
  AdminID: 871  IsRealAdminID: True
  ShiftStart (raw field on the row): '10:30'
  aggregate() orders[0]: {'working_days': 26, 'late_days': 8, 'on_time_days': 18, 'late_entries': ['2026-05-09', '2026-05-11', '2026-05-16', '2026-05-18', '2026-05-22', '2026-05-25', '2026-05-27', '2026-05-28'], 'shift_start': '10:30'}
  orders[0].shift_start: '10:30'
  EmployeeName: Anchal patidar
  AdminID: 907  IsRealAdminID: True
  ShiftStart (raw field on the row): '10:30'
  aggregate() orders[0]: {'working_days': 26, 'late_days': 0, 'on_time_days': 26, 'late_entries': [], 'shift_start': '10:30'}
  orders[0].shift_start: '10:30'

--- Searching for "Ankit Mehta" ---
  NOT FOUND in KekaLateComingsKPI.fetch() output at all.


In [33]:
"""
check_shift_timezone_bug.py
==============================
The UI shows Grace=10:45am but the per-entry Shift Start column shows
04:00pm for the SAME person's SAME underlying shift. Tracing the real
strings being produced, for the EXACT person/date in the screenshot
(June 19, late arrival, punch_in ~08:17pm, shift presumably 4:00pm),
to settle definitively whether toIST() is wrongly adding a timezone
shift on top of an already-correct IST value.
"""
import sys
sys.path.insert(0, '.')
from kpis.keka_attendance import KekaLateComingsKPI

MONTH, YEAR = 6, 2026

inst = KekaLateComingsKPI()
rows = inst.fetch(MONTH, YEAR)

# Find whoever has a late entry on June 19 with punch_in around 8:17pm
target_rows = [r for r in rows if any(
    e.get('date') == '2026-06-19' for e in (r.get('LateEntries') or [])
)]

print(f'Rows with a June 19 late entry: {len(target_rows)}')
for r in target_rows:
    print(f'\nEmployeeName: {r.get("EmployeeName")}')
    print(f'ShiftStart (summary, o.shift_start equivalent): {r.get("ShiftStart")!r}')
    for e in r.get('LateEntries', []):
        if e.get('date') == '2026-06-19':
            print(f'  Per-entry shift_start (raw string): {e.get("shift_start")!r}')
            print(f'  Per-entry punch_in    (raw string): {e.get("punch_in")!r}')

print('\n' + '=' * 70)
print('Manually reproducing what toIST() does in JS, for the per-entry value')
print('=' * 70)
from datetime import datetime, timezone, timedelta

for r in target_rows:
    for e in r.get('LateEntries', []):
        if e.get('date') != '2026-06-19':
            continue
        raw = e.get('shift_start')
        if not raw:
            continue
        # Simulate: raw.replace(' UTC','Z') then JS new Date(...) parses
        # as UTC, then toLocaleTimeString with timeZone Asia/Kolkata
        # converts FROM that UTC instant TO IST (UTC+5:30).
        cleaned = raw.replace(' UTC', '')
        naive = datetime.strptime(cleaned, '%Y-%m-%d %H:%M:%S')
        as_utc = naive.replace(tzinfo=timezone.utc)
        as_ist = as_utc.astimezone(timezone(timedelta(hours=5, minutes=30)))
        print(f'  Raw backend string:        {raw}')
        print(f'  Naive clock value:         {naive.strftime("%H:%M")} (no timezone info)')
        print(f'  If treated as UTC and converted to IST by toIST(): {as_ist.strftime("%I:%M %p")}')
        print(f'  (i.e. naive value + 5:30)')

Rows with a June 19 late entry: 0

Manually reproducing what toIST() does in JS, for the per-entry value


In [34]:
"""
find_by_punch_time.py
========================
Previous search for June 19 found nothing -- search assumption was
wrong somewhere. This searches BOTH May and June, across ALL late
entries, for any punch_in around 20:17 (8:17 PM) regardless of date,
to find the actual person/row from the screenshot without guessing.
"""
import sys
sys.path.insert(0, '.')
from kpis.keka_attendance import KekaLateComingsKPI

for month, year in [(5, 2026), (6, 2026)]:
    print(f'\n{"="*70}')
    print(f'Searching {month}/{year}')
    print('='*70)
    inst = KekaLateComingsKPI()
    rows = inst.fetch(month, year)
    print(f'Total rows: {len(rows)}')

    for r in rows:
        for e in (r.get('LateEntries') or []):
            punch_in = e.get('punch_in') or ''
            # Look for hour 20 (8 PM) and minute close to 17
            if ' 20:1' in punch_in or ' 20:0' in punch_in:
                print(f'\n  MATCH: {r.get("EmployeeName")}')
                print(f'    date: {e.get("date")}')
                print(f'    shift_start (per-entry): {e.get("shift_start")!r}')
                print(f'    punch_in (per-entry):    {e.get("punch_in")!r}')
                print(f'    ShiftStart (summary):    {r.get("ShiftStart")!r}')


Searching 5/2026
Total rows: 81

Searching 6/2026
Total rows: 80


In [35]:
"""
check_cache_vs_fresh.py
==========================
The screenshot's data (shift 4pm, punch_in 8:17pm, June 19) doesn't
exist in a FRESH fetch() for either month. This checks what's actually
sitting in keka_attendance_cache and score_cache right now -- if the
browser is showing stale cached data from before recent fixes, this
will reveal it directly.
"""
import sqlite3
import json

conn = sqlite3.connect('scorecard.db')

print('=' * 70)
print('keka_attendance_cache contents')
print('=' * 70)
try:
    rows = conn.execute(
        "SELECT month, year, computed_at, length(data_json) FROM keka_attendance_cache"
    ).fetchall()
    for r in rows:
        print(f'  month={r[0]} year={r[1]} computed_at={r[2]} size={r[3]} bytes')
except sqlite3.OperationalError as e:
    print(f'  ERROR: {e}')

print('\n' + '=' * 70)
print('score_cache contents')
print('=' * 70)
try:
    rows = conn.execute(
        "SELECT month, year, created_at, length(data_json) FROM score_cache"
    ).fetchall()
    for r in rows:
        print(f'  month={r[0]} year={r[1]} created_at={r[2]} size={r[3]} bytes')
except sqlite3.OperationalError as e:
    print(f'  ERROR: {e}')
    # try without created_at in case column name differs
    try:
        cols = conn.execute("PRAGMA table_info(score_cache)").fetchall()
        print(f'  Actual score_cache columns: {[c[1] for c in cols]}')
    except Exception as e2:
        print(f'  Could not inspect columns either: {e2}')

print('\n' + '=' * 70)
print('Searching cached keka_attendance_cache data_json for the screenshot values')
print('(punch_in containing 20:1, shift_start containing 16:00)')
print('=' * 70)
try:
    rows = conn.execute("SELECT month, year, data_json FROM keka_attendance_cache").fetchall()
    for month, year, data_json in rows:
        data = json.loads(data_json)
        for emp in data.get('employees', []):
            for entry in emp.get('late_entries', []):
                pin = entry.get('punch_in', '')
                if '20:1' in pin or '20:0' in pin:
                    print(f'  FOUND in {month}/{year} cache: {emp["name"]}')
                    print(f'    date={entry.get("date")} shift_start={entry.get("shift_start")} punch_in={pin}')
                    print(f'    modal_shift_start (person-level)={emp.get("modal_shift_start")}')
except Exception as e:
    print(f'  ERROR searching: {e}')

conn.close()

keka_attendance_cache contents
  month=6 year=2026 computed_at=2026-06-25T07:06:40.011992 size=79469 bytes
  month=5 year=2026 computed_at=2026-06-25T07:12:01.745376 size=87993 bytes
  month=1 year=2026 computed_at=2026-06-25T07:17:25.929854 size=160 bytes
  month=2 year=2026 computed_at=2026-06-25T07:20:24.971994 size=160 bytes
  month=3 year=2026 computed_at=2026-06-25T07:23:45.815179 size=160 bytes
  month=4 year=2026 computed_at=2026-06-25T07:26:31.507192 size=34197 bytes

score_cache contents
  month=6 year=2026 created_at=2026-06-25 07:06:40 size=478166 bytes
  month=5 year=2026 created_at=2026-06-25 07:12:01 size=555924 bytes
  month=1 year=2026 created_at=2026-06-25 07:17:26 size=222453 bytes
  month=2 year=2026 created_at=2026-06-25 07:20:25 size=302774 bytes
  month=3 year=2026 created_at=2026-06-25 07:23:45 size=359619 bytes
  month=4 year=2026 created_at=2026-06-25 07:26:31 size=445366 bytes

Searching cached keka_attendance_cache data_json for the screenshot values
(punch_

In [36]:
"""
broad_search_june19.py
=========================
Cast a wider net: dump EVERY late entry for June 19 specifically (no
time-pattern filtering at all), across the cached data directly, so we
see exactly what exists for that date regardless of format assumptions.
"""
import sqlite3
import json

conn = sqlite3.connect('scorecard.db')

rows = conn.execute(
    "SELECT month, year, data_json FROM keka_attendance_cache WHERE month=6 AND year=2026"
).fetchall()

for month, year, data_json in rows:
    data = json.loads(data_json)
    print(f'Checking cached {month}/{year}, {len(data.get("employees", []))} employees\n')

    found_any = False
    for emp in data.get('employees', []):
        for entry in emp.get('late_entries', []):
            if entry.get('date') == '2026-06-19':
                found_any = True
                print(f'  {emp["name"]:<25} shift_start={entry.get("shift_start")!r}  '
                      f'punch_in={entry.get("punch_in")!r}')

    if not found_any:
        print('  No late_entries at all dated 2026-06-19 in this cached data.')

    # Also check early_entries in case the screenshot is actually from
    # the early_leavings card, not late_comings (a "+258 min" delay is
    # unusually large for lateness but could be mislabeled)
    print('\n  Checking early_entries for 2026-06-19 too:')
    found_early = False
    for emp in data.get('employees', []):
        for entry in emp.get('early_entries', []):
            if entry.get('date') == '2026-06-19':
                found_early = True
                print(f'  {emp["name"]:<25} shift_end={entry.get("shift_end")!r}  '
                      f'punch_out={entry.get("punch_out")!r}')
    if not found_early:
        print('  No early_entries dated 2026-06-19 either.')

Checking cached 6/2026, 80 employees

  Mohammad Altaf            shift_start='2026-06-19 10:00:00 UTC'  punch_in='2026-06-19 11:03:02 UTC'
  Shivani Tiwari            shift_start='2026-06-19 10:30:00 UTC'  punch_in='2026-06-19 14:47:40 UTC'
  Sakina Ratlamwala         shift_start='2026-06-19 10:30:00 UTC'  punch_in='2026-06-19 14:33:28 UTC'
  Ajay Malviya              shift_start='2026-06-19 09:00:00 UTC'  punch_in='2026-06-19 10:32:54 UTC'
  Aman Rane                 shift_start='2026-06-19 11:00:00 UTC'  punch_in='2026-06-19 11:33:30 UTC'

  Checking early_entries for 2026-06-19 too:
  Shivam Nirale             shift_end='2026-06-19 19:00:00 UTC'  punch_out='2026-06-19 15:25:57 UTC'
  Archita Neema             shift_end='2026-06-19 19:00:00 UTC'  punch_out='2026-06-19 18:01:58 UTC'
  Arjun Bhadoriya           shift_end='2026-06-19 19:00:00 UTC'  punch_out='2026-06-19 18:14:33 UTC'
  Sakshi Jatav              shift_end='2026-06-19 19:00:00 UTC'  punch_out='2026-06-19 17:37:56 UTC'
  

In [37]:
"""
verify_reckon_platform_field.py
===================================
Finds the actual field that identifies a file's DESTINATION software
platform (e.g. Reckon, Xero, QBO), so we can correctly target Reckon
files for the completion-date fix, instead of guessing a field/table
name.

Strategy:
1. List every column on OrderDetails and Table_OrderDetailsMisc —
   scan for anything platform/destination/software-sounding.
2. For each candidate column, show its DISTINCT values — if "Reckon"
   (or similar) appears as a real value, that's our field.
3. Cross-check against a few Admin names we already know contain
   "Reckon" in the name itself (e.g. "Abhishek Gour Reckon") to see if
   files attributed to them correlate with whatever field we find.
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

print('=' * 78)
print('OrderDetails columns')
print('=' * 78)
cols = execute_query("SELECT TOP 1 * FROM OrderDetails")
if cols:
    all_cols = list(cols[0].keys())
    print(all_cols)
    candidates = [c for c in all_cols if any(
        kw in c.lower() for kw in ['platform', 'destination', 'software', 'source', 'app', 'system', 'type']
    )]
    print(f'\nLikely candidate columns: {candidates}')

print('\n' + '=' * 78)
print('Table_OrderDetailsMisc columns')
print('=' * 78)
cols2 = execute_query("SELECT TOP 1 * FROM Table_OrderDetailsMisc")
if cols2:
    all_cols2 = list(cols2[0].keys())
    print(all_cols2)
    candidates2 = [c for c in all_cols2 if any(
        kw in c.lower() for kw in ['platform', 'destination', 'software', 'source', 'app', 'system', 'type']
    )]
    print(f'\nLikely candidate columns: {candidates2}')

print('\n' + '=' * 78)
print('Checking DISTINCT values of each candidate column for "Reckon"')
print('=' * 78)
for table, cand_list in [('OrderDetails', candidates if cols else []),
                          ('Table_OrderDetailsMisc', candidates2 if cols2 else [])]:
    for col in cand_list:
        try:
            sql = f"SELECT DISTINCT {col} FROM {table} WHERE {col} IS NOT NULL"
            vals = execute_query(sql)
            val_list = [v[col] for v in vals][:30]
            has_reckon = any('reckon' in str(v).lower() for v in val_list if v)
            marker = '  <-- CONTAINS RECKON' if has_reckon else ''
            print(f'\n{table}.{col}: {val_list}{marker}')
        except Exception as e:
            print(f'\n{table}.{col}: ERROR — {e}')

print('\n' + '=' * 78)
print('Cross-check: orders attributed to AdminID known to have "Reckon" in name')
print('=' * 78)
sql = """
    SELECT ID_Admin, Admin_FirstName, Admin_LastName
    FROM Admin
    WHERE Admin_FirstName LIKE '%Reckon%' OR Admin_LastName LIKE '%Reckon%'
"""
reckon_admins = execute_query(sql)
print(f'Admin rows with "Reckon" in their name: {len(reckon_admins)}')
for a in reckon_admins[:10]:
    print(f'  ID_Admin={a["ID_Admin"]}  {a["Admin_FirstName"]} {a["Admin_LastName"]}')

OrderDetails columns
['ID', 'FName', 'LName', 'Email', 'PhoneNumber', 'PracticeName', 'CName', 'YearEndMonth', 'CustomisedConversions', 'ConversionType', 'AdditionalYear', 'DestinationAccount', 'FileType', 'UserName', 'Password', 'SpecialNote', 'Cost', 'UploadFile', 'StartDate', 'EndDate', 'isNew', 'Country', 'Currency', 'Altfuppath', 'FinalCost', 'PromoCode', 'PromoCodeAmount', 'CreatedOn', 'UserId', 'PaymentStatus', 'PaymentReceived', 'UserCountry', 'State', 'city', 'pincode', 'address', 'isMultiCurrency', 'IsPayroll', 'IsCOA', 'IsOtherCustomization', 'OtherCustomizationValue', 'IsCloud', 'IsOtherSW', 'txtXeroPromoCode', 'Order_Number', 'SoftwareName', 'XeroSubscriptionEmail', 'CompanyName', 'OrderDueDate', 'OrderStatus', 'AssignUserId', 'AssignDate', 'CompletionOn', 'ID_Software', 'ID_PaymentStatus', 'BillyLoginID', 'BillyPassword', 'AdminID', 'DestinationSW', 'Temp_Id', 'ReferralId', 'XeroSubscriptionPassword', 'PaymentGateway', 'SWVersion', 'MemorableWords', 'IsXeroProject', 'IsAt

In [38]:
"""
verify_reckon_review_date.py
================================
Confirmed: OrderDetails.DestinationSW = 'Reckon' (or
DestinationSW_SubCategory = 'RECKON Retail') identifies Reckon files.

Now checking whether Table_OrderDetailsMisc.Date_OrderReviewStatus
and/or Reckon_BookStatus are the real "order review date" / auto-
completion-after-7-days fields described, by:

1. Showing distinct values of Reckon_BookStatus (a status field,
   probably has values like 'Reviewed', 'Completed', etc.)
2. For actual Reckon files, comparing Date_OrderReviewStatus against
   CompletionOn and computing the gap in days -- if many Reckon files
   show exactly ~7 days between Date_OrderReviewStatus and
   CompletionOn, that strongly confirms this IS the auto-completion
   mechanism described.
3. Showing a handful of real example rows so the actual relationship
   between these dates is visible directly, not inferred.
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

print('=' * 78)
print('Distinct values of Reckon_BookStatus')
print('=' * 78)
vals = execute_query("SELECT DISTINCT Reckon_BookStatus FROM Table_OrderDetailsMisc WHERE Reckon_BookStatus IS NOT NULL")
for v in vals:
    print(f'  {v["Reckon_BookStatus"]!r}')

print('\n' + '=' * 78)
print('Real Reckon files: CompletionOn vs Date_OrderReviewStatus, gap in days')
print('=' * 78)
sql = """
    SELECT TOP 30
        od.Order_Number,
        od.CompanyName,
        od.DestinationSW,
        od.CompletionOn,
        m.Date_OrderReviewStatus,
        m.Reckon_BookStatus,
        m.Flag_OrderCompleted,
        DATEDIFF(DAY, m.Date_OrderReviewStatus, od.CompletionOn) AS GapDays
    FROM OrderDetails od
    INNER JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
    WHERE od.DestinationSW = 'Reckon'
      AND od.Flag_Deleted = 0
      AND m.Date_OrderReviewStatus IS NOT NULL
      AND od.CompletionOn IS NOT NULL
    ORDER BY od.CompletionOn DESC
"""
rows = execute_query(sql)
print(f'Found {len(rows)} Reckon files with both dates populated\n')
for r in rows:
    print(f'  {r["Order_Number"]:<20} {r["CompanyName"]:<25} '
          f'ReviewDate={r["Date_OrderReviewStatus"]}  CompletionOn={r["CompletionOn"]}  '
          f'Gap={r["GapDays"]}d  BookStatus={r["Reckon_BookStatus"]!r}  '
          f'OrderCompleted={r["Flag_OrderCompleted"]}')

print('\n' + '=' * 78)
print('Distribution of GapDays across ALL Reckon files (not just top 30)')
print('=' * 78)
sql2 = """
    SELECT DATEDIFF(DAY, m.Date_OrderReviewStatus, od.CompletionOn) AS GapDays, COUNT(*) AS cnt
    FROM OrderDetails od
    INNER JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
    WHERE od.DestinationSW = 'Reckon'
      AND od.Flag_Deleted = 0
      AND m.Date_OrderReviewStatus IS NOT NULL
      AND od.CompletionOn IS NOT NULL
    GROUP BY DATEDIFF(DAY, m.Date_OrderReviewStatus, od.CompletionOn)
    ORDER BY cnt DESC
"""
dist = execute_query(sql2)
for d in dist[:20]:
    print(f'  GapDays={d["GapDays"]:<5} count={d["cnt"]}')

Distinct values of Reckon_BookStatus
  'InProgress'
  'CustomerReview'
  'Completed'

Real Reckon files: CompletionOn vs Date_OrderReviewStatus, gap in days
Found 30 Reckon files with both dates populated

  MMC26062045651009    Schurmed Pty Ltd          ReviewDate=2026-06-21 10:42:21.443000  CompletionOn=2026-06-28 15:30:05.400000  Gap=7d  BookStatus='CustomerReview'  OrderCompleted=True
  MMC26061945460628    Green Products            ReviewDate=2026-06-21 10:38:32.180000  CompletionOn=2026-06-28 15:30:04.363000  Gap=7d  BookStatus='Completed'  OrderCompleted=True
  MMC26061945400037    Shailer Plantations       ReviewDate=2026-06-21 12:51:03.593000  CompletionOn=2026-06-28 15:30:03.330000  Gap=7d  BookStatus='Completed'  OrderCompleted=True
  MMC26061745020555    P&K McGrath               ReviewDate=2026-06-21 10:48:25.050000  CompletionOn=2026-06-28 15:30:02.297000  Gap=7d  BookStatus='Completed'  OrderCompleted=True
  MMC26061143900929    Farcru PL ATF Karratha Unit Trust ReviewDa

In [39]:
"""
JUPYTER CELL — paste this whole thing into one cell and run.
================================================================
Compares the OLD logic (plain od.CompletionOn) against the NEW logic
(Reckon override using Date_OrderReviewStatus) for real May/June 2026
data, so we can see the actual before/after difference before touching
any production files.
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

MONTH, YEAR = 5, 2026   # change this to test other months

print(f'Testing month={MONTH} year={YEAR}\n')

# ── OLD logic: plain CompletionOn ──────────────────────────────────────────
old_sql = f"""
DECLARE @Month INT={MONTH}
DECLARE @Year  INT={YEAR}
;WITH
LastAllocated AS (
    SELECT OrderID, AssignedTo_UserID,
        ROW_NUMBER() OVER (PARTITION BY OrderID ORDER BY Date_Ctreated DESC) AS rn
    FROM Table_OrderStatushistory
    WHERE StatusID=11 AND AssignedTo_UserID IS NOT NULL AND AssignedTo_UserID>0
      AND Date_Ctreated<=EOMONTH(DATEFROMPARTS(@Year,@Month,1))
),
BaseData AS (
    SELECT m.ID_OrderDetailsMisc, m.OrderID, m.PickedUpBy, m.Summary_AssignUserID,
        CAST(od.CompletionOn AS DATE) AS FinalCompletionDate,
        od.DestinationSW
    FROM Table_OrderDetailsMisc m
    INNER JOIN OrderDetails od ON od.ID=m.OrderID
    WHERE m.Flag_OrderCompleted=1
)
SELECT
    COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID) AS AdminID,
    b.OrderID, od.Order_Number AS OrderNumber, od.CompanyName,
    b.FinalCompletionDate, b.DestinationSW
FROM BaseData b
INNER JOIN Table_OrderDetailsMisc m ON m.ID_OrderDetailsMisc=b.ID_OrderDetailsMisc
INNER JOIN OrderDetails od           ON od.ID=b.OrderID
LEFT  JOIN LastAllocated la          ON la.OrderID=b.OrderID AND la.rn=1
WHERE MONTH(b.FinalCompletionDate)=@Month AND YEAR(b.FinalCompletionDate)=@Year
  AND COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID) IS NOT NULL
  AND b.DestinationSW = 'Reckon'
ORDER BY b.FinalCompletionDate
"""

# ── NEW logic: Reckon uses Date_OrderReviewStatus ──────────────────────────
new_sql = f"""
DECLARE @Month INT={MONTH}
DECLARE @Year  INT={YEAR}
;WITH
LastAllocated AS (
    SELECT OrderID, AssignedTo_UserID,
        ROW_NUMBER() OVER (PARTITION BY OrderID ORDER BY Date_Ctreated DESC) AS rn
    FROM Table_OrderStatushistory
    WHERE StatusID=11 AND AssignedTo_UserID IS NOT NULL AND AssignedTo_UserID>0
      AND Date_Ctreated<=EOMONTH(DATEFROMPARTS(@Year,@Month,1))
),
BaseData AS (
    SELECT m.ID_OrderDetailsMisc, m.OrderID, m.PickedUpBy, m.Summary_AssignUserID,
        CAST(
            CASE
                WHEN od.DestinationSW = 'Reckon' AND m.Date_OrderReviewStatus IS NOT NULL
                THEN m.Date_OrderReviewStatus
                ELSE od.CompletionOn
            END
        AS DATE) AS FinalCompletionDate,
        od.DestinationSW,
        CAST(od.CompletionOn AS DATE) AS OldCompletionDate
    FROM Table_OrderDetailsMisc m
    INNER JOIN OrderDetails od ON od.ID=m.OrderID
    WHERE m.Flag_OrderCompleted=1
)
SELECT
    COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID) AS AdminID,
    b.OrderID, od.Order_Number AS OrderNumber, od.CompanyName,
    b.FinalCompletionDate, b.OldCompletionDate, b.DestinationSW
FROM BaseData b
INNER JOIN Table_OrderDetailsMisc m ON m.ID_OrderDetailsMisc=b.ID_OrderDetailsMisc
INNER JOIN OrderDetails od           ON od.ID=b.OrderID
LEFT  JOIN LastAllocated la          ON la.OrderID=b.OrderID AND la.rn=1
WHERE MONTH(b.FinalCompletionDate)=@Month AND YEAR(b.FinalCompletionDate)=@Year
  AND COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID) IS NOT NULL
  AND b.DestinationSW = 'Reckon'
ORDER BY b.FinalCompletionDate
"""

print('=' * 78)
print(f'OLD LOGIC — Reckon files landing in {MONTH}/{YEAR} (by plain CompletionOn)')
print('=' * 78)
old_rows = execute_query(old_sql)
print(f'Count: {len(old_rows)}')
for r in old_rows[:15]:
    print(f"  {r['OrderNumber']:<20} {r['CompanyName']:<25} CompletionOn={r['FinalCompletionDate']}")

print('\n' + '=' * 78)
print(f'NEW LOGIC — Reckon files landing in {MONTH}/{YEAR} (by Date_OrderReviewStatus)')
print('=' * 78)
new_rows = execute_query(new_sql)
print(f'Count: {len(new_rows)}')
for r in new_rows[:15]:
    print(f"  {r['OrderNumber']:<20} {r['CompanyName']:<25} "
          f"NewDate={r['FinalCompletionDate']}  OldCompletionOn={r['OldCompletionDate']}")

print('\n' + '=' * 78)
print('DIFF — orders that changed which MONTH they land in')
print('=' * 78)
old_ids = {r['OrderID'] for r in old_rows}
new_ids = {r['OrderID'] for r in new_rows}

only_in_old = old_ids - new_ids
only_in_new = new_ids - old_ids

print(f'In OLD month but NOT in NEW month (moved to a different month): {len(only_in_old)}')
for r in old_rows:
    if r['OrderID'] in only_in_old:
        print(f"  {r['OrderNumber']:<20} {r['CompanyName']:<25} was CompletionOn={r['FinalCompletionDate']}")

print(f'\nIn NEW month but NOT in OLD month (moved IN from a different month): {len(only_in_new)}')
for r in new_rows:
    if r['OrderID'] in only_in_new:
        print(f"  {r['OrderNumber']:<20} {r['CompanyName']:<25} now ReviewDate={r['FinalCompletionDate']} (was CompletionOn={r['OldCompletionDate']})")

print(f'\nTotal Reckon files unaffected (same month either way): {len(old_ids & new_ids)}')

Testing month=5 year=2026

OLD LOGIC — Reckon files landing in 5/2026 (by plain CompletionOn)
Count: 257
  MMC26042031750116    Bingo Sales Staff Supperannuation Fund CompletionOn=2026-05-01
  MMC26042031760117    The Lever - Howell Supperannuation Fund CompletionOn=2026-05-01
  MMC26042032152333    TAPE                      CompletionOn=2026-05-01
  MMC26042132270550    NJ Barton                 CompletionOn=2026-05-01
  MMC26042132280551    Barton Superfund          CompletionOn=2026-05-01
  MMC26042132290615    Tam City                  CompletionOn=2026-05-01
  MMC26042132310627    Lestaway review           CompletionOn=2026-05-01
  MMC26042132320629    Lifestyle Design Homes Pty Ltd CompletionOn=2026-05-01
  MMC26042132330633    Matthew Ramm              CompletionOn=2026-05-01
  MMC26042132350636    David & Leah Jericho      CompletionOn=2026-05-01
  MMC26042232510359    Omar Quiroga              CompletionOn=2026-05-01
  MMC26042332911008    Coronada Development Services Pty Ltd

In [40]:
"""
JUPYTER CELL — verify_missing_status_303.py
================================================
Two things:
1. Run the CURRENT (pre-Reckon-fix) missing_status.py logic for
   AdminID 303, May 2026 — establish the baseline.
2. Check which of those files are actually on the Reckon platform —
   if none are, this employee won't be affected by the fix at all,
   which tells us whether 303 is even a useful test case for THIS
   specific fix, or whether we need a different employee who
   actually handles Reckon files.
"""
import sys
sys.path.insert(0, '.')
from kpis.missing_status import MissingStatusKPI
from db import execute_query

MONTH, YEAR = 5, 2026
TEST_ADMIN_ID = 303

inst = MissingStatusKPI()
rows = inst.fetch(MONTH, YEAR)

emp_rows = [r for r in rows if r.get('AdminID') == TEST_ADMIN_ID]
print(f'Rows for AdminID={TEST_ADMIN_ID}: {len(emp_rows)}')

if emp_rows:
    r = emp_rows[0]
    print(f'EmployeeName: {r.get("EmployeeName")}')
    print(f'WorkingDaysInMonth: {r.get("WorkingDaysInMonth")}')
    print(f'DaysWithUpdate: {r.get("DaysWithUpdate")}')
    print(f'MissedDays: {r.get("MissedDays")}')

    agg = inst.aggregate(emp_rows)
    print(f'\nAggregate: {agg["numerator"]}/{agg["denominator"]} = {agg["success_ratio"]}%')

    file_breakdown = agg['orders'][0]['file_breakdown']
    print(f'\nPer-file breakdown ({len(file_breakdown)} files):')
    order_ids = []
    for fb in file_breakdown:
        print(f"  {fb['order_number']:<20} {fb['company']:<30} "
              f"completed={fb['completed']} active_from={fb['active_from']} "
              f"active_to={fb.get('active_to','—')} active={fb['active_days']} "
              f"updated={fb['days_updated']} missed={fb['days_missed']}")
        order_ids.append(fb['order_id'])

    print('\n' + '=' * 78)
    print(f'Checking which of these {len(order_ids)} files are on Reckon platform')
    print('=' * 78)
    if order_ids:
        ids_str = ','.join(str(i) for i in order_ids if i)
        sql = f"""
            SELECT od.ID, od.Order_Number, od.DestinationSW,
                   m.Date_OrderReviewStatus, od.CompletionOn,
                   DATEDIFF(DAY, m.Date_OrderReviewStatus, od.CompletionOn) AS GapDays
            FROM OrderDetails od
            LEFT JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
            WHERE od.ID IN ({ids_str})
        """
        platform_check = execute_query(sql)
        reckon_count = 0
        for p in platform_check:
            is_reckon = p['DestinationSW'] == 'Reckon'
            if is_reckon:
                reckon_count += 1
            marker = '  <-- RECKON FILE' if is_reckon else ''
            print(f"  OrderID={p['ID']:<10} {p['Order_Number']:<20} "
                  f"DestinationSW={p['DestinationSW']!r:<15} "
                  f"ReviewDate={p['Date_OrderReviewStatus']} CompletionOn={p['CompletionOn']} "
                  f"Gap={p['GapDays']}{marker}")

        print(f'\nReckon files in this employee\'s May data: {reckon_count} / {len(order_ids)}')
        if reckon_count == 0:
            print('\n>>> AdminID 303 has ZERO Reckon files this month.')
            print('>>> The missing_status Reckon fix will NOT change anything for this')
            print('>>> employee. Need a DIFFERENT test employee who actually handles')
            print('>>> Reckon files to properly verify that specific fix.')
else:
    print('No rows found for this AdminID — check if 303 is correct for this month.')

print('\n' + '=' * 78)
print('Finding an employee who DOES have Reckon files this month (better test case)')
print('=' * 78)
sql2 = f"""
    SELECT TOP 10
        COALESCE(m.PickedUpBy, m.Summary_AssignUserID) AS AdminID,
        a.Admin_FirstName + ' ' + a.Admin_LastName AS EmployeeName,
        COUNT(*) AS ReckonFileCount
    FROM OrderDetails od
    INNER JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
    LEFT JOIN Admin a ON a.ID_Admin = COALESCE(m.PickedUpBy, m.Summary_AssignUserID)
    WHERE od.DestinationSW = 'Reckon'
      AND m.Flag_OrderCompleted = 1
      AND m.Date_OrderReviewStatus IS NOT NULL
      AND MONTH(m.Date_OrderReviewStatus) = {MONTH} AND YEAR(m.Date_OrderReviewStatus) = {YEAR}
    GROUP BY COALESCE(m.PickedUpBy, m.Summary_AssignUserID), a.Admin_FirstName, a.Admin_LastName
    ORDER BY ReckonFileCount DESC
"""
top_reckon_handlers = execute_query(sql2)
for t in top_reckon_handlers:
    print(f"  AdminID={t['AdminID']:<8} {t['EmployeeName']:<25} ReckonFiles={t['ReckonFileCount']}")

[MissingStatus] Per-file: 466 rows
Rows for AdminID=303: 1
EmployeeName: Aarti Jagtap
WorkingDaysInMonth: 26
DaysWithUpdate: 18
MissedDays: 8

Aggregate: 18/26 = 69.23%

Per-file breakdown (20 files):
  MMC26042533340619    Stevenette                     completed=2026-05-01 active_from=2026-04-25 active_to=2026-04-30 active=5 updated=1 missed=4
  MMC26043034490400    Michael Hayman                 completed=2026-05-04 active_from=2026-05-01 active_to=2026-05-02 active=2 updated=1 missed=1
  MMC26050235060612    NewCo Enterprises Australia Pty Ltd  completed=2026-05-08 active_from=2026-05-02 active_to=2026-05-07 active=5 updated=1 missed=4
  MMC26050635801031    Accent Live 11                 completed=2026-05-09 active_from=2026-05-07 active_to=2026-05-08 active=2 updated=1 missed=1
  MMC26050736181205    Solar Advice (Pty) Ltd         completed=2026-05-12 active_from=2026-05-07 active_to=2026-05-11 active=4 updated=1 missed=3
  MMC26050936620623    Brady NewCity                  comp

TypeError: unsupported format string passed to NoneType.__format__

In [41]:
"""
JUPYTER CELL — verify_completion_day_inclusion.py
=====================================================
Issue 1 claim: denominator should NOT include the completion day
itself (only a completion EMAIL goes out that day, no status update
expected). Checking the CURRENT live SQL directly to see whether
WorkDay <= CompletedDate (inclusive) is really producing one extra day
in the denominator versus what it should be.
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

MONTH, YEAR = 5, 2026

# Run the EXACT current FileActiveDays logic in isolation, for AdminID
# 303's real files, showing the raw day-by-day WorkingDays join so we
# can see literally which calendar dates get counted for one file.
sql = f"""
DECLARE @Month INT={MONTH}
DECLARE @Year  INT={YEAR}
;WITH
WorkingDays AS (
    SELECT CAST(DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)) AS DATE) AS WorkDay
    FROM (SELECT TOP 31 ROW_NUMBER() OVER (ORDER BY (SELECT NULL))-1 AS n
          FROM master..spt_values) nums
    WHERE MONTH(DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)))=@Month
      AND DATEPART(WEEKDAY,DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)))<>1
)
SELECT od.Order_Number, od.CompanyName,
       CAST(od.CompletionOn AS DATE) AS CompletedDate,
       wd.WorkDay
FROM OrderDetails od
CROSS JOIN WorkingDays wd
WHERE od.Order_Number = 'MMC26051437821101'   -- Harmon Appointers, from real output
  AND wd.WorkDay BETWEEN '2026-05-10' AND '2026-05-16'
ORDER BY wd.WorkDay
"""
print('Raw WorkingDays rows for Harmon Appointers around its completion date:')
rows = execute_query(sql)
for r in rows:
    print(f"  WorkDay={r['WorkDay']}  CompletedDate={r['CompletedDate']}  "
          f"{'<= completion (INCLUDED by current logic)' if r['WorkDay'] <= r['CompletedDate'] else '> completion (excluded)'}")

print('\n' + '=' * 78)
print('Now checking: what does Order_Current_Status / Flag_OrderCompleted')
print('actually represent for this file -- is CompletionOn the COMPLETION')
print('MAIL date, or a different event?')
print('=' * 78)
sql2 = """
    SELECT od.Order_Number, od.CompletionOn, m.Date_Created, m.Flag_OrderCompleted,
           m.Date_OrderReviewStatus, m.Summary_OrderStatusName
    FROM OrderDetails od
    INNER JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
    WHERE od.Order_Number = 'MMC26051437821101'
"""
detail = execute_query(sql2)
for d in detail:
    print(f"  CompletionOn={d['CompletionOn']}  Date_Created={d['Date_Created']}  "
          f"Flag_OrderCompleted={d['Flag_OrderCompleted']}  "
          f"Summary_OrderStatusName={d['Summary_OrderStatusName']!r}")

print('\n' + '=' * 78)
print('Checking actual status-history entries around the completion date —')
print('does an update get logged ON the completion day itself in practice?')
print('=' * 78)
sql3 = """
    SELECT h.Date_Ctreated, h.StatusID, h.AssignedTo_UserID
    FROM Table_OrderStatushistory h
    INNER JOIN OrderDetails od ON od.ID = h.OrderID
    WHERE od.Order_Number = 'MMC26051437821101'
    ORDER BY h.Date_Ctreated
"""
history = execute_query(sql3)
for h in history:
    print(f"  {h['Date_Ctreated']}  StatusID={h['StatusID']}  AssignedTo={h['AssignedTo_UserID']}")

Raw WorkingDays rows for Harmon Appointers around its completion date:
  WorkDay=2026-05-11  CompletedDate=2026-05-15  <= completion (INCLUDED by current logic)
  WorkDay=2026-05-12  CompletedDate=2026-05-15  <= completion (INCLUDED by current logic)
  WorkDay=2026-05-13  CompletedDate=2026-05-15  <= completion (INCLUDED by current logic)
  WorkDay=2026-05-14  CompletedDate=2026-05-15  <= completion (INCLUDED by current logic)
  WorkDay=2026-05-15  CompletedDate=2026-05-15  <= completion (INCLUDED by current logic)
  WorkDay=2026-05-16  CompletedDate=2026-05-15  > completion (excluded)

Now checking: what does Order_Current_Status / Flag_OrderCompleted
actually represent for this file -- is CompletionOn the COMPLETION
MAIL date, or a different event?
  CompletionOn=2026-05-15 16:33:05.647000  Date_Created=2026-05-14 11:02:27.407000  Flag_OrderCompleted=True  Summary_OrderStatusName=None

Checking actual status-history entries around the completion date —
does an update get logged ON th

In [42]:
"""
JUPYTER CELL — verify_missing_file_for_303.py
==================================================
Issue 2 claim: there's one fewer file for Aarti Jagtap than there
should be -- a file exists with a BLANK missing-status value (not
appearing at all, or appearing with nulls) instead of a real number.

Strategy: find EVERY file attributed to AdminID 303 in May via the
SAME attribution logic missing_status.py uses (COALESCE(PickedUpBy,
AssignedTo_UserID, Summary_AssignUserID)), filtered by completion
month -- then compare against the 20 files the KPI actually returned,
to find what's missing and WHY (null date? excluded by some filter?).
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

MONTH, YEAR = 5, 2026
TEST_ADMIN_ID = 303

# Every file COMPLETED in May, attributed to 303 via the same COALESCE
# chain missing_status.py's FilesCompleted CTE uses -- WITHOUT the
# Admin.Flag_Active filter, to see if that filter is excluding anything
sql = f"""
SELECT od.ID, od.Order_Number, od.CompanyName,
       od.CompletionOn,
       m.PickedUpBy, m.Summary_AssignUserID,
       COALESCE(m.PickedUpBy, m.Summary_AssignUserID) AS ResolvedAdminNoHistory,
       m.Flag_OrderCompleted,
       od.Flag_Deleted
FROM OrderDetails od
INNER JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
WHERE COALESCE(m.PickedUpBy, m.Summary_AssignUserID) = {TEST_ADMIN_ID}
  AND MONTH(od.CompletionOn) = {MONTH} AND YEAR(od.CompletionOn) = {YEAR}
ORDER BY od.CompletionOn
"""
all_files = execute_query(sql)
print(f'Files attributed to AdminID {TEST_ADMIN_ID} in May 2026 (by PickedUpBy/Summary_AssignUserID only): {len(all_files)}')
for f in all_files:
    print(f"  {f['Order_Number']:<20} {f['CompanyName']:<30} CompletionOn={f['CompletionOn']} "
          f"Flag_OrderCompleted={f['Flag_OrderCompleted']} Flag_Deleted={f['Flag_Deleted']}")

# The 20 order numbers we KNOW the KPI actually returned (from the real
# output pasted earlier this session)
known_returned = {
    'MMC26042533340619','MMC26043034490400','MMC26050235060612','MMC26050635801031',
    'MMC26050736181205','MMC26050936620623','MMC26051437821101','MMC26051437841104',
    'MMC26051437941111','MMC26050936650710','MMC26051437741056','MMC26051337230918',
    'MMC26051437380537','MMC26051437661047','MMC26041531071201','MMC26051337251014',
    'MMC26052239680537','MMC26052339810544','MMC26043034630629','MMC26042433180627',
}

print(f'\n{"="*78}')
print('Files found here but NOT in the KPI\'s actual 20-file output')
print('='*78)
missing = [f for f in all_files if f['Order_Number'].strip() not in known_returned]
for f in missing:
    print(f"  {f['Order_Number']:<20} {f['CompanyName']:<30} CompletionOn={f['CompletionOn']} "
          f"Flag_OrderCompleted={f['Flag_OrderCompleted']} Flag_Deleted={f['Flag_Deleted']}")
if not missing:
    print('  None found via this simpler attribution -- the missing file may be')
    print('  attributed via the LastAlloc/status-history fallback instead. Checking that next.')

print(f'\n{"="*78}')
print('Same check but including the LastAlloc (status-history) fallback too')
print('(the FULL COALESCE chain missing_status.py actually uses)')
print('='*78)
sql2 = f"""
;WITH
LastAllocated AS (
    SELECT h.OrderID, h.AssignedTo_UserID,
        ROW_NUMBER() OVER (PARTITION BY h.OrderID ORDER BY h.Date_Ctreated DESC) AS rn
    FROM Table_OrderStatushistory h
    WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
),
LastAlloc AS (SELECT OrderID, AssignedTo_UserID FROM LastAllocated WHERE rn=1)
SELECT od.ID, od.Order_Number, od.CompanyName, od.CompletionOn,
       m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID,
       COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID) AS FullResolvedAdmin,
       m.Flag_OrderCompleted, od.Flag_Deleted
FROM OrderDetails od
INNER JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
LEFT JOIN LastAlloc la ON la.OrderID = od.ID
WHERE COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID) = {TEST_ADMIN_ID}
  AND MONTH(od.CompletionOn) = {MONTH} AND YEAR(od.CompletionOn) = {YEAR}
ORDER BY od.CompletionOn
"""
all_files_full = execute_query(sql2)
print(f'Files via FULL attribution chain: {len(all_files_full)}')
missing_full = [f for f in all_files_full if f['Order_Number'].strip() not in known_returned]
print(f'\nOf those, NOT in the KPI\'s actual output: {len(missing_full)}')
for f in missing_full:
    print(f"  {f['Order_Number']:<20} {f['CompanyName']:<30} CompletionOn={f['CompletionOn']} "
          f"Flag_OrderCompleted={f['Flag_OrderCompleted']} Flag_Deleted={f['Flag_Deleted']}")

Files attributed to AdminID 303 in May 2026 (by PickedUpBy/Summary_AssignUserID only): 14
  MMC26042533340619    Stevenette                     CompletionOn=2026-05-01 18:22:13.343000 Flag_OrderCompleted=True Flag_Deleted=False
  MMC26050635670629    The Australian Motorlife Museum CompletionOn=2026-05-07 17:10:41.050000 Flag_OrderCompleted=True Flag_Deleted=False
  MMC26050235060612    NewCo Enterprises Australia Pty Ltd  CompletionOn=2026-05-08 16:17:12.413000 Flag_OrderCompleted=True Flag_Deleted=False
  MMC26050635801031    Accent Live 11                 CompletionOn=2026-05-09 11:42:45.947000 Flag_OrderCompleted=True Flag_Deleted=False
  MMC26050736181205    Solar Advice (Pty) Ltd         CompletionOn=2026-05-12 17:05:40.383000 Flag_OrderCompleted=True Flag_Deleted=False
  MMC26050936620623    Brady NewCity                  CompletionOn=2026-05-14 16:56:06.573000 Flag_OrderCompleted=True Flag_Deleted=False
  MMC26050936650710    MG Warwick Street Holdco Limited CompletionOn=2026-0

In [43]:
"""
JUPYTER CELL — verify_email_status_updates.py
==================================================
Issue 3 claim: some files have a status update sent via EMAIL, but the
KPI still shows a low/missed count for those days -- suggesting
email-based updates aren't being captured by the
Table_OrderStatushistory-based DaysUpdated count.

This checks: does Table_OrderStatushistory actually LOG an entry when
a status update is sent via email, or is email a SEPARATE channel that
never creates a Table_OrderStatushistory row at all? If the latter,
that's the real bug -- the KPI only counts in-system status changes,
completely blind to anything communicated via email.
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

MONTH, YEAR = 5, 2026
TEST_ADMIN_ID = 303

# Pick one of 303's real files with a HIGH missed count and LOW updated
# count, and look for ANY email-related fields/tables that might record
# a separate update channel.
print('Checking Table_OrderDetailsMisc for any email-related status fields...')
sql = "SELECT TOP 1 * FROM Table_OrderDetailsMisc"
cols = execute_query(sql)
if cols:
    all_cols = list(cols[0].keys())
    email_related = [c for c in all_cols if 'email' in c.lower() or 'mail' in c.lower()
                      or 'sent' in c.lower() or 'notif' in c.lower()]
    print(f'Email/notification-related columns found: {email_related}')

print('\nChecking OrderDetails for the same...')
sql2 = "SELECT TOP 1 * FROM OrderDetails"
cols2 = execute_query(sql2)
if cols2:
    all_cols2 = list(cols2[0].keys())
    email_related2 = [c for c in all_cols2 if 'email' in c.lower() or 'mail' in c.lower()
                       or 'sent' in c.lower() or 'notif' in c.lower()]
    print(f'Email/notification-related columns found: {email_related2}')

# Specific high-missed-count file from real output: SB Comm Inc,
# active=25, updated=4, missed=21 -- a LOT of missed days for one file
print('\n' + '=' * 78)
print('SB Comm Inc — full status history (everything logged in-system)')
print('=' * 78)
sql3 = """
    SELECT h.Date_Ctreated, h.StatusID, h.AssignedTo_UserID
    FROM Table_OrderStatushistory h
    INNER JOIN OrderDetails od ON od.ID = h.OrderID
    WHERE od.Order_Number = 'MMC26043034630629'
    ORDER BY h.Date_Ctreated
"""
history = execute_query(sql3)
print(f'Total status-history entries: {len(history)}')
for h in history:
    print(f"  {h['Date_Ctreated']}  StatusID={h['StatusID']}  AssignedTo={h['AssignedTo_UserID']}")

print('\nChecking if there is a SEPARATE table for emails/communications sent...')
sql4 = """
    SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_NAME LIKE '%mail%' OR TABLE_NAME LIKE '%email%'
       OR TABLE_NAME LIKE '%notif%' OR TABLE_NAME LIKE '%communication%'
"""
mail_tables = execute_query(sql4)
print(f'Tables matching mail/email/notification/communication: {len(mail_tables)}')
for t in mail_tables:
    print(f"  {t['TABLE_NAME']}")

Checking Table_OrderDetailsMisc for any email-related status fields...
Email/notification-related columns found: ['SentOn', 'PendingInfoEmail_Count', 'QuotationSentCount', 'InvoiceSentDate']

Checking OrderDetails for the same...
Email/notification-related columns found: ['Email', 'XeroSubscriptionEmail', 'FBRepEmailAddress', 'ReckonRepresentative']

SB Comm Inc — full status history (everything logged in-system)
Total status-history entries: 9
  2026-04-30 06:29:39.990000  StatusID=14  AssignedTo=None
  2026-04-30 06:30:43.427000  StatusID=1  AssignedTo=None
  2026-04-30 17:18:36.793000  StatusID=11  AssignedTo=303
  2026-04-30 17:42:31.920000  StatusID=45  AssignedTo=303
  2026-05-05 17:29:41.643000  StatusID=8  AssignedTo=303
  2026-05-20 05:35:44.927000  StatusID=3  AssignedTo=17
  2026-05-20 11:50:20.643000  StatusID=8  AssignedTo=303
  2026-05-27 05:13:51.213000  StatusID=3  AssignedTo=17
  2026-05-29 18:22:58.387000  StatusID=8  AssignedTo=None

Checking if there is a SEPARATE t

In [44]:
"""
JUPYTER CELL — verify_issue2_and_3_root_causes.py
======================================================
Issue 2: confirm WHY Australian Motorlife Museum vanishes -- is
ActiveFrom genuinely landing after CompletedDate?

Issue 3: check OrderEmails / Table_OrderStatusEmail / Table_EmailNotificationHistory
for SB Comm Inc during its real "missed" gap days (May 6-19, May 21-26)
-- if emails were sent then, that's the proof email updates should count.
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

print('=' * 78)
print('ISSUE 2 — Australian Motorlife Museum: ActiveFrom vs CompletedDate')
print('=' * 78)
sql = """
;WITH
LastAllocated AS (
    SELECT h.OrderID, h.AssignedTo_UserID,
        ROW_NUMBER() OVER (PARTITION BY h.OrderID ORDER BY h.Date_Ctreated DESC) AS rn
    FROM Table_OrderStatushistory h
    WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
),
LastAlloc AS (SELECT OrderID, AssignedTo_UserID FROM LastAllocated WHERE rn=1),
FirstAllocDate AS (
    SELECT h.OrderID, h.AssignedTo_UserID, CAST(MIN(h.Date_Ctreated) AS DATE) AS FirstDate
    FROM Table_OrderStatushistory h
    WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
    GROUP BY h.OrderID, h.AssignedTo_UserID
)
SELECT od.Order_Number, od.CompanyName,
       CAST(od.CompletionOn AS DATE) AS CompletedDate,
       la.AssignedTo_UserID AS LastAllocAdmin,
       fad.FirstDate AS FirstAllocDate_ForLastAllocAdmin
FROM OrderDetails od
LEFT JOIN LastAlloc la ON la.OrderID = od.ID
LEFT JOIN FirstAllocDate fad ON fad.OrderID = od.ID AND fad.AssignedTo_UserID = la.AssignedTo_UserID
WHERE od.Order_Number = 'MMC26050635670629'
"""
result = execute_query(sql)
for r in result:
    print(f"  CompletedDate={r['CompletedDate']}  LastAllocAdmin={r['LastAllocAdmin']}  "
          f"FirstAllocDate(for that admin)={r['FirstAllocDate_ForLastAllocAdmin']}")
    if r['FirstAllocDate_ForLastAllocAdmin'] and r['CompletedDate']:
        if r['FirstAllocDate_ForLastAllocAdmin'] > r['CompletedDate']:
            print('  >>> CONFIRMED: FirstAllocDate is AFTER CompletedDate -- this is why')
            print('  >>> the INNER JOIN to WorkingDays produces an empty range and the')
            print('  >>> file silently vanishes instead of showing 0 days.')
        else:
            print('  >>> FirstAllocDate is NOT after CompletedDate -- different root cause needed.')

print('\nFull status history for this file, to see the real allocation timeline:')
sql_hist = """
    SELECT h.Date_Ctreated, h.StatusID, h.AssignedTo_UserID
    FROM Table_OrderStatushistory h
    INNER JOIN OrderDetails od ON od.ID = h.OrderID
    WHERE od.Order_Number = 'MMC26050635670629'
    ORDER BY h.Date_Ctreated
"""
hist = execute_query(sql_hist)
for h in hist:
    print(f"  {h['Date_Ctreated']}  StatusID={h['StatusID']}  AssignedTo={h['AssignedTo_UserID']}")

print('\n' + '=' * 78)
print('ISSUE 3 — SB Comm Inc: were emails sent during the "missed" gap days?')
print('=' * 78)
for table in ['OrderEmails', 'Table_OrderStatusEmail', 'Table_EmailNotificationHistory']:
    print(f'\nChecking {table}...')
    try:
        cols = execute_query(f"SELECT TOP 1 * FROM {table}")
        if cols:
            print(f'  Columns: {list(cols[0].keys())}')
        sql_email = f"""
            SELECT * FROM {table}
            WHERE OrderID = (SELECT ID FROM OrderDetails WHERE Order_Number = 'MMC26043034630629')
        """
        emails = execute_query(sql_email)
        print(f'  Rows for SB Comm Inc: {len(emails)}')
        for e in emails[:10]:
            print(f'    {dict(e)}')
    except Exception as ex:
        print(f'  ERROR or no OrderID column: {ex}')

ISSUE 2 — Australian Motorlife Museum: ActiveFrom vs CompletedDate
  CompletedDate=2026-05-07  LastAllocAdmin=303  FirstAllocDate(for that admin)=2026-05-07
  >>> FirstAllocDate is NOT after CompletedDate -- different root cause needed.

Full status history for this file, to see the real allocation timeline:
  2026-05-06 06:29:43.663000  StatusID=14  AssignedTo=None
  2026-05-06 06:31:22.487000  StatusID=1  AssignedTo=None
  2026-05-07 05:21:20.520000  StatusID=11  AssignedTo=303
  2026-05-07 07:51:09.463000  StatusID=45  AssignedTo=303
  2026-05-07 17:10:41.020000  StatusID=8  AssignedTo=None

ISSUE 3 — SB Comm Inc: were emails sent during the "missed" gap days?

Checking OrderEmails...
  Columns: ['Id', 'ToEmail', 'TemplateName', 'Parameters', 'IsSuccess', 'Counter', 'SentDate', 'CreatedDate', 'TempId', 'IsComplete', 'CCEmail', 'BCCEmail', 'AWS_S3_Download_URL']
  ERROR or no OrderID column: ('42S22', "[42S22] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Invalid column name 

In [45]:
"""
JUPYTER CELL — verify_order_conversion_status_emails.py
============================================================
Checking if 'Order Conversion Status' is a recurring/periodic status
update email (distinct from the one-time completion email we found for
SB Comm Inc) -- and whether it exists for our test files: Motorlife
Museum (Issue 2 — vanishing file) and Aarti Jagtap's other May files
(Issue 3 — undercounted despite real updates being sent).
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

print('=' * 78)
print('All Table_EmailNotificationHistory rows where Email_Name LIKE Order Conversion Status')
print('=' * 78)
sql = """
    SELECT TOP 20 ID_Notification, Email_Name, EMail_Subject, Date_Created, OrderID
    FROM Table_EmailNotificationHistory
    WHERE Email_Name LIKE '%Order Conversion Status%'
    ORDER BY Date_Created DESC
"""
rows = execute_query(sql)
print(f'Total matching (showing most recent 20): {len(rows)}')
for r in rows:
    print(f"  OrderID={r['OrderID']:<8} Date={r['Date_Created']}  Subject={r['EMail_Subject']!r}")

print('\n' + '=' * 78)
print('Distinct Email_Name values containing "Status" or "Conversion" — full picture')
print('of every status-related email template that exists')
print('=' * 78)
sql2 = """
    SELECT DISTINCT Email_Name, COUNT(*) AS cnt
    FROM Table_EmailNotificationHistory
    WHERE Email_Name LIKE '%Status%' OR Email_Name LIKE '%Conversion%'
    GROUP BY Email_Name
    ORDER BY cnt DESC
"""
names = execute_query(sql2)
for n in names:
    print(f"  {n['Email_Name']!r}: {n['cnt']} emails sent")

print('\n' + '=' * 78)
print('Order Conversion Status emails for Motorlife Museum (OrderID lookup)')
print('=' * 78)
sql3 = """
    SELECT e.ID_Notification, e.Email_Name, e.EMail_Subject, e.Date_Created
    FROM Table_EmailNotificationHistory e
    INNER JOIN OrderDetails od ON od.ID = e.OrderID
    WHERE od.Order_Number = 'MMC26050635670629'
    ORDER BY e.Date_Created
"""
motorlife_emails = execute_query(sql3)
print(f'Total emails for this file: {len(motorlife_emails)}')
for m in motorlife_emails:
    print(f"  {m['Date_Created']}  {m['Email_Name']!r}  {m['EMail_Subject']!r}")

print('\n' + '=' * 78)
print('Order Conversion Status emails for ALL of AdminID 303\'s May files')
print('(to see if these emails happen DURING the active/missed-day window,')
print(' not just at start/completion)')
print('=' * 78)
sql4 = """
    SELECT od.Order_Number, od.CompanyName, e.Date_Created, e.Email_Name, e.EMail_Subject
    FROM Table_EmailNotificationHistory e
    INNER JOIN OrderDetails od ON od.ID = e.OrderID
    INNER JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
    WHERE COALESCE(m.PickedUpBy, m.Summary_AssignUserID) = 303
      AND MONTH(od.CompletionOn) = 5 AND YEAR(od.CompletionOn) = 2026
      AND (e.Email_Name LIKE '%Status%' OR e.Email_Name LIKE '%Conversion%')
    ORDER BY od.Order_Number, e.Date_Created
"""
all_303_emails = execute_query(sql4)
print(f'Total: {len(all_303_emails)}')
current_order = None
for e in all_303_emails:
    if e['Order_Number'] != current_order:
        print(f"\n  {e['Order_Number']} — {e['CompanyName']}")
        current_order = e['Order_Number']
    print(f"    {e['Date_Created']}  {e['Email_Name']!r}")

All Table_EmailNotificationHistory rows where Email_Name LIKE Order Conversion Status
Total matching (showing most recent 20): 20
  OrderID=0        Date=2026-06-27 18:25:04.873000  Subject='Conversion Status : Intercap Services LLC : QuickBooks Online to Retail Migration : 27/06/2026'
  OrderID=0        Date=2026-06-27 18:25:04.707000  Subject='Conversion Status : Abraham Lowinger : FreshBooks to Retail Migration : 27/06/2026'
  OrderID=0        Date=2026-06-27 18:25:04.460000  Subject='Conversion Status : Stucco And Polished Plaster Limited’s : Zoho Books to Retail Migration : 27/06/2026'
  OrderID=0        Date=2026-06-27 18:25:04.273000  Subject='Conversion Status : Strategic Lawyers : Xero to Xero : 27/06/2026'
  OrderID=0        Date=2026-06-27 18:25:04.090000  Subject='Conversion Status : Avon Service Specialists : Reckon (Desktop) to Retail Migration : 27/06/2026'
  OrderID=0        Date=2026-06-27 18:25:03.893000  Subject='Conversion Status : Macktez Corporation : Account Edge

In [48]:
"""
JUPYTER CELL — verify_motorlife_direct.py
=============================================
Both prior theories for issue 2 are disproven by real data. Running the
ACTUAL FileActiveDays CTE logic directly for this one file, to see with
certainty whether it produces a row, and if not, exactly which JOIN or
WHERE condition is silently excluding it.
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

MONTH, YEAR = 5, 2026

sql = f"""
DECLARE @Month INT={MONTH}
DECLARE @Year  INT={YEAR}
;WITH
LastAllocated AS (
    SELECT h.OrderID, h.AssignedTo_UserID,
        ROW_NUMBER() OVER (PARTITION BY h.OrderID ORDER BY h.Date_Ctreated DESC) AS rn
    FROM Table_OrderStatushistory h
    WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
),
LastAlloc AS (SELECT OrderID, AssignedTo_UserID FROM LastAllocated WHERE rn=1),
FirstAllocDate AS (
    SELECT h.OrderID, h.AssignedTo_UserID, CAST(MIN(h.Date_Ctreated) AS DATE) AS FirstDate
    FROM Table_OrderStatushistory h
    WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
    GROUP BY h.OrderID, h.AssignedTo_UserID
),
WorkingDays AS (
    SELECT CAST(DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)) AS DATE) AS WorkDay
    FROM (SELECT TOP 31 ROW_NUMBER() OVER (ORDER BY (SELECT NULL))-1 AS n
          FROM master..spt_values) nums
    WHERE MONTH(DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)))=@Month
      AND DATEPART(WEEKDAY,DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)))<>1
),
FilesCompleted AS (
    SELECT
        la.AssignedTo_UserID AS AdminID,
        od.ID AS OrderID,
        od.Order_Number AS OrderNumber,
        od.CompanyName,
        CAST(od.CompletionOn AS DATE) AS CompletedDate,
        CASE
            WHEN CAST(fad.FirstDate AS DATE) >= DATEFROMPARTS(@Year,@Month,1)
             AND CAST(fad.FirstDate AS DATE) <= CAST(od.CompletionOn AS DATE)
            THEN CAST(fad.FirstDate AS DATE)
            ELSE DATEFROMPARTS(@Year,@Month,1)
        END AS ActiveFrom
    FROM OrderDetails od
    INNER JOIN LastAlloc      la  ON la.OrderID=od.ID
    LEFT  JOIN FirstAllocDate fad ON fad.OrderID=od.ID AND fad.AssignedTo_UserID=la.AssignedTo_UserID
    LEFT  JOIN Table_OrderDetailsMisc m ON m.OrderID=od.ID
    INNER JOIN Admin af ON af.ID_Admin=la.AssignedTo_UserID
                       AND af.Flag_Active=1 AND (af.Flag_Delete=0 OR af.Flag_Delete IS NULL)
    WHERE od.Flag_Deleted=0
      AND MONTH(od.CompletionOn)=@Month AND YEAR(od.CompletionOn)=@Year
      AND od.Order_Number = 'MMC26050635670629'
)
SELECT * FROM FilesCompleted
"""
print('Step 1 — does this file survive the FilesCompleted CTE at all?')
result = execute_query(sql)
print(f'Rows returned: {len(result)}')
for r in result:
    print(f"  AdminID={r['AdminID']}  ActiveFrom={r['ActiveFrom']}  CompletedDate={r['CompletedDate']}")

if not result:
    print('\n>>> FAILS at FilesCompleted itself. Checking WHICH join/filter excludes it:')

    print('\n  Check 1: Is there a LastAlloc row at all for this OrderID?')
    sql_check1 = """
        ;WITH LastAllocated AS (
            SELECT h.OrderID, h.AssignedTo_UserID,
                ROW_NUMBER() OVER (PARTITION BY h.OrderID ORDER BY h.Date_Ctreated DESC) AS rn
            FROM Table_OrderStatushistory h
            WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
        )
        SELECT la.OrderID, la.AssignedTo_UserID, la.rn
        FROM LastAllocated la
        INNER JOIN OrderDetails od ON od.ID = la.OrderID
        WHERE od.Order_Number = 'MMC26050635670629'
    """
    c1 = execute_query(sql_check1)
    print(f'    LastAllocated rows (all rn values): {len(c1)}')
    for r in c1:
        print(f"      OrderID={r['OrderID']} AssignedTo={r['AssignedTo_UserID']} rn={r['rn']}")

    print('\n  Check 2: Is the Admin (303) Flag_Active=1 and not deleted?')
    sql_check2 = """
        SELECT ID_Admin, Admin_FirstName, Admin_LastName, Flag_Active, Flag_Delete
        FROM Admin WHERE ID_Admin = 303
    """
    c2 = execute_query(sql_check2)
    for r in c2:
        print(f"    ID_Admin=303: Flag_Active={r['Flag_Active']} Flag_Delete={r['Flag_Delete']} "
              f"Name={r['Admin_FirstName']} {r['Admin_LastName']}")

    print('\n  Check 3: od.Flag_Deleted value for this order?')
    sql_check3 = """
        SELECT ID, Order_Number, Flag_Deleted, CompletionOn
        FROM OrderDetails WHERE Order_Number = 'MMC26050635670629'
    """
    c3 = execute_query(sql_check3)
    for r in c3:
        print(f"    OrderID={r['ID']} Flag_Deleted={r['Flag_Deleted']} CompletionOn={r['CompletionOn']}")

Step 1 — does this file survive the FilesCompleted CTE at all?
Rows returned: 1
  AdminID=303  ActiveFrom=2026-05-07  CompletedDate=2026-05-07


In [49]:
"""
JUPYTER CELL — verify_missing_status_email_template.py
============================================================
'Running Conversions - Missing Status Update' (5,645 sent) sounds like
it could be directly relevant -- checking who it's sent TO and what it
contains, to determine if it's:
(a) a notification SENT TO THE EMPLOYEE reminding them to update status
    (meaning it does NOT count as an update itself), or
(b) an actual status communication that should count, or
(c) an internal/management alert about missing updates (relevant context,
    not something to credit as an update)
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

sql = """
    SELECT TOP 5 ID_Notification, Email_Name, EMail_Subject, Email_TO,
           Email_CC, Email_BCC, Email_From, Date_Created, OrderID
    FROM Table_EmailNotificationHistory
    WHERE Email_Name = 'Running Conversions - Missing Status Update'
    ORDER BY Date_Created DESC
"""
rows = execute_query(sql)
for r in rows:
    print(f"OrderID={r['OrderID']}")
    print(f"  Subject: {r['EMail_Subject']}")
    print(f"  To: {r['Email_TO']}")
    print(f"  CC: {r['Email_CC']}")
    print(f"  From: {r['Email_From']}")
    print(f"  Date: {r['Date_Created']}")
    print()

print('=' * 78)
print('Does this email exist for any of Aarti Jagtap\'s May files?')
print('=' * 78)
sql2 = """
    SELECT od.Order_Number, od.CompanyName, e.Date_Created, e.Email_TO
    FROM Table_EmailNotificationHistory e
    INNER JOIN OrderDetails od ON od.ID = e.OrderID
    INNER JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
    WHERE COALESCE(m.PickedUpBy, m.Summary_AssignUserID) = 303
      AND MONTH(od.CompletionOn) = 5 AND YEAR(od.CompletionOn) = 2026
      AND e.Email_Name = 'Running Conversions - Missing Status Update'
    ORDER BY od.Order_Number, e.Date_Created
"""
matches = execute_query(sql2)
print(f'Total: {len(matches)}')
for m in matches:
    print(f"  {m['Order_Number']} — {m['CompanyName']}  sent={m['Date_Created']}  to={m['Email_TO']}")

OrderID=0
  Subject: Odoo Retail : Running Conversions - Missing Status Update  : 27/06/2026
  To: shraddha.farkiya@beyondkey.com
  CC: shraddha.farkiya@beyondkey.com
  From: info@mmcconvert.com
  Date: 2026-06-27 18:25:07.827000

OrderID=0
  Subject: Zoho Books Retail : Running Conversions - Missing Status Update  : 27/06/2026
  To: ankit@mmcconvert.com
  CC: shraddha.farkiya@beyondkey.com
  From: migrations@mmcconvert.com
  Date: 2026-06-27 18:25:07.547000

OrderID=0
  Subject: Xero Retail : Running Conversions - Missing Status Update  : 27/06/2026
  To: ankit@mmcconvert.com
  CC: shraddha.farkiya@beyondkey.com
  From: migrations@mmcconvert.com
  Date: 2026-06-27 18:25:07.260000

OrderID=0
  Subject: RECKON : Running Conversions - Missing Status Update  : 27/06/2026
  To: ankit@mmcconvert.com
  CC: shraddha.farkiya@beyondkey.com
  From: reckon@mmcconvert.com
  Date: 2026-06-27 18:25:06.910000

OrderID=0
  Subject: MYOB : Running Conversions - Missing Status Update  : 27/06/2026
  To:

In [51]:
"""
JUPYTER CELL — verify_motorlife_direct.py
=============================================
Both prior theories for issue 2 are disproven by real data. Running the
ACTUAL FileActiveDays CTE logic directly for this one file, to see with
certainty whether it produces a row, and if not, exactly which JOIN or
WHERE condition is silently excluding it.
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

MONTH, YEAR = 5, 2026

sql = f"""
DECLARE @Month INT={MONTH}
DECLARE @Year  INT={YEAR}
;WITH
LastAllocated AS (
    SELECT h.OrderID, h.AssignedTo_UserID,
        ROW_NUMBER() OVER (PARTITION BY h.OrderID ORDER BY h.Date_Ctreated DESC) AS rn
    FROM Table_OrderStatushistory h
    WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
),
LastAlloc AS (SELECT OrderID, AssignedTo_UserID FROM LastAllocated WHERE rn=1),
FirstAllocDate AS (
    SELECT h.OrderID, h.AssignedTo_UserID, CAST(MIN(h.Date_Ctreated) AS DATE) AS FirstDate
    FROM Table_OrderStatushistory h
    WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
    GROUP BY h.OrderID, h.AssignedTo_UserID
),
WorkingDays AS (
    SELECT CAST(DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)) AS DATE) AS WorkDay
    FROM (SELECT TOP 31 ROW_NUMBER() OVER (ORDER BY (SELECT NULL))-1 AS n
          FROM master..spt_values) nums
    WHERE MONTH(DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)))=@Month
      AND DATEPART(WEEKDAY,DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)))<>1
),
FilesCompleted AS (
    SELECT
        la.AssignedTo_UserID AS AdminID,
        od.ID AS OrderID,
        od.Order_Number AS OrderNumber,
        od.CompanyName,
        CAST(od.CompletionOn AS DATE) AS CompletedDate,
        CASE
            WHEN CAST(fad.FirstDate AS DATE) >= DATEFROMPARTS(@Year,@Month,1)
             AND CAST(fad.FirstDate AS DATE) <= CAST(od.CompletionOn AS DATE)
            THEN CAST(fad.FirstDate AS DATE)
            ELSE DATEFROMPARTS(@Year,@Month,1)
        END AS ActiveFrom
    FROM OrderDetails od
    INNER JOIN LastAlloc      la  ON la.OrderID=od.ID
    LEFT  JOIN FirstAllocDate fad ON fad.OrderID=od.ID AND fad.AssignedTo_UserID=la.AssignedTo_UserID
    LEFT  JOIN Table_OrderDetailsMisc m ON m.OrderID=od.ID
    INNER JOIN Admin af ON af.ID_Admin=la.AssignedTo_UserID
                       AND af.Flag_Active=1 AND (af.Flag_Delete=0 OR af.Flag_Delete IS NULL)
    WHERE od.Flag_Deleted=0
      AND MONTH(od.CompletionOn)=@Month AND YEAR(od.CompletionOn)=@Year
      AND od.Order_Number = 'MMC26050635670629'
)
SELECT * FROM FilesCompleted
"""
print('Step 1 — does this file survive the FilesCompleted CTE at all?')
result = execute_query(sql)
print(f'Rows returned: {len(result)}')
for r in result:
    print(f"  AdminID={r['AdminID']}  ActiveFrom={r['ActiveFrom']}  CompletedDate={r['CompletedDate']}")

if not result:
    print('\n>>> FAILS at FilesCompleted itself. Checking WHICH join/filter excludes it:')

    print('\n  Check 1: Is there a LastAlloc row at all for this OrderID?')
    sql_check1 = """
        ;WITH LastAllocated AS (
            SELECT h.OrderID, h.AssignedTo_UserID,
                ROW_NUMBER() OVER (PARTITION BY h.OrderID ORDER BY h.Date_Ctreated DESC) AS rn
            FROM Table_OrderStatushistory h
            WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
        )
        SELECT la.OrderID, la.AssignedTo_UserID, la.rn
        FROM LastAllocated la
        INNER JOIN OrderDetails od ON od.ID = la.OrderID
        WHERE od.Order_Number = 'MMC26050635670629'
    """
    c1 = execute_query(sql_check1)
    print(f'    LastAllocated rows (all rn values): {len(c1)}')
    for r in c1:
        print(f"      OrderID={r['OrderID']} AssignedTo={r['AssignedTo_UserID']} rn={r['rn']}")

    print('\n  Check 2: Is the Admin (303) Flag_Active=1 and not deleted?')
    sql_check2 = """
        SELECT ID_Admin, Admin_FirstName, Admin_LastName, Flag_Active, Flag_Delete
        FROM Admin WHERE ID_Admin = 303
    """
    c2 = execute_query(sql_check2)
    for r in c2:
        print(f"    ID_Admin=303: Flag_Active={r['Flag_Active']} Flag_Delete={r['Flag_Delete']} "
              f"Name={r['Admin_FirstName']} {r['Admin_LastName']}")

    print('\n  Check 3: od.Flag_Deleted value for this order?')
    sql_check3 = """
        SELECT ID, Order_Number, Flag_Deleted, CompletionOn
        FROM OrderDetails WHERE Order_Number = 'MMC26050635670629'
    """
    c3 = execute_query(sql_check3)
    for r in c3:
        print(f"    OrderID={r['ID']} Flag_Deleted={r['Flag_Deleted']} CompletionOn={r['CompletionOn']}")

Step 1 — does this file survive the FilesCompleted CTE at all?
Rows returned: 1
  AdminID=303  ActiveFrom=2026-05-07  CompletedDate=2026-05-07


In [52]:
from kpis.base import BaseKPI
from db import execute_query

class MissingStatusKPI(BaseKPI):
    def fetch(self, month, year):
        # ── Aggregate query — Option A scoring (days updated / working days) ──
        # Attribution: COALESCE(PickedUpBy, AssignedTo_UserID, Summary_AssignUserID)
        # — matches post_conversion.py / delayed_conversion.py exactly.
        agg_sql = f"""
DECLARE @Month INT={month}
DECLARE @Year  INT={year}
DECLARE @MonthStart DATE = DATEFROMPARTS(@Year,@Month,1)
DECLARE @MonthEndExcl DATE = DATEADD(MONTH,1,@MonthStart)
;WITH
LastAllocated AS (
    SELECT h.OrderID, h.AssignedTo_UserID,
        ROW_NUMBER() OVER (PARTITION BY h.OrderID ORDER BY h.Date_Ctreated DESC) AS rn
    FROM Table_OrderStatushistory h
    WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
      AND h.Date_Ctreated<=EOMONTH(DATEFROMPARTS(@Year,@Month,1))
),
LastAlloc AS (SELECT OrderID, AssignedTo_UserID FROM LastAllocated WHERE rn=1),
ResolvedAttribution AS (
    SELECT
        od.ID AS OrderID,
        COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID) AS AdminID
    FROM OrderDetails od
    LEFT JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
    LEFT JOIN LastAlloc la             ON la.OrderID = od.ID
    WHERE od.Flag_Deleted = 0
      AND od.CompletionOn >= @MonthStart
      AND od.CompletionOn <  @MonthEndExcl
),
WorkingDays AS (
    SELECT CAST(DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)) AS DATE) AS WorkDay
    FROM (SELECT TOP 31 ROW_NUMBER() OVER (ORDER BY (SELECT NULL))-1 AS n
          FROM master..spt_values) nums
    WHERE MONTH(DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)))=@Month
      AND DATEPART(WEEKDAY,DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)))<>1
),
TotalWD AS (SELECT COUNT(*) AS cnt FROM WorkingDays),
ActiveEmps AS (
    SELECT DISTINCT ra.AdminID
    FROM ResolvedAttribution ra
    INNER JOIN Admin af ON af.ID_Admin=ra.AdminID
                       AND af.Flag_Active=1 AND (af.Flag_Delete=0 OR af.Flag_Delete IS NULL)
    WHERE ra.AdminID IS NOT NULL
),
UpdateDays AS (
    SELECT ra.AdminID,
        COUNT(DISTINCT CAST(h.Date_Ctreated AS DATE)) AS DaysWithUpdate
    FROM Table_OrderStatushistory h
    INNER JOIN ResolvedAttribution ra ON ra.OrderID = h.OrderID
    WHERE h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
      AND MONTH(h.Date_Ctreated)=@Month AND YEAR(h.Date_Ctreated)=@Year
      AND ra.AdminID IS NOT NULL
    GROUP BY ra.AdminID
)
SELECT ae.AdminID, a.Admin_FirstName+' '+a.Admin_LastName AS EmployeeName,
    tw.cnt AS WorkingDaysInMonth,
    ISNULL(ud.DaysWithUpdate,0) AS DaysWithUpdate,
    tw.cnt - ISNULL(ud.DaysWithUpdate,0) AS MissedDays
FROM ActiveEmps ae
CROSS JOIN TotalWD tw
LEFT  JOIN UpdateDays ud ON ud.AdminID=ae.AdminID
INNER JOIN Admin a ON a.ID_Admin=ae.AdminID
    AND a.Flag_Active=1 AND (a.Flag_Delete=0 OR a.Flag_Delete IS NULL)
ORDER BY EmployeeName"""

        try:
            agg_rows = execute_query(agg_sql)
        except Exception as e:
            print(f'[MissingStatus] Aggregate error: {e}')
            return []

        if not agg_rows:
            return []

        # ── Per-file query (display only) ─────────────────────────────────
        # FIX (Issue 1, confirmed against real data — Harmon Appointers PTY
        # LTD's status history showed the completion-day entry has
        # AssignedTo=None, i.e. it's the automated completion trigger, not
        # a human update; only a completion EMAIL goes out that day, no
        # status update is realistically expected): ActiveTo is now the
        # last WORKING DAY strictly BEFORE CompletedDate, not CompletedDate
        # itself. This removes exactly one day from each file's
        # denominator — the completion day — which was being counted as
        # "active and requiring an update" when nothing of the sort was
        # ever expected to happen that day.
        file_sql = f"""
DECLARE @Month INT={month}
DECLARE @Year  INT={year}
DECLARE @MonthStart DATE = DATEFROMPARTS(@Year,@Month,1)
DECLARE @MonthEnd   DATE = EOMONTH(@MonthStart)
DECLARE @MonthEndExcl DATE = DATEADD(MONTH,1,@MonthStart)
DECLARE @RangeStart DATE = DATEADD(DAY,-90,@MonthStart)
;WITH
LastAllocated AS (
    SELECT h.OrderID, h.AssignedTo_UserID,
        ROW_NUMBER() OVER (PARTITION BY h.OrderID ORDER BY h.Date_Ctreated DESC) AS rn
    FROM Table_OrderStatushistory h
    WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
),
LastAlloc AS (SELECT OrderID, AssignedTo_UserID FROM LastAllocated WHERE rn=1),
FirstAllocDate AS (
    SELECT h.OrderID, h.AssignedTo_UserID,
        CAST(MIN(h.Date_Ctreated) AS DATE) AS FirstDate
    FROM Table_OrderStatushistory h
    WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
    GROUP BY h.OrderID, h.AssignedTo_UserID
),
WorkingDays AS (
    SELECT CAST(DATEADD(DAY,n,@RangeStart) AS DATE) AS WorkDay
    FROM (SELECT TOP 1000 ROW_NUMBER() OVER (ORDER BY (SELECT NULL))-1 AS n
          FROM master..spt_values) nums
    WHERE DATEADD(DAY,n,@RangeStart) <= @MonthEnd
      AND DATEPART(WEEKDAY,DATEADD(DAY,n,@RangeStart))<>1
),
FilesCompleted AS (
    SELECT
        COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID) AS AdminID,
        od.ID                                                               AS OrderID,
        od.Order_Number                                                     AS OrderNumber,
        od.CompanyName,
        CAST(od.CompletionOn AS DATE)                                       AS CompletedDate,
        CASE
            WHEN fad.FirstDate IS NOT NULL
             AND CAST(fad.FirstDate AS DATE) <= CAST(od.CompletionOn AS DATE)
            THEN CAST(fad.FirstDate AS DATE)
            ELSE @MonthStart
        END                                                                 AS ActiveFromRaw
    FROM OrderDetails od
    LEFT  JOIN LastAlloc      la  ON la.OrderID=od.ID
    LEFT  JOIN FirstAllocDate fad ON fad.OrderID=od.ID
                                 AND fad.AssignedTo_UserID=la.AssignedTo_UserID
    LEFT  JOIN Table_OrderDetailsMisc m ON m.OrderID=od.ID
    INNER JOIN Admin af ON af.ID_Admin=COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID)
                       AND af.Flag_Active=1
                       AND (af.Flag_Delete=0 OR af.Flag_Delete IS NULL)
    WHERE od.Flag_Deleted=0
      AND od.CompletionOn >= @MonthStart
      AND od.CompletionOn <  @MonthEndExcl
),
-- FIX (Issue 1): ActiveTo = last WORKING DAY strictly BEFORE
-- CompletedDate (not <=, which previously included the completion day).
FilesWithActiveTo AS (
    SELECT
        fc.*,
        (SELECT MAX(wd2.WorkDay) FROM WorkingDays wd2 WHERE wd2.WorkDay < fc.CompletedDate) AS ActiveTo
    FROM FilesCompleted fc
),
FileActiveDays AS (
    SELECT
        f.AdminID, f.OrderID, f.OrderNumber,
        f.CompanyName, f.CompletedDate, f.ActiveFromRaw AS ActiveFrom, f.ActiveTo,
        COUNT(wd.WorkDay) AS ActiveWorkingDays
    FROM FilesWithActiveTo f
    INNER JOIN WorkingDays wd
           ON wd.WorkDay >= f.ActiveFromRaw
          AND wd.WorkDay <= f.ActiveTo
    GROUP BY f.AdminID, f.OrderID, f.OrderNumber,
             f.CompanyName, f.CompletedDate, f.ActiveFromRaw, f.ActiveTo
),
FileUpdateDays AS (
    SELECT h.OrderID, fad.AdminID,
        COUNT(DISTINCT CAST(h.Date_Ctreated AS DATE)) AS DaysUpdated
    FROM Table_OrderStatushistory h
    INNER JOIN FileActiveDays fad ON fad.OrderID = h.OrderID
    INNER JOIN WorkingDays wd ON CAST(h.Date_Ctreated AS DATE)=wd.WorkDay
    WHERE h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
      AND CAST(h.Date_Ctreated AS DATE) >= fad.ActiveFrom
      AND CAST(h.Date_Ctreated AS DATE) <= fad.ActiveTo
    GROUP BY h.OrderID, fad.AdminID
)
SELECT
    fad.AdminID,
    fad.OrderID,
    fad.OrderNumber,
    fad.CompanyName,
    fad.CompletedDate,
    fad.ActiveFrom,
    fad.ActiveTo,
    fad.ActiveWorkingDays                                AS ActiveDays,
    ISNULL(fud.DaysUpdated, 0)                           AS DaysUpdated,
    fad.ActiveWorkingDays - ISNULL(fud.DaysUpdated, 0)  AS DaysMissed
FROM FileActiveDays fad
LEFT JOIN FileUpdateDays fud
       ON fud.OrderID=fad.OrderID AND fud.AdminID=fad.AdminID
ORDER BY fad.AdminID, fad.CompletedDate"""

        file_rows = []
        try:
            file_rows = execute_query(file_sql)
            print(f'[MissingStatus] Per-file: {len(file_rows)} rows')
        except Exception as e:
            print(f'[MissingStatus] Per-file failed (non-critical): {e}')

        from collections import defaultdict
        files_by_admin = defaultdict(list)
        for fr in file_rows:
            files_by_admin[fr['AdminID']].append(fr)

        for row in agg_rows:
            row['FileBreakdown'] = files_by_admin.get(row['AdminID'], [])

        return agg_rows

    def aggregate(self, rows):
        if not rows:
            return {'numerator':0,'denominator':0,'success_ratio':None,'orders':[]}

        row   = rows[0]
        wdays = row.get('WorkingDaysInMonth', 0)
        upd   = row.get('DaysWithUpdate', 0)
        miss  = row.get('MissedDays', 0)
        ratio = round(upd/wdays*100, 2) if wdays else None

        file_breakdown = []
        file_orders    = []

        for fb in row.get('FileBreakdown', []):
            days_updated  = fb.get('DaysUpdated', 0)
            active_days   = fb.get('ActiveDays', 0)
            days_missed   = fb.get('DaysMissed', 0)

            file_breakdown.append({
                'order_id':     fb.get('OrderID'),
                'order_number': fb.get('OrderNumber', '—'),
                'company':      fb.get('CompanyName', '—'),
                'completed':    str(fb.get('CompletedDate', '—')),
                'active_from':  str(fb.get('ActiveFrom', '—')),
                'active_to':    str(fb.get('ActiveTo', '—')),
                'active_days':  active_days,
                'days_updated': days_updated,
                'days_missed':  days_missed,
            })

            file_orders.append({
                'order_id':       fb.get('OrderID'),
                'order_number':   fb.get('OrderNumber', '—'),
                'company':        fb.get('CompanyName', '—'),
                'completed':      str(fb.get('CompletedDate', '—')),
                'DaysUpdated':    days_updated,
                'ActiveDays':     active_days,
                'DaysMissed':     days_missed,
                '_is_file_entry': True,
            })

        primary = {
            'TotalActiveDays': wdays,
            'working_days':    wdays,
            'DaysWithUpdate':  upd,
            'MissedDays':      miss,
            'file_breakdown':  file_breakdown,
        }

        return {
            'numerator':     upd,
            'denominator':   wdays,
            'success_ratio': ratio,
            'orders':        [primary] + file_orders,
        }


if __name__ == '__main__':
    import sys
    m = int(sys.argv[1]) if len(sys.argv) > 1 else 5
    y = int(sys.argv[2]) if len(sys.argv) > 2 else 2026
    test_admin_id = int(sys.argv[3]) if len(sys.argv) > 3 else None

    print(f'Running MissingStatusKPI for {m}/{y}...')
    inst = MissingStatusKPI()
    rows = inst.fetch(m, y)
    print(f'Total rows: {len(rows)}')

    if test_admin_id:
        emp_rows = [r for r in rows if r.get('AdminID') == test_admin_id]
        print(f'\nRows for AdminID={test_admin_id}: {len(emp_rows)}')
        if emp_rows:
            print(f'  EmployeeName: {emp_rows[0].get("EmployeeName")}')
            print(f'  WorkingDaysInMonth: {emp_rows[0].get("WorkingDaysInMonth")}')
            print(f'  DaysWithUpdate: {emp_rows[0].get("DaysWithUpdate")}')
            print(f'  MissedDays: {emp_rows[0].get("MissedDays")}')
            agg = inst.aggregate(emp_rows)
            print(f'\n  Aggregate: {agg["numerator"]}/{agg["denominator"]} = {agg["success_ratio"]}%')
            print(f'  File breakdown count: {len(agg["orders"][0]["file_breakdown"])}')
            for fb in agg['orders'][0]['file_breakdown'][:20]:
                print(f'    {fb["order_number"]:<22} {fb["company"]:<30} '
                      f'completed={fb["completed"]} active_from={fb["active_from"]} '
                      f'active_to={fb["active_to"]} active={fb["active_days"]} '
                      f'updated={fb["days_updated"]} missed={fb["days_missed"]}')
        else:
            print(f'  No rows found for this AdminID.')
    else:
        print('\nSample of first 10 employees:')
        for r in rows[:10]:
            print(f'  AdminID={r.get("AdminID"):<6} {r.get("EmployeeName"):<25} '
                  f'updated={r.get("DaysWithUpdate")}/{r.get("WorkingDaysInMonth")} '
                  f'missed={r.get("MissedDays")}')
        print('\nTip: pass an AdminID as 3rd argument for full per-file detail:')
        print(f'  python -m kpis.missing_status {m} {y} 303')

ValueError: invalid literal for int() with base 10: '--f=c:\\Users\\ACER\\AppData\\Roaming\\jupyter\\runtime\\kernel-v354cb863e17dc48b0e4cc4e6cbabd4896e9eb125f.json'

In [53]:
"""
JUPYTER CELL — missing_status_jupyter.py
============================================
Same logic as kpis/missing_status.py (Issue 1 fix applied: completion
day excluded from the active-days denominator). The sys.argv-based
__main__ block has been replaced with plain variables you can edit
directly below — that's what was crashing when pasted into Jupyter,
since sys.argv there holds Jupyter's own kernel-connection arguments,
not real month/year/admin_id values.

Paste this WHOLE thing into one cell and run it.
"""
import sys
sys.path.insert(0, '.')
from db import execute_query


class MissingStatusKPI:
    def fetch(self, month, year):
        agg_sql = f"""
DECLARE @Month INT={month}
DECLARE @Year  INT={year}
DECLARE @MonthStart DATE = DATEFROMPARTS(@Year,@Month,1)
DECLARE @MonthEndExcl DATE = DATEADD(MONTH,1,@MonthStart)
;WITH
LastAllocated AS (
    SELECT h.OrderID, h.AssignedTo_UserID,
        ROW_NUMBER() OVER (PARTITION BY h.OrderID ORDER BY h.Date_Ctreated DESC) AS rn
    FROM Table_OrderStatushistory h
    WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
      AND h.Date_Ctreated<=EOMONTH(DATEFROMPARTS(@Year,@Month,1))
),
LastAlloc AS (SELECT OrderID, AssignedTo_UserID FROM LastAllocated WHERE rn=1),
ResolvedAttribution AS (
    SELECT
        od.ID AS OrderID,
        COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID) AS AdminID
    FROM OrderDetails od
    LEFT JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
    LEFT JOIN LastAlloc la             ON la.OrderID = od.ID
    WHERE od.Flag_Deleted = 0
      AND od.CompletionOn >= @MonthStart
      AND od.CompletionOn <  @MonthEndExcl
),
WorkingDays AS (
    SELECT CAST(DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)) AS DATE) AS WorkDay
    FROM (SELECT TOP 31 ROW_NUMBER() OVER (ORDER BY (SELECT NULL))-1 AS n
          FROM master..spt_values) nums
    WHERE MONTH(DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)))=@Month
      AND DATEPART(WEEKDAY,DATEADD(DAY,n,DATEFROMPARTS(@Year,@Month,1)))<>1
),
TotalWD AS (SELECT COUNT(*) AS cnt FROM WorkingDays),
ActiveEmps AS (
    SELECT DISTINCT ra.AdminID
    FROM ResolvedAttribution ra
    INNER JOIN Admin af ON af.ID_Admin=ra.AdminID
                       AND af.Flag_Active=1 AND (af.Flag_Delete=0 OR af.Flag_Delete IS NULL)
    WHERE ra.AdminID IS NOT NULL
),
UpdateDays AS (
    SELECT ra.AdminID,
        COUNT(DISTINCT CAST(h.Date_Ctreated AS DATE)) AS DaysWithUpdate
    FROM Table_OrderStatushistory h
    INNER JOIN ResolvedAttribution ra ON ra.OrderID = h.OrderID
    WHERE h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
      AND MONTH(h.Date_Ctreated)=@Month AND YEAR(h.Date_Ctreated)=@Year
      AND ra.AdminID IS NOT NULL
    GROUP BY ra.AdminID
)
SELECT ae.AdminID, a.Admin_FirstName+' '+a.Admin_LastName AS EmployeeName,
    tw.cnt AS WorkingDaysInMonth,
    ISNULL(ud.DaysWithUpdate,0) AS DaysWithUpdate,
    tw.cnt - ISNULL(ud.DaysWithUpdate,0) AS MissedDays
FROM ActiveEmps ae
CROSS JOIN TotalWD tw
LEFT  JOIN UpdateDays ud ON ud.AdminID=ae.AdminID
INNER JOIN Admin a ON a.ID_Admin=ae.AdminID
    AND a.Flag_Active=1 AND (a.Flag_Delete=0 OR a.Flag_Delete IS NULL)
ORDER BY EmployeeName"""

        try:
            agg_rows = execute_query(agg_sql)
        except Exception as e:
            print(f'[MissingStatus] Aggregate error: {e}')
            return []

        if not agg_rows:
            return []

        file_sql = f"""
DECLARE @Month INT={month}
DECLARE @Year  INT={year}
DECLARE @MonthStart DATE = DATEFROMPARTS(@Year,@Month,1)
DECLARE @MonthEnd   DATE = EOMONTH(@MonthStart)
DECLARE @MonthEndExcl DATE = DATEADD(MONTH,1,@MonthStart)
DECLARE @RangeStart DATE = DATEADD(DAY,-90,@MonthStart)
;WITH
LastAllocated AS (
    SELECT h.OrderID, h.AssignedTo_UserID,
        ROW_NUMBER() OVER (PARTITION BY h.OrderID ORDER BY h.Date_Ctreated DESC) AS rn
    FROM Table_OrderStatushistory h
    WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
),
LastAlloc AS (SELECT OrderID, AssignedTo_UserID FROM LastAllocated WHERE rn=1),
FirstAllocDate AS (
    SELECT h.OrderID, h.AssignedTo_UserID,
        CAST(MIN(h.Date_Ctreated) AS DATE) AS FirstDate
    FROM Table_OrderStatushistory h
    WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
    GROUP BY h.OrderID, h.AssignedTo_UserID
),
WorkingDays AS (
    SELECT CAST(DATEADD(DAY,n,@RangeStart) AS DATE) AS WorkDay
    FROM (SELECT TOP 1000 ROW_NUMBER() OVER (ORDER BY (SELECT NULL))-1 AS n
          FROM master..spt_values) nums
    WHERE DATEADD(DAY,n,@RangeStart) <= @MonthEnd
      AND DATEPART(WEEKDAY,DATEADD(DAY,n,@RangeStart))<>1
),
FilesCompleted AS (
    SELECT
        COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID) AS AdminID,
        od.ID                                                               AS OrderID,
        od.Order_Number                                                     AS OrderNumber,
        od.CompanyName,
        CAST(od.CompletionOn AS DATE)                                       AS CompletedDate,
        CASE
            WHEN fad.FirstDate IS NOT NULL
             AND CAST(fad.FirstDate AS DATE) <= CAST(od.CompletionOn AS DATE)
            THEN CAST(fad.FirstDate AS DATE)
            ELSE @MonthStart
        END                                                                 AS ActiveFromRaw
    FROM OrderDetails od
    LEFT  JOIN LastAlloc      la  ON la.OrderID=od.ID
    LEFT  JOIN FirstAllocDate fad ON fad.OrderID=od.ID
                                 AND fad.AssignedTo_UserID=la.AssignedTo_UserID
    LEFT  JOIN Table_OrderDetailsMisc m ON m.OrderID=od.ID
    INNER JOIN Admin af ON af.ID_Admin=COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID)
                       AND af.Flag_Active=1
                       AND (af.Flag_Delete=0 OR af.Flag_Delete IS NULL)
    WHERE od.Flag_Deleted=0
      AND od.CompletionOn >= @MonthStart
      AND od.CompletionOn <  @MonthEndExcl
),
FilesWithActiveTo AS (
    SELECT
        fc.*,
        (SELECT MAX(wd2.WorkDay) FROM WorkingDays wd2 WHERE wd2.WorkDay < fc.CompletedDate) AS ActiveTo
    FROM FilesCompleted fc
),
FileActiveDays AS (
    SELECT
        f.AdminID, f.OrderID, f.OrderNumber,
        f.CompanyName, f.CompletedDate, f.ActiveFromRaw AS ActiveFrom, f.ActiveTo,
        COUNT(wd.WorkDay) AS ActiveWorkingDays
    FROM FilesWithActiveTo f
    INNER JOIN WorkingDays wd
           ON wd.WorkDay >= f.ActiveFromRaw
          AND wd.WorkDay <= f.ActiveTo
    GROUP BY f.AdminID, f.OrderID, f.OrderNumber,
             f.CompanyName, f.CompletedDate, f.ActiveFromRaw, f.ActiveTo
),
FileUpdateDays AS (
    SELECT h.OrderID, fad.AdminID,
        COUNT(DISTINCT CAST(h.Date_Ctreated AS DATE)) AS DaysUpdated
    FROM Table_OrderStatushistory h
    INNER JOIN FileActiveDays fad ON fad.OrderID = h.OrderID
    INNER JOIN WorkingDays wd ON CAST(h.Date_Ctreated AS DATE)=wd.WorkDay
    WHERE h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
      AND CAST(h.Date_Ctreated AS DATE) >= fad.ActiveFrom
      AND CAST(h.Date_Ctreated AS DATE) <= fad.ActiveTo
    GROUP BY h.OrderID, fad.AdminID
)
SELECT
    fad.AdminID,
    fad.OrderID,
    fad.OrderNumber,
    fad.CompanyName,
    fad.CompletedDate,
    fad.ActiveFrom,
    fad.ActiveTo,
    fad.ActiveWorkingDays                                AS ActiveDays,
    ISNULL(fud.DaysUpdated, 0)                           AS DaysUpdated,
    fad.ActiveWorkingDays - ISNULL(fud.DaysUpdated, 0)  AS DaysMissed
FROM FileActiveDays fad
LEFT JOIN FileUpdateDays fud
       ON fud.OrderID=fad.OrderID AND fud.AdminID=fad.AdminID
ORDER BY fad.AdminID, fad.CompletedDate"""

        file_rows = []
        try:
            file_rows = execute_query(file_sql)
            print(f'[MissingStatus] Per-file: {len(file_rows)} rows')
        except Exception as e:
            print(f'[MissingStatus] Per-file failed (non-critical): {e}')

        from collections import defaultdict
        files_by_admin = defaultdict(list)
        for fr in file_rows:
            files_by_admin[fr['AdminID']].append(fr)

        for row in agg_rows:
            row['FileBreakdown'] = files_by_admin.get(row['AdminID'], [])

        return agg_rows

    def aggregate(self, rows):
        if not rows:
            return {'numerator':0,'denominator':0,'success_ratio':None,'orders':[]}

        row   = rows[0]
        wdays = row.get('WorkingDaysInMonth', 0)
        upd   = row.get('DaysWithUpdate', 0)
        miss  = row.get('MissedDays', 0)
        ratio = round(upd/wdays*100, 2) if wdays else None

        file_breakdown = []
        for fb in row.get('FileBreakdown', []):
            file_breakdown.append({
                'order_id':     fb.get('OrderID'),
                'order_number': fb.get('OrderNumber', '—'),
                'company':      fb.get('CompanyName', '—'),
                'completed':    str(fb.get('CompletedDate', '—')),
                'active_from':  str(fb.get('ActiveFrom', '—')),
                'active_to':    str(fb.get('ActiveTo', '—')),
                'active_days':  fb.get('ActiveDays', 0),
                'days_updated': fb.get('DaysUpdated', 0),
                'days_missed':  fb.get('DaysMissed', 0),
            })

        return {
            'numerator':     upd,
            'denominator':   wdays,
            'success_ratio': ratio,
            'orders':        [{'working_days': wdays, 'DaysWithUpdate': upd,
                                'MissedDays': miss, 'file_breakdown': file_breakdown}],
        }


# ── EDIT THESE THREE VALUES, THEN RUN THE CELL ──────────────────────────────
MONTH = 5
YEAR = 2026
TEST_ADMIN_ID = 303
# ─────────────────────────────────────────────────────────────────────────────

print(f'Running MissingStatusKPI for {MONTH}/{YEAR}...')
inst = MissingStatusKPI()
rows = inst.fetch(MONTH, YEAR)
print(f'Total rows: {len(rows)}')

emp_rows = [r for r in rows if r.get('AdminID') == TEST_ADMIN_ID]
print(f'\nRows for AdminID={TEST_ADMIN_ID}: {len(emp_rows)}')

if emp_rows:
    print(f'  EmployeeName: {emp_rows[0].get("EmployeeName")}')
    print(f'  WorkingDaysInMonth: {emp_rows[0].get("WorkingDaysInMonth")}')
    print(f'  DaysWithUpdate: {emp_rows[0].get("DaysWithUpdate")}')
    print(f'  MissedDays: {emp_rows[0].get("MissedDays")}')

    agg = inst.aggregate(emp_rows)
    print(f'\n  Aggregate: {agg["numerator"]}/{agg["denominator"]} = {agg["success_ratio"]}%')

    fb_list = agg['orders'][0]['file_breakdown']
    print(f'  File breakdown count: {len(fb_list)}')
    for fb in fb_list[:20]:
        print(f'    {fb["order_number"]:<22} {fb["company"]:<30} '
              f'completed={fb["completed"]} active_from={fb["active_from"]} '
              f'active_to={fb["active_to"]} active={fb["active_days"]} '
              f'updated={fb["days_updated"]} missed={fb["days_missed"]}')
else:
    print('  No rows found for this AdminID.')

print('\n--- Sample of first 10 employees ---')
for r in rows[:10]:
    print(f'  AdminID={r.get("AdminID")}  {r.get("EmployeeName")}  '
          f'updated={r.get("DaysWithUpdate")}/{r.get("WorkingDaysInMonth")}  '
          f'missed={r.get("MissedDays")}')

Running MissingStatusKPI for 5/2026...
[MissingStatus] Per-file: 466 rows
Total rows: 40

Rows for AdminID=303: 1
  EmployeeName: Aarti Jagtap
  WorkingDaysInMonth: 26
  DaysWithUpdate: 18
  MissedDays: 8

  Aggregate: 18/26 = 69.23%
  File breakdown count: 20
    MMC26042533340619      Stevenette                     completed=2026-05-01 active_from=2026-04-25 active_to=2026-04-30 active=5 updated=1 missed=4
    MMC26043034490400      Michael Hayman                 completed=2026-05-04 active_from=2026-05-01 active_to=2026-05-02 active=2 updated=1 missed=1
    MMC26050235060612      NewCo Enterprises Australia Pty Ltd  completed=2026-05-08 active_from=2026-05-02 active_to=2026-05-07 active=5 updated=1 missed=4
    MMC26050635801031      Accent Live 11                 completed=2026-05-09 active_from=2026-05-07 active_to=2026-05-08 active=2 updated=1 missed=1
    MMC26050736181205      Solar Advice (Pty) Ltd         completed=2026-05-12 active_from=2026-05-07 active_to=2026-05-11 active

In [54]:
"""
JUPYTER CELL — verify_email_table_all_employees.py
=======================================================
Full-scale check across ALL employees/files completed in May 2026:
1. How many emails per file, on average? (file-level density)
2. Do emails cluster ONLY at completion, or spread across the active
   period? (day-level distribution)
3. Which Email_Name templates actually fire DURING a file's active
   window (between allocation and completion), not just at the
   bookends?
4. Cross-reference: for files where Table_OrderStatushistory shows ZERO
   updates on a given day, does an email exist on that SAME day for
   that SAME file? This is the real test of whether emails fill gaps
   status-history misses.
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

MONTH, YEAR = 5, 2026

print('=' * 78)
print('STEP 1 — Email count per file, for all files completed in May 2026')
print('=' * 78)
sql1 = f"""
DECLARE @MonthStart DATE = '{YEAR}-{MONTH:02d}-01'
DECLARE @MonthEndExcl DATE = DATEADD(MONTH, 1, @MonthStart)
SELECT od.ID AS OrderID, od.Order_Number, od.CompanyName,
       COUNT(e.ID_Notification) AS EmailCount
FROM OrderDetails od
LEFT JOIN Table_EmailNotificationHistory e ON e.OrderID = od.ID
WHERE od.CompletionOn >= @MonthStart AND od.CompletionOn < @MonthEndExcl
  AND od.Flag_Deleted = 0
GROUP BY od.ID, od.Order_Number, od.CompanyName
"""
file_email_counts = execute_query(sql1)
print(f'Total files completed in May: {len(file_email_counts)}')
counts = [f['EmailCount'] for f in file_email_counts]
if counts:
    print(f'Avg emails per file: {sum(counts)/len(counts):.2f}')
    print(f'Min: {min(counts)}  Max: {max(counts)}')
    from collections import Counter
    dist = Counter(counts)
    print(f'Distribution of email-count-per-file: {dict(sorted(dist.items()))}')

print('\n' + '=' * 78)
print('STEP 2 — For files with MULTIPLE emails, what are the Email_Name')
print('values and WHEN do they fire relative to allocation/completion?')
print('=' * 78)
sql2 = f"""
DECLARE @MonthStart DATE = '{YEAR}-{MONTH:02d}-01'
DECLARE @MonthEndExcl DATE = DATEADD(MONTH, 1, @MonthStart)
;WITH MultiEmailFiles AS (
    SELECT od.ID AS OrderID
    FROM OrderDetails od
    INNER JOIN Table_EmailNotificationHistory e ON e.OrderID = od.ID
    WHERE od.CompletionOn >= @MonthStart AND od.CompletionOn < @MonthEndExcl
      AND od.Flag_Deleted = 0
    GROUP BY od.ID
    HAVING COUNT(e.ID_Notification) >= 3
)
SELECT TOP 50 od.Order_Number, od.CompanyName, od.CompletionOn,
       e.Email_Name, e.Date_Created
FROM Table_EmailNotificationHistory e
INNER JOIN OrderDetails od ON od.ID = e.OrderID
WHERE e.OrderID IN (SELECT OrderID FROM MultiEmailFiles)
ORDER BY od.Order_Number, e.Date_Created
"""
multi = execute_query(sql2)
print(f'Sample rows from files with 3+ emails: {len(multi)}')
current = None
for m in multi:
    if m['Order_Number'] != current:
        print(f"\n  {m['Order_Number']} — {m['CompanyName']}  (completed {m['CompletionOn']})")
        current = m['Order_Number']
    print(f"    {m['Date_Created']}  {m['Email_Name']!r}")

print('\n' + '=' * 78)
print('STEP 3 — Distinct Email_Name values across ALL May-completed files,')
print('with count of how many files each template appears on')
print('=' * 78)
sql3 = f"""
DECLARE @MonthStart DATE = '{YEAR}-{MONTH:02d}-01'
DECLARE @MonthEndExcl DATE = DATEADD(MONTH, 1, @MonthStart)
SELECT e.Email_Name, COUNT(DISTINCT e.OrderID) AS DistinctFiles, COUNT(*) AS TotalEmails
FROM Table_EmailNotificationHistory e
INNER JOIN OrderDetails od ON od.ID = e.OrderID
WHERE od.CompletionOn >= @MonthStart AND od.CompletionOn < @MonthEndExcl
  AND od.Flag_Deleted = 0
GROUP BY e.Email_Name
ORDER BY DistinctFiles DESC
"""
template_stats = execute_query(sql3)
for t in template_stats:
    print(f"  {t['Email_Name']!r:<45} files={t['DistinctFiles']:<6} total_emails={t['TotalEmails']}")

print('\n' + '=' * 78)
print('STEP 4 — THE KEY TEST: days where Table_OrderStatushistory shows')
print('ZERO updates for a file, but an email EXISTS that same day')
print('(this is the only thing that would prove emails fill a real gap)')
print('=' * 78)
sql4 = f"""
DECLARE @MonthStart DATE = '{YEAR}-{MONTH:02d}-01'
DECLARE @MonthEndExcl DATE = DATEADD(MONTH, 1, @MonthStart)
;WITH MayFiles AS (
    SELECT od.ID AS OrderID, od.Order_Number, od.CompanyName, od.CompletionOn
    FROM OrderDetails od
    WHERE od.CompletionOn >= @MonthStart AND od.CompletionOn < @MonthEndExcl
      AND od.Flag_Deleted = 0
),
StatusDays AS (
    SELECT DISTINCT h.OrderID, CAST(h.Date_Ctreated AS DATE) AS StatusDate
    FROM Table_OrderStatushistory h
    WHERE h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID > 0
),
EmailDays AS (
    SELECT DISTINCT e.OrderID, CAST(e.Date_Created AS DATE) AS EmailDate, e.Email_Name
    FROM Table_EmailNotificationHistory e
)
SELECT mf.Order_Number, mf.CompanyName, ed.EmailDate, ed.Email_Name
FROM MayFiles mf
INNER JOIN EmailDays ed ON ed.OrderID = mf.OrderID
WHERE NOT EXISTS (
    SELECT 1 FROM StatusDays sd
    WHERE sd.OrderID = mf.OrderID AND sd.StatusDate = ed.EmailDate
)
ORDER BY mf.Order_Number, ed.EmailDate
"""
gap_fills = execute_query(sql4)
print(f'Email events on a day with ZERO status-history update for that file: {len(gap_fills)}')
print('(If this is non-zero and NOT just completion-day emails, that is real evidence)')
for g in gap_fills[:40]:
    print(f"  {g['Order_Number']:<20} {g['CompanyName']:<25} EmailDate={g['EmailDate']} "
          f"Email_Name={g['Email_Name']!r}")

STEP 1 — Email count per file, for all files completed in May 2026
Total files completed in May: 539
Avg emails per file: 2.32
Min: 0  Max: 8
Distribution of email-count-per-file: {0: 35, 1: 3, 2: 336, 3: 114, 4: 32, 5: 14, 6: 3, 7: 1, 8: 1}

STEP 2 — For files with MULTIPLE emails, what are the Email_Name
values and WHEN do they fire relative to allocation/completion?
Sample rows from files with 3+ emails: 50

  MMC26022620550747 — Biedermann Motech Inc., USA  (completed 2026-05-28 07:26:57.373000)
    2026-02-26 10:08:59.140000  'Summary Order Prepare'
    2026-05-28 07:26:57.333000  'Conversion Status Completion'
    2026-05-28 07:26:57.720000  'Order Review Mail'

  MMC26031925841112 — Devemos Plastics LTD  (completed 2026-05-13 13:05:56.607000)
    2026-03-19 12:16:28.320000  'Summary Order Prepare'
    2026-05-13 13:05:56.553000  'Conversion Status Completion'
    2026-05-13 13:05:56.970000  'Order Review Mail'

  MMC26040128470346 — LocPlan Company  (completed 2026-05-12 15:30:0

In [55]:
"""
backend/clear_all_caches.py
==============================
Clears EVERY cache touched by this session's changes, before a fresh
restart:
  - score_cache          (main dashboard blob cache)
  - keka_attendance_cache (per-month Keka computation cache)
  - employee_kpi_cache    (new per-employee/month/KPI cache — detail
                            page + trend chart)

Run this ONCE before starting app.py, any time you've changed KPI
logic (post_conversion's Reckon fix, missing_status's completion-day
fix, etc.) and want every cache layer to recompute fresh rather than
serve stale data from before the fix.

Note: this does NOT delete keka_admin_map, keka_upgrade_log, or
keka_sync_job's bookkeeping tables — those are identity/mapping data,
not score caches, and clearing them would lose real progress (name
resolutions, upgrade history) for no benefit.
"""
import sqlite3

DB_PATH = 'scorecard.db'

TABLES_TO_CLEAR = [
    'score_cache',
    'keka_attendance_cache',
    'employee_kpi_cache',
]

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

for table in TABLES_TO_CLEAR:
    try:
        cur.execute(f"DELETE FROM {table}")
        print(f'[ClearCache] Cleared {table} ({cur.rowcount} rows removed)')
    except sqlite3.OperationalError as e:
        # Table doesn't exist yet — fine, it'll be created fresh on
        # first use, nothing to clear.
        print(f'[ClearCache] {table} does not exist yet (will be created on first use) — skipped')

conn.commit()
conn.close()

print('\n[ClearCache] Done. All score-related caches cleared.')
print('[ClearCache] Identity/mapping tables (keka_admin_map, keka_upgrade_log)')
print('[ClearCache] were intentionally left untouched.')

[ClearCache] Cleared score_cache (1 rows removed)
[ClearCache] Cleared keka_attendance_cache (1 rows removed)
[ClearCache] employee_kpi_cache does not exist yet (will be created on first use) — skipped

[ClearCache] Done. All score-related caches cleared.
[ClearCache] Identity/mapping tables (keka_admin_map, keka_upgrade_log)
[ClearCache] were intentionally left untouched.


In [56]:
"""
JUPYTER CELL — diagnose_two_dropped_files.py
=================================================
TWO confirmed real examples of files that appear in post_conversion
(Clean, On Time) for AdminID 303, but are MISSING from
missing_status.py's per-file breakdown:
  - MMC26050635670629 (Motorlife Museum, completed 2026-05-07)
  - MMC26052941110737 (Rosebud Property, completed 2026-05-31)

This walks BOTH files through missing_status.py's EXACT FilesCompleted
CTE, step by step, isolating each JOIN to find precisely where they
get excluded. No more guessing -- print every intermediate result.
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

MONTH, YEAR = 5, 2026
TEST_ORDERS = ['MMC26050635670629', 'MMC26052941110737']

for order_num in TEST_ORDERS:
    print('=' * 78)
    print(f'ORDER: {order_num}')
    print('=' * 78)

    # Step 0: basic order info
    sql0 = f"""
        SELECT od.ID, od.Order_Number, od.CompanyName, od.CompletionOn,
               od.Flag_Deleted, m.PickedUpBy, m.Summary_AssignUserID
        FROM OrderDetails od
        LEFT JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
        WHERE od.Order_Number = '{order_num}'
    """
    base = execute_query(sql0)
    if not base:
        print('  NOT FOUND in OrderDetails at all!')
        continue
    b = base[0]
    print(f"  OrderID={b['ID']}  CompletionOn={b['CompletionOn']}  Flag_Deleted={b['Flag_Deleted']}")
    print(f"  PickedUpBy={b['PickedUpBy']}  Summary_AssignUserID={b['Summary_AssignUserID']}")

    order_id = b['ID']

    # Step 1: does LastAlloc produce a row for this order?
    sql1 = f"""
        ;WITH LastAllocated AS (
            SELECT h.OrderID, h.AssignedTo_UserID,
                ROW_NUMBER() OVER (PARTITION BY h.OrderID ORDER BY h.Date_Ctreated DESC) AS rn
            FROM Table_OrderStatushistory h
            WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
        )
        SELECT OrderID, AssignedTo_UserID, rn
        FROM LastAllocated
        WHERE OrderID = {order_id}
        ORDER BY rn
    """
    la = execute_query(sql1)
    print(f'\n  LastAllocated rows (StatusID=11 events): {len(la)}')
    for r in la[:5]:
        print(f"    AssignedTo={r['AssignedTo_UserID']}  rn={r['rn']}")
    if not la:
        print('  >>> NO StatusID=11 allocation event exists for this order at all!')
        print('  >>> This means LastAlloc.AssignedTo_UserID will be NULL via the LEFT JOIN,')
        print('  >>> so COALESCE falls through to PickedUpBy/Summary_AssignUserID --')
        print('  >>> should still resolve IF those are populated. Checking COALESCE result:')

    resolved_admin = la[0]['AssignedTo_UserID'] if la else (b['PickedUpBy'] or b['Summary_AssignUserID'])
    print(f'\n  Effective resolved AdminID (COALESCE logic): {resolved_admin}')

    # Step 2: Admin filter check
    if resolved_admin:
        sql2 = f"""
            SELECT ID_Admin, Admin_FirstName, Admin_LastName, Flag_Active, Flag_Delete
            FROM Admin WHERE ID_Admin = {resolved_admin}
        """
        adm = execute_query(sql2)
        if adm:
            a = adm[0]
            print(f"  Admin check: Flag_Active={a['Flag_Active']}  Flag_Delete={a['Flag_Delete']}  "
                  f"Name={a['Admin_FirstName']} {a['Admin_LastName']}")
        else:
            print(f'  >>> Admin record for ID={resolved_admin} NOT FOUND AT ALL')

    # Step 3: Full ResolvedAttribution + ActiveEmps replication
    sql3 = f"""
        ;WITH
        LastAllocated AS (
            SELECT h.OrderID, h.AssignedTo_UserID,
                ROW_NUMBER() OVER (PARTITION BY h.OrderID ORDER BY h.Date_Ctreated DESC) AS rn
            FROM Table_OrderStatushistory h
            WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
        ),
        LastAlloc AS (SELECT OrderID, AssignedTo_UserID FROM LastAllocated WHERE rn=1)
        SELECT
            od.ID AS OrderID,
            m.PickedUpBy, la.AssignedTo_UserID AS LastAllocAdmin, m.Summary_AssignUserID,
            COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID) AS ResolvedAdminID
        FROM OrderDetails od
        LEFT JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
        LEFT JOIN LastAlloc la ON la.OrderID = od.ID
        WHERE od.ID = {order_id}
    """
    full = execute_query(sql3)
    for r in full:
        print(f"\n  FULL chain: PickedUpBy={r['PickedUpBy']}  LastAllocAdmin={r['LastAllocAdmin']}  "
              f"Summary_AssignUserID={r['Summary_AssignUserID']}  -> ResolvedAdminID={r['ResolvedAdminID']}")

    # Step 4: Does this order have a FirstAllocDate row matching the
    # resolved admin specifically? (the join in FilesCompleted requires
    # fad.AssignedTo_UserID = la.AssignedTo_UserID -- if PickedUpBy/
    # Summary_AssignUserID resolved the AdminID but la.AssignedTo_UserID
    # is a DIFFERENT person or NULL, fad will not match)
    sql4 = f"""
        ;WITH FirstAllocDate AS (
            SELECT h.OrderID, h.AssignedTo_UserID, CAST(MIN(h.Date_Ctreated) AS DATE) AS FirstDate
            FROM Table_OrderStatushistory h
            WHERE h.StatusID=11 AND h.AssignedTo_UserID IS NOT NULL AND h.AssignedTo_UserID>0
            GROUP BY h.OrderID, h.AssignedTo_UserID
        )
        SELECT OrderID, AssignedTo_UserID, FirstDate
        FROM FirstAllocDate WHERE OrderID = {order_id}
    """
    fad = execute_query(sql4)
    print(f'\n  FirstAllocDate rows for this order: {len(fad)}')
    for r in fad:
        print(f"    AssignedTo={r['AssignedTo_UserID']}  FirstDate={r['FirstDate']}")
    if not fad:
        print('  >>> ZERO FirstAllocDate rows -- this order has NO StatusID=11 event,')
        print('  >>> meaning fad will be NULL in the join, ActiveFromRaw falls back to')
        print('  >>> month start. This alone should NOT cause exclusion though, since')
        print('  >>> the CASE has an ELSE branch for that.')

    print()

ORDER: MMC26050635670629
  OrderID=34803  CompletionOn=2026-05-07 17:10:41.050000  Flag_Deleted=False
  PickedUpBy=303  Summary_AssignUserID=None

  LastAllocated rows (StatusID=11 events): 1
    AssignedTo=303  rn=1

  Effective resolved AdminID (COALESCE logic): 303
  Admin check: Flag_Active=True  Flag_Delete=False  Name=Aarti Jagtap

  FULL chain: PickedUpBy=303  LastAllocAdmin=303  Summary_AssignUserID=None  -> ResolvedAdminID=303

  FirstAllocDate rows for this order: 1
    AssignedTo=303  FirstDate=2026-05-07

ORDER: MMC26052941110737
  OrderID=35347  CompletionOn=2026-06-07 15:30:18.697000  Flag_Deleted=False
  PickedUpBy=None  Summary_AssignUserID=None

  LastAllocated rows (StatusID=11 events): 1
    AssignedTo=303  rn=1

  Effective resolved AdminID (COALESCE logic): 303
  Admin check: Flag_Active=True  Flag_Delete=False  Name=Aarti Jagtap

  FULL chain: PickedUpBy=None  LastAllocAdmin=303  Summary_AssignUserID=None  -> ResolvedAdminID=303

  FirstAllocDate rows for this ord

In [58]:
"""
JUPYTER CELL — check_rosebud_reckon.py
==========================================
Rosebud Property (MMC26052941110737) shows CompletionOn=2026-06-07,
but appeared in a screenshot context suggesting May. Checking if it's
a Reckon file -- if so, the SAME review-date logic we already fixed in
post_conversion.py/delayed_conversion.py would explain this: its real
customer review happened in May, with the +7-day auto-completion
landing in June.
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

sql = """
    SELECT od.Order_Number, od.CompanyName, od.DestinationSW,
           od.CompletionOn, m.Date_OrderReviewStatus,
           DATEDIFF(DAY, m.Date_OrderReviewStatus, od.CompletionOn) AS GapDays
    FROM OrderDetails od
    LEFT JOIN Table_OrderDetailsMisc m ON m.OrderID = od.ID
    WHERE od.Order_Number = 'MMC26052941110737'
"""
result = execute_query(sql)
for r in result:
    print(f"Order: {r['Order_Number']} — {r['CompanyName']}")
    print(f"  DestinationSW: {r['DestinationSW']!r}")
    print(f"  CompletionOn: {r['CompletionOn']}")
    print(f"  Date_OrderReviewStatus: {r['Date_OrderReviewStatus']}")
    print(f"  GapDays: {r['GapDays']}")
    if r['DestinationSW'] == 'Reckon':
        print()
        print('  >>> CONFIRMED: This IS a Reckon file.')
        print('  >>> Its real review date determines which month it should be')
        print('  >>> credited under, per the already-deployed Reckon fix in')
        print('  >>> post_conversion.py / delayed_conversion.py.')
        print('  >>> missing_status.py does NOT yet have this same Reckon fix --')
        print('  >>> it still filters purely by od.CompletionOn, which is why')
        print('  >>> this file (review in May, auto-complete in June) is excluded')
        print('  >>> from this months missing_status report but WOULD show in')
        print('  >>> the post_conversion/delayed_conversion report for that month')
        print('  >>> once that fix is live there too.')
    else:
        print()
        print('  >>> NOT a Reckon file. The May/June discrepancy in the original')
        print('  >>> screenshot is unrelated to the Reckon auto-completion issue --')
        print('  >>> worth checking what field that OTHER report/screenshot used')
        print('  >>> to decide May for this file, since it disagrees with')
        print('  >>> CompletionOn directly.')

Order: MMC26052941110737 — Rosebud Property
  DestinationSW: 'Reckon'
  CompletionOn: 2026-06-07 15:30:18.697000
  Date_OrderReviewStatus: 2026-05-31 12:50:52.280000
  GapDays: 7

  >>> CONFIRMED: This IS a Reckon file.
  >>> Its real review date determines which month it should be
  >>> credited under, per the already-deployed Reckon fix in
  >>> post_conversion.py / delayed_conversion.py.
  >>> missing_status.py does NOT yet have this same Reckon fix --
  >>> it still filters purely by od.CompletionOn, which is why
  >>> this file (review in May, auto-complete in June) is excluded
  >>> from this months missing_status report but WOULD show in
  >>> the post_conversion/delayed_conversion report for that month
  >>> once that fix is live there too.


In [59]:
"""
JUPYTER CELL — verify_preeti_shinde_duplicate.py
=====================================================
Two "Preeti Shinde" rows show up with different scores (87.9 vs 16.8).
Checking the real SQL Admin records to determine: are these the SAME
person under two duplicate AdminIDs (dedup bug), or two DIFFERENT real
people who happen to share this name (correct to show separately)?
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

sql = """
    SELECT ID_Admin, Admin_FirstName, Admin_LastName, Admin_Email,
           Flag_Active, Flag_Delete, Date_Created
    FROM Admin
    WHERE Admin_FirstName LIKE '%Preeti%' AND Admin_LastName LIKE '%Shinde%'
    ORDER BY ID_Admin
"""
rows = execute_query(sql)
print(f'Found {len(rows)} Admin record(s) matching "Preeti Shinde":\n')
for r in rows:
    print(f"  ID_Admin={r['ID_Admin']:<6} Name={r['Admin_FirstName']} {r['Admin_LastName']}")
    print(f"    Email={r['Admin_Email']}  Active={r['Flag_Active']}  Deleted={r['Flag_Delete']}  Created={r['Date_Created']}")

# Now check: which AdminIDs actually appear in May 2026 KPI data, and
# how many rows/files each one has, to see the REAL source of the two
# different scores
print('\n' + '=' * 78)
print('Checking post_conversion / missing_status row counts for each AdminID found above')
print('=' * 78)
for r in rows:
    aid = r['ID_Admin']
    sql2 = f"""
        SELECT COUNT(*) AS cnt
        FROM Table_OrderDetailsMisc m
        INNER JOIN OrderDetails od ON od.ID = m.OrderID
        WHERE m.Flag_OrderCompleted = 1
          AND COALESCE(m.PickedUpBy, m.Summary_AssignUserID) = {aid}
          AND od.CompletionOn >= '2026-05-01' AND od.CompletionOn < '2026-06-01'
    """
    count = execute_query(sql2)
    print(f"  AdminID={aid}: {count[0]['cnt']} files completed in May 2026 (PickedUpBy/Summary_AssignUserID)")

Found 7 Admin record(s) matching "Preeti Shinde":

  ID_Admin=19     Name=Preeti  Shinde
    Email=Preeti.Shinde@mmcconvert.com  Active=True  Deleted=False  Created=2023-07-17 09:23:36.800000
  ID_Admin=75     Name=Preeti Shinde qbo
    Email=preeti.xero@gmail.com  Active=False  Deleted=False  Created=2023-09-27 09:45:17.460000
  ID_Admin=141    Name=Preeti  Shinde qbo retail
    Email=preetiqboretail@gmail.com  Active=False  Deleted=False  Created=2023-12-25 06:19:44.877000
  ID_Admin=143    Name=Preeti Shinde Xero Retail
    Email=Preetixeroretail@gmail.com  Active=False  Deleted=False  Created=2023-12-25 06:21:21.957000
  ID_Admin=174    Name=Preeti ROW Shinde
    Email=preetirow@gmail.com  Active=False  Deleted=False  Created=2024-01-16 07:58:26.703000
  ID_Admin=306    Name=Preeti  Shinde Sage One to Xero
    Email=preeti@mmconvert.com  Active=False  Deleted=False  Created=2024-06-20 13:46:25.300000
  ID_Admin=347    Name=Preeti Shinde
    Email=Preeti123@gmail.com  Active=False  

In [60]:
"""
JUPYTER CELL — verify_blank_conversion_employees.py
========================================================
~21 employees show blank Post Conversion / Delayed Conversion /
Missing Status but have real attendance data. Checking:
1. The FULL list of who's affected this month
2. For each, do they ACTUALLY have files in SQL Server attributed to
   them this month? (if yes, they're being wrongly excluded; if no,
   blanks are correct)
3. What does score_engine.py's conversion_ids set look like for them
   specifically -- are they missing from it, and why?
"""
import sys
sys.path.insert(0, '.')
from db import execute_query
from kpis.post_conversion import PostConversionKPI
from kpis.missing_status import MissingStatusKPI

MONTH, YEAR = 5, 2026

print('Fetching post_conversion and missing_status data...')
pc_rows = PostConversionKPI().fetch(MONTH, YEAR)
ms_rows = MissingStatusKPI().fetch(MONTH, YEAR)

pc_admin_ids = {r['AdminID'] for r in pc_rows if r.get('AdminID')}
ms_admin_ids = {r['AdminID'] for r in ms_rows if r.get('AdminID')}

print(f'AdminIDs with post_conversion data: {len(pc_admin_ids)}')
print(f'AdminIDs with missing_status data: {len(ms_admin_ids)}')

# Get every employee currently showing in the Conversion department,
# per the SQL Admin side, regardless of whether they have KPI data
print('\n' + '=' * 78)
print('Checking ANY file attributed to each of these specific employees')
print('(from the screenshot) for May 2026 -- do they have real files at all?')
print('=' * 78)
target_names = ['Bharti Bhavsar', 'Bhumika Singh', 'Deepak Kumar Namdeo',
                 'Deepak Mishra', 'Deepak Parmar', 'Ekta Hirke']

for name in target_names:
    parts = name.split()
    first, last = parts[0], parts[-1]
    sql = f"""
        SELECT ID_Admin, Admin_FirstName, Admin_LastName, Flag_Active, Flag_Delete
        FROM Admin
        WHERE Admin_FirstName LIKE '%{first}%' AND Admin_LastName LIKE '%{last}%'
    """
    admins = execute_query(sql)
    print(f'\n{name}: {len(admins)} Admin record(s) found')
    for a in admins:
        aid = a['ID_Admin']
        in_pc = aid in pc_admin_ids
        in_ms = aid in ms_admin_ids
        sql2 = f"""
            SELECT COUNT(*) AS cnt
            FROM Table_OrderDetailsMisc m
            INNER JOIN OrderDetails od ON od.ID = m.OrderID
            WHERE m.Flag_OrderCompleted = 1
              AND COALESCE(m.PickedUpBy, m.Summary_AssignUserID) = {aid}
              AND od.CompletionOn >= '2026-05-01' AND od.CompletionOn < '2026-06-01'
        """
        file_count = execute_query(sql2)[0]['cnt']
        print(f"  AdminID={aid:<6} Active={a['Flag_Active']} Deleted={a['Flag_Delete']} "
              f"in_post_conversion_output={in_pc} in_missing_status_output={in_ms} "
              f"real_files_this_month={file_count}")

print('\n' + '=' * 78)
print('Department check: what department does score_engine see for these people')
print('(via the post_conversion/missing_status Department field, if they have ANY KPI row)')
print('=' * 78)
all_rows = pc_rows + ms_rows
for name in target_names:
    matches = [r for r in all_rows if name.lower() in (r.get('EmployeeName') or '').lower()]
    if matches:
        for m in matches[:1]:
            print(f"  {name}: Department={m.get('Department')!r}  AdminID={m.get('AdminID')}")
    else:
        print(f"  {name}: NOT FOUND in any conversion KPI row at all this month")

Fetching post_conversion and missing_status data...
[MissingStatus] Per-file: 466 rows
AdminIDs with post_conversion data: 41
AdminIDs with missing_status data: 40

Checking ANY file attributed to each of these specific employees
(from the screenshot) for May 2026 -- do they have real files at all?

Bharti Bhavsar: 0 Admin record(s) found

Bhumika Singh: 1 Admin record(s) found
  AdminID=928    Active=True Deleted=False in_post_conversion_output=False in_missing_status_output=False real_files_this_month=0

Deepak Kumar Namdeo: 0 Admin record(s) found

Deepak Mishra: 1 Admin record(s) found
  AdminID=389    Active=True Deleted=False in_post_conversion_output=False in_missing_status_output=False real_files_this_month=43

Deepak Parmar: 0 Admin record(s) found

Ekta Hirke: 1 Admin record(s) found
  AdminID=344    Active=False Deleted=False in_post_conversion_output=False in_missing_status_output=False real_files_this_month=5

Department check: what department does score_engine see for the

In [61]:
"""
JUPYTER CELL — trace_deepak_mishra_exclusion.py
====================================================
AdminID=389, Active=True, 43 real files completed in May -- yet ZERO
appearance in post_conversion.py's output. Running the EXACT
post_conversion.py query but checking each stage to find where he
drops out.
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

AID = 389

print('Step 1 — does Admin 389 pass the Flag_Active/Flag_Delete filter EXACTLY as the query checks it?')
sql1 = f"""
    SELECT ID_Admin, Admin_FirstName, Admin_LastName, Flag_Active, Flag_Delete
    FROM Admin WHERE ID_Admin = {AID}
"""
r = execute_query(sql1)
for x in r:
    print(f"  Flag_Active={x['Flag_Active']!r} (type={type(x['Flag_Active'])})  "
          f"Flag_Delete={x['Flag_Delete']!r} (type={type(x['Flag_Delete'])})")

print('\nStep 2 — sample 5 of his real files, check Flag_OrderCompleted, CompletionOn, attribution')
sql2 = f"""
    SELECT TOP 5 od.ID, od.Order_Number, od.CompanyName, od.CompletionOn,
           m.Flag_OrderCompleted, m.PickedUpBy, m.Summary_AssignUserID,
           od.Flag_Deleted
    FROM Table_OrderDetailsMisc m
    INNER JOIN OrderDetails od ON od.ID = m.OrderID
    WHERE COALESCE(m.PickedUpBy, m.Summary_AssignUserID) = {AID}
      AND od.CompletionOn >= '2026-05-01' AND od.CompletionOn < '2026-06-01'
      AND m.Flag_OrderCompleted = 1
"""
files = execute_query(sql2)
for f in files:
    print(f"  {f['Order_Number']:<20} CompletionOn={f['CompletionOn']} Flag_Deleted={f['Flag_Deleted']} "
          f"PickedUpBy={f['PickedUpBy']} Summary_AssignUserID={f['Summary_AssignUserID']}")

if files:
    test_order_id = files[0]['ID']
    print(f"\nStep 3 — run post_conversion.py's EXACT query, but ONLY filtered to OrderID={test_order_id}")
    sql3 = f"""
    ;WITH
    LastAllocated AS (
        SELECT OrderID, AssignedTo_UserID,
            ROW_NUMBER() OVER (PARTITION BY OrderID ORDER BY Date_Ctreated DESC) AS rn
        FROM Table_OrderStatushistory
        WHERE StatusID=11 AND AssignedTo_UserID IS NOT NULL AND AssignedTo_UserID>0
          AND Date_Ctreated<=EOMONTH('2026-05-01')
    ),
    PostConvRaised AS (
        SELECT DISTINCT OrderID FROM Table_OrderStatushistory WHERE StatusID=13
    ),
    BaseData AS (
        SELECT m.ID_OrderDetailsMisc, m.OrderID, m.PickedUpBy, m.Summary_AssignUserID,
            CAST(
                CASE
                    WHEN od.DestinationSW = 'Reckon' AND m.Date_OrderReviewStatus IS NOT NULL
                    THEN m.Date_OrderReviewStatus
                    ELSE od.CompletionOn
                END
            AS DATE) AS FinalCompletionDate
        FROM Table_OrderDetailsMisc m
        INNER JOIN OrderDetails od ON od.ID=m.OrderID
        WHERE m.Flag_OrderCompleted=1
          AND m.OrderID = {test_order_id}
    )
    SELECT
        COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID) AS AdminID,
        a1.Admin_FirstName AS A1First, a1.Admin_LastName AS A1Last,
        a2.Admin_FirstName AS A2First, a2.Admin_LastName AS A2Last,
        a3.Admin_FirstName AS A3First, a3.Admin_LastName AS A3Last,
        b.OrderID, od.Order_Number, b.FinalCompletionDate
    FROM BaseData b
    INNER JOIN Table_OrderDetailsMisc m ON m.ID_OrderDetailsMisc=b.ID_OrderDetailsMisc
    INNER JOIN OrderDetails od           ON od.ID=b.OrderID
    LEFT  JOIN Admin a1                  ON a1.ID_Admin=m.PickedUpBy
    LEFT  JOIN Admin a2                  ON a2.ID_Admin=m.Summary_AssignUserID
    LEFT  JOIN LastAllocated la          ON la.OrderID=b.OrderID AND la.rn=1
    LEFT  JOIN Admin a3                  ON a3.ID_Admin=la.AssignedTo_UserID
    """
    r3 = execute_query(sql3)
    print(f'  Rows from BaseData join (before the active-employee EXISTS filter): {len(r3)}')
    for row in r3:
        print(f"    AdminID={row['AdminID']}  A1(PickedUpBy)={row['A1First']} {row['A1Last']}  "
              f"A2(Summary)={row['A2First']} {row['A2Last']}  A3(LastAlloc)={row['A3First']} {row['A3Last']}")

    print(f"\nStep 4 — does the EXISTS active-employee filter pass for AdminID={AID}?")
    sql4 = f"""
        SELECT 1 AS found FROM Admin ax
        WHERE ax.ID_Admin = {AID}
          AND ax.Flag_Active=1 AND (ax.Flag_Delete=0 OR ax.Flag_Delete IS NULL)
    """
    r4 = execute_query(sql4)
    print(f'  EXISTS filter result: {"PASSES" if r4 else "FAILS — this is the exclusion point"}')

    print(f"\nStep 5 — is OrderID={test_order_id} EVER referenced in Table_OrderStatushistory at all?")
    sql5 = f"""
        SELECT Date_Ctreated, StatusID, AssignedTo_UserID
        FROM Table_OrderStatushistory
        WHERE OrderID = {test_order_id}
        ORDER BY Date_Ctreated
    """
    r5 = execute_query(sql5)
    print(f'  Status history rows for this order: {len(r5)}')
    for h in r5[:10]:
        print(f"    {h['Date_Ctreated']}  StatusID={h['StatusID']}  AssignedTo={h['AssignedTo_UserID']}")

Step 1 — does Admin 389 pass the Flag_Active/Flag_Delete filter EXACTLY as the query checks it?
  Flag_Active=True (type=<class 'bool'>)  Flag_Delete=False (type=<class 'bool'>)

Step 2 — sample 5 of his real files, check Flag_OrderCompleted, CompletionOn, attribution
  MMC26022620550747    CompletionOn=2026-05-28 07:26:57.373000 Flag_Deleted=False PickedUpBy=None Summary_AssignUserID=389
  MMC26031925841112    CompletionOn=2026-05-13 13:05:56.607000 Flag_Deleted=False PickedUpBy=None Summary_AssignUserID=389
  MMC26041030231119    CompletionOn=2026-05-12 17:27:03.100000 Flag_Deleted=False PickedUpBy=None Summary_AssignUserID=389
  MMC26041631381206    CompletionOn=2026-05-08 07:20:24.347000 Flag_Deleted=False PickedUpBy=None Summary_AssignUserID=389
  MMC26042132260540    CompletionOn=2026-05-02 17:34:48.643000 Flag_Deleted=False PickedUpBy=None Summary_AssignUserID=389

Step 3 — run post_conversion.py's EXACT query, but ONLY filtered to OrderID=33307
  Rows from BaseData join (before

In [62]:
"""
JUPYTER CELL — check_all_deepak_mishra_files.py
====================================================
One file was confirmed reassigned (via LastAlloc) from Deepak Mishra
(389, Summary_AssignUserID) to Archita Neema (322, LastAlloc). Checking
ALL 43 of his Summary_AssignUserID files to see if this is the pattern
for EVERY file (meaning post_conversion.py is working correctly and
he simply doesn't have any files where he's the FINAL/current owner)
or whether most resolve correctly to 389 and something else is
excluding them.
"""
import sys
sys.path.insert(0, '.')
from db import execute_query

AID = 389

sql = f"""
;WITH
LastAllocated AS (
    SELECT OrderID, AssignedTo_UserID,
        ROW_NUMBER() OVER (PARTITION BY OrderID ORDER BY Date_Ctreated DESC) AS rn
    FROM Table_OrderStatushistory
    WHERE StatusID=11 AND AssignedTo_UserID IS NOT NULL AND AssignedTo_UserID>0
      AND Date_Ctreated<=EOMONTH('2026-05-01')
),
LastAlloc AS (SELECT OrderID, AssignedTo_UserID FROM LastAllocated WHERE rn=1)
SELECT
    od.Order_Number, od.ID AS OrderID,
    m.PickedUpBy, la.AssignedTo_UserID AS LastAllocAdmin, m.Summary_AssignUserID,
    COALESCE(m.PickedUpBy, la.AssignedTo_UserID, m.Summary_AssignUserID) AS ResolvedAdminID
FROM Table_OrderDetailsMisc m
INNER JOIN OrderDetails od ON od.ID = m.OrderID
LEFT JOIN LastAlloc la ON la.OrderID = od.ID
WHERE m.Flag_OrderCompleted = 1
  AND m.Summary_AssignUserID = {AID}
  AND od.CompletionOn >= '2026-05-01' AND od.CompletionOn < '2026-06-01'
"""
rows = execute_query(sql)
print(f'Total files where Summary_AssignUserID = {AID}: {len(rows)}\n')

resolved_to_389 = 0
resolved_elsewhere = 0
resolved_counter = {}

for r in rows:
    resolved = r['ResolvedAdminID']
    if resolved == AID:
        resolved_to_389 += 1
    else:
        resolved_elsewhere += 1
        resolved_counter[resolved] = resolved_counter.get(resolved, 0) + 1

print(f'Files that RESOLVE to AdminID {AID} (correctly his): {resolved_to_389}')
print(f'Files that resolve to someone ELSE (reassigned via PickedUpBy or LastAlloc): {resolved_elsewhere}')
print(f'\nBreakdown of WHO they got reassigned to:')
for aid, cnt in sorted(resolved_counter.items(), key=lambda x: -x[1]):
    name_sql = f"SELECT Admin_FirstName, Admin_LastName FROM Admin WHERE ID_Admin = {aid}"
    name_r = execute_query(name_sql)
    name = f"{name_r[0]['Admin_FirstName']} {name_r[0]['Admin_LastName']}" if name_r else '(unknown)'
    print(f'  AdminID={aid} ({name}): {cnt} files')

if resolved_to_389 > 0:
    print(f'\n>>> {resolved_to_389} files SHOULD appear under AdminID 389 in post_conversion output.')
    print('>>> If post_conversion.py STILL shows 0 for him despite this, the bug is')
    print('>>> elsewhere (e.g. the active-employee filter, or dedup logic dropping him).')
else:
    print(f'\n>>> ALL of his files get reassigned to someone else via PickedUpBy/LastAlloc.')
    print('>>> post_conversion.py showing 0 for him is then CORRECT — he has no files')
    print('>>> where he is the final/current resolved owner, only the original')
    print('>>> Summary_AssignUserID, which is being correctly superseded.')

Total files where Summary_AssignUserID = 389: 43

Files that RESOLVE to AdminID 389 (correctly his): 0
Files that resolve to someone ELSE (reassigned via PickedUpBy or LastAlloc): 43

Breakdown of WHO they got reassigned to:
  AdminID=336 (Bharti  Bhawsar): 10 files
  AdminID=861 (Asif Khan): 8 files
  AdminID=322 (Archita Neema): 7 files
  AdminID=871 (Chanchal  Keshariya): 7 files
  AdminID=910 (Ishika Gaur): 4 files
  AdminID=903 (Srushti Gupta): 4 files
  AdminID=344 (Ekta Hirke): 2 files
  AdminID=346 (Ayushi Modi): 1 files

>>> ALL of his files get reassigned to someone else via PickedUpBy/LastAlloc.
>>> post_conversion.py showing 0 for him is then CORRECT — he has no files
>>> where he is the final/current resolved owner, only the original
>>> Summary_AssignUserID, which is being correctly superseded.


In [63]:
"""
JUPYTER CELL — trace_preeti_shinde_attendance.py
=====================================================
Confirmed: only AdminID=19 has real conversion files. The other 6
(75, 141, 143, 174, 306, 347) are dead duplicate accounts. But TWO
"Preeti Shinde" rows show up on the dashboard with real attendance
percentages. This finds exactly which AdminID(s) the Keka name-mapper
resolved "Preeti Shinde" to, and whether there are TWO distinct real
people in Keka under this name (a genuine name collision) or a mapping
bug pointing at the wrong dead AdminID.
"""
import sys
sys.path.insert(0, '.')
from kpis.keka_attendance import KekaLeavesKPI, _fetch_all, TBL_EMPLOYEES

MONTH, YEAR = 5, 2026

print('Step 1 — every Keka employee record with "Preeti" and "Shinde" in the name')
print('=' * 78)
emps = _fetch_all(TBL_EMPLOYEES)
matches = [e for e in emps if 'preeti' in (e.get('display_name') or '').lower()
           and 'shinde' in (e.get('display_name') or '').lower()]
print(f'Found {len(matches)} raw Keka rows (before dedup):')
for m in matches:
    print(f"  keka_id={m.get('name')}  display_name={m.get('display_name')!r}  "
          f"email={m.get('email')}  createdat={m.get('createdat')}")

distinct_kids = set(m.get('name') for m in matches)
print(f'\nDistinct keka_id values: {len(distinct_kids)} -> {distinct_kids}')

print('\n' + '=' * 78)
print('Step 2 — what does KekaLeavesKPI.fetch() actually output for "Preeti Shinde"?')
print('=' * 78)
rows = KekaLeavesKPI().fetch(MONTH, YEAR)
preeti_rows = [r for r in rows if 'preeti' in (r.get('EmployeeName') or '').lower()
               and 'shinde' in (r.get('EmployeeName') or '').lower()]
print(f'Rows matching "Preeti Shinde": {len(preeti_rows)}')
for r in preeti_rows:
    print(f"  AdminID={r['AdminID']}  EmployeeName={r['EmployeeName']!r}  "
          f"IsRealAdminID={r['IsRealAdminID']}  KekaID={r['KekaID']}  "
          f"WorkingDays={r['WorkingDays']}  PresentDays={r['PresentDays']}")

Step 1 — every Keka employee record with "Preeti" and "Shinde" in the name
Found 3 raw Keka rows (before dedup):
  keka_id=6db7c8db-bb42-4f3c-a957-8a77b4829ebb  display_name='Preeti Shinde'  email=preeti.shinde2050@gmail.com  createdat=2026-06-18T05:24:55.662Z
  keka_id=6db7c8db-bb42-4f3c-a957-8a77b4829ebb  display_name='Preeti Shinde'  email=preeti.shinde2050@gmail.com  createdat=2026-06-20T05:24:56.921Z
  keka_id=6db7c8db-bb42-4f3c-a957-8a77b4829ebb  display_name='Preeti Shinde'  email=preeti.shinde2050@gmail.com  createdat=2026-06-27T19:43:14.110Z

Distinct keka_id values: 1 -> {'6db7c8db-bb42-4f3c-a957-8a77b4829ebb'}

Step 2 — what does KekaLeavesKPI.fetch() actually output for "Preeti Shinde"?
Rows matching "Preeti Shinde": 1
  AdminID=-591544  EmployeeName='Preeti Shinde'  IsRealAdminID=False  KekaID=6db7c8db-bb42-4f3c-a957-8a77b4829ebb  WorkingDays=26  PresentDays=21


In [66]:
"""
JUPYTER CELL — check_preeti_mapping_status_v2.py
=====================================================
Fixed: checking the REAL keka_admin_map schema first instead of
guessing column names.
"""
import sys
sys.path.insert(0, '.')
import sqlite3
from db import execute_query

KID = '6db7c8db-bb42-4f3c-a957-8a77b4829ebb'

conn = sqlite3.connect('scorecard.db')

print('Actual keka_admin_map columns:')
cols = conn.execute("PRAGMA table_info(keka_admin_map)").fetchall()
col_names = [c[1] for c in cols]
print(f'  {col_names}')

print('\nFull row for Preeti Shinde (SELECT *):')
row = conn.execute("SELECT * FROM keka_admin_map WHERE keka_id = ?", (KID,)).fetchone()
if row:
    for name, val in zip(col_names, row):
        print(f'  {name} = {val!r}')
else:
    print('  NOT FOUND IN keka_admin_map AT ALL — never been processed by the matcher.')

conn.close()

print('\n' + '=' * 78)
print('Manual test: does "Preeti Shinde" match AdminID=19 in the Admin table directly?')
print('=' * 78)
sql = """
    SELECT ID_Admin, Admin_FirstName, Admin_LastName, Flag_Active, Flag_Delete
    FROM Admin WHERE ID_Admin = 19
"""
admin19 = execute_query(sql)
for a in admin19:
    full_name = (a['Admin_FirstName'] + ' ' + a['Admin_LastName'])
    print(f"  AdminID=19 raw: '{a['Admin_FirstName']}' + '{a['Admin_LastName']}'")
    print(f"  Normalized: {full_name.strip().lower()!r}")
    print(f"  Keka name normalized: {'preeti shinde'!r}")
    print(f"  Active={a['Flag_Active']}  Deleted={a['Flag_Delete']}")
    print(f"  Exact string match: {full_name.strip().lower() == 'preeti shinde'}")

print('\n' + '=' * 78)
print('Checking for ANY other Admin rows with the exact same normalized name')
print('(would explain an AMBIGUOUS_EXACT verdict, not a clean failure)')
print('=' * 78)
sql2 = """
    SELECT ID_Admin, Admin_FirstName, Admin_LastName, Flag_Active, Flag_Delete
    FROM Admin
    WHERE LTRIM(RTRIM(LOWER(Admin_FirstName))) = 'preeti'
      AND LTRIM(RTRIM(LOWER(Admin_LastName))) = 'shinde'
"""
exact_matches = execute_query(sql2)
print(f'Admin rows with normalized name EXACTLY "preeti" + "shinde": {len(exact_matches)}')
for a in exact_matches:
    print(f"  ID_Admin={a['ID_Admin']}  Active={a['Flag_Active']}  Deleted={a['Flag_Delete']}")

Actual keka_admin_map columns:
  ['keka_id', 'keka_name', 'admin_id', 'admin_name', 'match_method', 'is_manual', 'last_resolved']

Full row for Preeti Shinde (SELECT *):
  keka_id = '6db7c8db-bb42-4f3c-a957-8a77b4829ebb'
  keka_name = 'Preeti Shinde'
  admin_id = None
  admin_name = None
  match_method = 'ambiguous_exact'
  is_manual = 0
  last_resolved = '2026-06-29T10:09:16.143432'

Manual test: does "Preeti Shinde" match AdminID=19 in the Admin table directly?
  AdminID=19 raw: 'Preeti ' + 'Shinde'
  Normalized: 'preeti  shinde'
  Keka name normalized: 'preeti shinde'
  Active=True  Deleted=False
  Exact string match: False

Checking for ANY other Admin rows with the exact same normalized name
(would explain an AMBIGUOUS_EXACT verdict, not a clean failure)
Admin rows with normalized name EXACTLY "preeti" + "shinde": 2
  ID_Admin=19  Active=True  Deleted=False
  ID_Admin=347  Active=False  Deleted=False
